# 이번 프로젝트 기초정보 🌝

| 항목                | 현재 실험                                                                        |
| ----------------- | ---------------------------------------------------------------------------- |
| 기반 모델             | `skt/kogpt2-base-v2`                                                         |
| 모델 클래스            | `GPT2LMHeadModel`                                                            |
| 기반 모델 파라미터        | **125,164,032개 ≈ 125.2M**                                                    |
| SFT 방식            | LoRA가 아닌 **전체 parameter Fine-Tuning**                                        |
| Tokenizer vocab   | 51,200                                                                       |
| GPU               | Colab **NVIDIA L4, VRAM 22.03GB**                                            |
| 정밀도               | SFT 주 학습 **BF16**                                                            |
| SFT sequence      | `max_length=256`                                                             |
| SFT batch         | 32                                                                           |
| SFT learning rate | `5e-5`                                                                       |
| seed              | 42                                                                           |
| 주요 환경             | PyTorch 2.11.0+cu128, Transformers 5.16.1, datasets 4.8.5, accelerate 1.14.0 |
| TRL               | 설치하지 않고 진행                                                                   |
| PEFT              | 설치되어 있지만 실험에는 사용하지 않음                                                        |

## 👩‍🔬
## 실험 조건


```
M0  KoGPT-2 pretrained      125.2M
 │
 ├─ M1 Raw SFT             125.2M
 ├─ M2 Clean-v1 SFT        125.2M
 ├─ M2b Clean-v2 SFT       125.2M
 └─ M2c Clean-v2 matched   125.2M

바뀐 것
= 모델 크기 X
= 학습 데이터 / 학습 step / decoding O
```
## 실험 진행 과정 미리보기 (오늘도 분량이 많네요)
| 단계       | 모델/실험                          | 무엇을 검증했나                      |
| -------- | ------------------------------ | ----------------------------- |
| M0       | Pretrained KoGPT-2             | 사전학습 LM의 기본 생성 행동             |
| M1       | Raw SFT                        | SFT 자체의 효과                    |
| M2       | Clean-v1 SFT                   | 영어 오염 제거 효과                   |
| Decoding | Greedy / Beam / Top-p / Top-k  | 같은 모델에서 출력 선택법의 효과            |
| M2b      | Clean-v2, 1 epoch              | 한국어 AI boilerplate 추가 제거      |
| M2c      | Clean-v2, 292 steps            | 데이터 감소에 따른 학습량 차이 통제          |
| RM       | Pairwise Reward Model          | chosen > rejected 선호 학습       |
| PPO      | Actor + Reference + RM + Value | RLHF 연결 시도, 최종적으로 불안정성 때문에 중단 |





In [ ]:
# ============================================================
# GD08 - Runtime / GPU Check
# ============================================================

import sys
import platform
import subprocess

print("=" * 70)
print("[Python]")
print(sys.version)

print("\n[Platform]")
print(platform.platform())

print("\n[GPU]")
subprocess.run(["nvidia-smi"])

[Python]
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

[Platform]
Linux-6.6.122+-x86_64-with-glibc2.39

[GPU]


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher"
)

DATA_DIR = ROOT / "data" / "gd08"
MODEL_DIR = ROOT / "models" / "gd08"
OUTPUT_DIR = ROOT / "outputs" / "gd08"

for path in [DATA_DIR, MODEL_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("DATA_DIR  :", DATA_DIR)
print("MODEL_DIR :", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

ROOT      : /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher
DATA_DIR  : /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/data/gd08
MODEL_DIR : /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/models/gd08
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/outputs/gd08


In [ ]:
# ============================================================
# Step 0-3. GPU capability check
# ============================================================

import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"BF16 supported : {torch.cuda.is_bf16_supported()}")

props = torch.cuda.get_device_properties(0)
print(f"VRAM           : {props.total_memory / 1024**3:.2f} GB")

print("\n[Memory]")
print(f"allocated : {torch.cuda.memory_allocated() / 1024**3:.3f} GB")
print(f"reserved  : {torch.cuda.memory_reserved() / 1024**3:.3f} GB")

CUDA available : True
GPU            : NVIDIA L4
BF16 supported : True
VRAM           : 22.03 GB

[Memory]
allocated : 0.000 GB
reserved  : 0.000 GB


In [ ]:
# ============================================================
# Step 0-4. Package audit
# ============================================================

from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "evaluate",
    "trl",
    "peft",
    "bitsandbytes",
    "sentencepiece",
    "rouge-score",
    "sacrebleu",
]

for pkg in packages:
    try:
        print(f"{pkg:<15} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<15} NOT INSTALLED")

torch           2.11.0+cu128
transformers    5.16.1
datasets        4.8.5
accelerate      1.14.0
evaluate        NOT INSTALLED
trl             NOT INSTALLED
peft            0.20.0
bitsandbytes    NOT INSTALLED
sentencepiece   0.2.2
rouge-score     NOT INSTALLED
sacrebleu       NOT INSTALLED


In [ ]:
# ============================================================
# Step 0-5. Fast local workspace
# ============================================================

from pathlib import Path

LOCAL_ROOT = Path("/content/gd08")
LOCAL_DATA = LOCAL_ROOT / "data"
LOCAL_MODELS = LOCAL_ROOT / "models"
LOCAL_OUTPUTS = LOCAL_ROOT / "outputs"
HF_CACHE = LOCAL_ROOT / "hf_cache"

for path in [
    LOCAL_DATA,
    LOCAL_MODELS,
    LOCAL_OUTPUTS,
    HF_CACHE,
]:
    path.mkdir(parents=True, exist_ok=True)

print("LOCAL_ROOT   :", LOCAL_ROOT)
print("LOCAL_DATA   :", LOCAL_DATA)
print("LOCAL_MODELS :", LOCAL_MODELS)
print("LOCAL_OUTPUTS:", LOCAL_OUTPUTS)
print("HF_CACHE     :", HF_CACHE)

LOCAL_ROOT   : /content/gd08
LOCAL_DATA   : /content/gd08/data
LOCAL_MODELS : /content/gd08/models
LOCAL_OUTPUTS: /content/gd08/outputs
HF_CACHE     : /content/gd08/hf_cache


In [ ]:
# ============================================================
# Step 1-1. KoGPT-2 loading smoke test
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "skt/kogpt2-base-v2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
).to("cuda")

print("Tokenizer :", tokenizer.__class__.__name__)
print("Model     :", model.__class__.__name__)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Device    :", next(model.parameters()).device)
print("Dtype     :", next(model.parameters()).dtype)

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  513MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer : GPT2Tokenizer
Model     : GPT2LMHeadModel
Parameters: 125,164,032
Device    : cuda:0
Dtype     : torch.bfloat16


In [ ]:
# ============================================================
# Step 1-2. KoGPT-2 토크나이저 재설정 및 입출력 확인
# ============================================================

from transformers import PreTrainedTokenizerFast

MODEL_ID = "skt/kogpt2-base-v2"

# KoGPT-2는 SKT가 공개한 Character BPE tokenizer를 사용한다.
# 공식 사용 예제에 맞춰 Fast tokenizer와 특수 토큰을 명시한다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_ID,
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    cache_dir=str(HF_CACHE),
)

# 생성 시 모델도 동일한 특수 토큰 ID를 사용하도록 맞춘다.
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print("Tokenizer class :", tokenizer.__class__.__name__)
print("Tokenizer size  :", len(tokenizer))
print("Model vocab size:", model.config.vocab_size)

print("\n[특수 토큰]")
print("BOS :", tokenizer.bos_token, tokenizer.bos_token_id)
print("EOS :", tokenizer.eos_token, tokenizer.eos_token_id)
print("UNK :", tokenizer.unk_token, tokenizer.unk_token_id)
print("PAD :", tokenizer.pad_token, tokenizer.pad_token_id)
print("MASK:", tokenizer.mask_token, tokenizer.mask_token_id)

# 토크나이징 → ID → 다시 문자열로 복원되는지 먼저 확인한다.
test_text = "인공지능이란"

test_ids = tokenizer.encode(
    test_text,
    add_special_tokens=False,
)

test_tokens = tokenizer.convert_ids_to_tokens(test_ids)
restored_text = tokenizer.decode(test_ids)

print("\n[입출력 확인]")
print("원문 :", test_text)
print("토큰 :", test_tokens)
print("ID   :", test_ids)
print("복원 :", restored_text)

Tokenizer class : TokenizersBackend
Tokenizer size  : 51200
Model vocab size: 51200

[특수 토큰]
BOS : </s> 1
EOS : </s> 1
UNK : <unk> 5
PAD : <pad> 3
MASK: <mask> 6

[입출력 확인]
원문 : 인공지능이란
토큰 : ['▁인공', '지', '능', '이란']
ID   : [13753, 8263, 7166, 10479]
복원 : 인공지능이란


### KoGPT-2 토크나이저 확인

- KoGPT-2 모델 자체는 L4 GPU에 정상적으로 로드되었으며 약 1억 2,500만 개의 parameter를 가진 모델임을 확인했다.
- 처음 `AutoTokenizer`로 불러왔을 때 `GPT2Tokenizer`가 선택되었고, 생성 결과가 `�` 문자로 깨지는 현상이 발생했다.
- KoGPT-2는 일반 영어 GPT-2와 동일한 tokenizer를 사용하는 것이 아니라 SKT가 학습한 Character BPE tokenizer를 사용한다.
- 따라서 원본 모델과 동일한 `PreTrainedTokenizerFast` 계열 tokenizer를 사용하는 것이 중요하다.
- tokenizer와 model은 별개 객체이지만, token ID의 의미를 공유하므로 서로 호환되는 조합을 사용해야 한다.

In [ ]:
# ============================================================
# Step 1-3. 사전학습 KoGPT-2의 기본 문장 생성 확인
# ============================================================

import torch

prompt = "인공지능이란"

input_ids = tokenizer.encode(
    prompt,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=50,
        do_sample=False,
        repetition_penalty=2.0,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
)

print("[입력]")
print(prompt)

print("\n[생성 결과]")
print(generated_text)

[입력]
인공지능이란

[생성 결과]
인공지능이란, 인간의 뇌가 스스로 학습하는 능력을 말한다.
이러한 지능은 인간이 가진 모든 것을 가능하게 한다.
인간의 두뇌는 다른 사람의 도움 없이 혼자서만 학습을 할 수 있다.
따라서 인간은 자신의 생각을 논리적으로 표현할 줄 아는 능력이 필요하다.
또한, 이러한 능력은 인간


### 사전학습 KoGPT-2 기본 생성

- 아직 SFT나 RLHF를 적용하지 않은 KoGPT-2 자체의 생성 결과를 baseline으로 확인했다.
- 현재 모델은 사용자의 질문에 답하도록 별도로 학습된 챗봇이 아니라 다음 token을 예측하도록 사전학습된 Causal Language Model이다.
- 따라서 문장이 자연스럽게 이어질 수는 있지만 사용자의 질문 의도를 따르거나 답변 형식을 지키는 능력은 제한적일 수 있다.
- 이번 결과는 이후 SFT 모델과 비교하기 위한 출발점이므로 동일한 평가 prompt와 생성 조건을 유지하는 것이 중요하다.
- 현재 단계에서는 model parameter를 변경하지 않았으며, 사전학습된 모델의 추론만 수행했다.

In [ ]:
# ============================================================
# Step 1-4. 원본 KoGPT-2 생성 결과 저장
# ============================================================

import json
from datetime import datetime

baseline_result = {
    "model": MODEL_ID,
    "stage": "pretrained_baseline",
    "prompt": prompt,
    "generated_text": generated_text,
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 50,
        "repetition_penalty": 2.0,
    },
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "dtype": str(next(model.parameters()).dtype),
    },
}

baseline_path = OUTPUT_DIR / "kogpt2_baseline_smoke_test.json"

with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump(
        baseline_result,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("저장 완료:", baseline_path)

저장 완료: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/outputs/gd08/kogpt2_baseline_smoke_test.json


### 원본 KoGPT-2 baseline

- 잘못 로드된 tokenizer를 수정하자 한글 입력과 생성 결과가 정상화되었다.
- KoGPT-2는 약 1억 2,500만 개의 parameter를 가진 Causal Language Model이며, 현재 단계에서는 별도의 SFT를 수행하지 않았다.
- `"인공지능이란"`이라는 입력 뒤에 문법적으로 비교적 자연스러운 한국어 문장을 이어서 생성했다.
- 그러나 `"인공지능은 인간의 뇌가 스스로 학습하는 능력"`이라고 설명하는 등 사실적으로 부정확한 내용도 함께 생성했다.
- 이를 통해 **문장을 자연스럽게 생성하는 능력과 사용자의 질문에 정확하게 답변하는 능력은 같지 않다**는 점을 확인했다.
- 이후 동일한 평가 조건에서 원본 KoGPT-2와 SFT 적용 모델을 비교하여 Instruction Tuning의 효과를 확인할 예정이다.
- 현재 결과는 이후 실험과 비교하기 위한 baseline으로 별도 저장했다.

In [ ]:
# ============================================================
# Step 2-1. KoChatGPT 원본 학습 데이터 내려받기
# ============================================================

import requests
from pathlib import Path

DATA_URLS = {
    "SFT": (
        "https://raw.githubusercontent.com/"
        "airobotlab/KoChatGPT/main/"
        "data_kochatgpt/kochatgpt_1_SFT.jsonl"
    ),
    "RM": (
        "https://raw.githubusercontent.com/"
        "airobotlab/KoChatGPT/main/"
        "data_kochatgpt/kochatgpt_2_RM.jsonl"
    ),
    "PPO": (
        "https://raw.githubusercontent.com/"
        "airobotlab/KoChatGPT/main/"
        "data_kochatgpt/kochatgpt_3_PPO.jsonl"
    ),
}

LOCAL_FILES = {}

for name, url in DATA_URLS.items():
    filename = url.split("/")[-1]

    local_path = LOCAL_DATA / filename
    drive_path = DATA_DIR / filename

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    # 빠른 작업용 로컬 저장
    local_path.write_bytes(response.content)

    # 영구 보존용 Google Drive 저장
    drive_path.write_bytes(response.content)

    LOCAL_FILES[name] = local_path

    print(
        f"{name:<4}",
        filename,
        f"{len(response.content) / 1024**2:.2f} MB"
    )

print("\n다운로드 완료")

SFT  kochatgpt_1_SFT.jsonl 5.45 MB
RM   kochatgpt_2_RM.jsonl 9.88 MB
PPO  kochatgpt_3_PPO.jsonl 1.02 MB

다운로드 완료


In [ ]:
# ============================================================
# Step 2-2. SFT·RM·PPO 데이터 구조 확인
# ============================================================

import json

datasets_raw = {}

for name, path in LOCAL_FILES.items():
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    datasets_raw[name] = data

    print("=" * 70)
    print(f"[{name}]")
    print("데이터 수:", len(data))
    print("필드:", list(data[0].keys()))
    print("\n첫 번째 예시:")
    print(json.dumps(
        data[0],
        ensure_ascii=False,
        indent=2,
    ))
    print()

[SFT]
데이터 수: 12000
필드: ['prompt', 'completion', 'tokens']

첫 번째 예시:
{
  "prompt": "불고기용 고기 한우에요?",
  "completion": "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  "tokens": 193
}

[RM]
데이터 수: 10220
필드: ['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']

첫 번째 예시:
{
  "prompt": "번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?",
  "completion_0": "Allow me to answer your question. I know that you are curious about me.",
  "completion_1": "번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.",
  "completion_2": "라이언에게 말했다.",
  "ranking": [
    2,
    1,
    0
  ]
}

[PPO]
데이터 수: 12000
필드: ['prompt']

첫 번째 예시:
{
  "prompt": "번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?"
}



### 데이터셋 구성 이해


```
                 질문 Prompt
                     │
        ┌────────────┼────────────┐
        │            │            │
        ▼            ▼            ▼

      SFT            RM           PPO
 prompt         prompt         prompt
 completion     답변 A
                답변 B
                답변 C
                ranking
        │            │            │
        ▼            ▼            ▼

 "어떻게        "어느 답이      "현재 모델은
  답할까?"       더 좋은가?"      무엇을 생성하나?"
```
### 모델 학습 목적 다시 한 번 정리.


```
SFT
Prompt → Completion
             │
             ▼
Cross Entropy Loss


RM
Prompt + Response
             │
             ▼
Scalar Reward
             │
             ▼
chosen > rejected


PPO
Prompt
  │
  ▼
Policy가 Response 생성
  │
  ▼
Reward Model이 점수
  │
  ▼
Policy update
```




In [ ]:
# ============================================================
# Step 2-3. 학습 데이터의 기본 품질과 길이 확인
# ============================================================

import pandas as pd

sft_df = pd.DataFrame(datasets_raw["SFT"])
rm_df = pd.DataFrame(datasets_raw["RM"])
ppo_df = pd.DataFrame(datasets_raw["PPO"])

print("[데이터 크기]")
print("SFT :", sft_df.shape)
print("RM  :", rm_df.shape)
print("PPO :", ppo_df.shape)

# 문자열 길이 계산
sft_df["prompt_chars"] = (
    sft_df["prompt"]
    .fillna("")
    .astype(str)
    .str.len()
)

sft_df["completion_chars"] = (
    sft_df["completion"]
    .fillna("")
    .astype(str)
    .str.len()
)

rm_df["prompt_chars"] = (
    rm_df["prompt"]
    .fillna("")
    .astype(str)
    .str.len()
)

for col in [
    "completion_0",
    "completion_1",
    "completion_2",
]:
    rm_df[f"{col}_chars"] = (
        rm_df[col]
        .fillna("")
        .astype(str)
        .str.len()
    )

print("\n[SFT 길이 통계]")
display(
    sft_df[
        ["prompt_chars", "completion_chars"]
    ].describe()
)

print("\n[RM 길이 통계]")
display(
    rm_df[
        [
            "prompt_chars",
            "completion_0_chars",
            "completion_1_chars",
            "completion_2_chars",
        ]
    ].describe()
)

print("\n[SFT 결측치]")
print(
    sft_df[
        ["prompt", "completion"]
    ].isna().sum()
)

print("\n[SFT 중복 prompt]")
print(sft_df["prompt"].duplicated().sum())

print("\n[RM 중복 prompt]")
print(rm_df["prompt"].duplicated().sum())

[데이터 크기]
SFT : (12000, 3)
RM  : (10220, 5)
PPO : (12000, 1)

[SFT 길이 통계]


,prompt_chars,completion_chars
count,12000.000000,12000.000000
mean,22.180583,144.107250
std,14.110028,122.843692
min,0.000000,4.000000
25%,13.000000,62.000000
50%,19.000000,118.000000
75%,28.000000,185.000000
max,295.000000,1553.000000



[RM 길이 통계]


,prompt_chars,completion_0_chars,completion_1_chars,completion_2_chars
count,10220.000000,10220.000000,10220.000000,10220.000000
mean,22.203229,117.493151,116.805675,116.005479
std,14.297097,120.476514,126.148677,120.154507
min,0.000000,0.000000,0.000000,0.000000
25%,13.000000,42.000000,42.000000,41.000000
50%,19.000000,102.000000,100.500000,99.500000
75%,28.000000,143.000000,143.000000,142.000000
max,295.000000,3088.000000,3694.000000,2979.000000



[SFT 결측치]
prompt        0
completion    0
dtype: int64

[SFT 중복 prompt]
54

[RM 중복 prompt]
39


In [ ]:
# ============================================================
# Step 2-4. SFT 데이터의 실제 token 길이 확인
# ============================================================

from tqdm.auto import tqdm

def count_tokens(text):
    return len(
        tokenizer.encode(
            str(text),
            add_special_tokens=False,
        )
    )

tqdm.pandas()

sft_df["prompt_tokens"] = (
    sft_df["prompt"]
    .progress_apply(count_tokens)
)

sft_df["completion_tokens"] = (
    sft_df["completion"]
    .progress_apply(count_tokens)
)

sft_df["total_tokens"] = (
    sft_df["prompt_tokens"]
    + sft_df["completion_tokens"]
)

display(
    sft_df[
        [
            "prompt_tokens",
            "completion_tokens",
            "total_tokens",
        ]
    ].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

  0%|          | 0/12000 [00:00<?, ?it/s]

  0%|          | 0/12000 [00:00<?, ?it/s]

,prompt_tokens,completion_tokens,total_tokens
count,12000.000000,12000.000000,12000.000000
mean,10.579250,55.689667,66.268917
std,6.015264,48.109558,47.951665
min,0.000000,3.000000,5.000000
50%,9.000000,45.000000,56.000000
90%,17.000000,99.000000,111.000000
95%,20.000000,152.000000,162.050000
99%,29.000000,250.000000,259.000000
max,118.000000,881.000000,894.000000


#### EDA 주요 내용 정리해보면...


```
SFT 12,000개
├─ 빈 prompt 존재          ← min chars = 0
├─ 중복 prompt 54개
├─ 일반적인 전체 길이      ← median 56 tokens
├─ 95%                     ← 162 tokens 이하
├─ 99%                     ← 259 tokens 이하
└─ 극단값                   ← 최대 894 tokens

RM 10,220개
├─ 빈 prompt 가능
├─ 빈 completion 존재      ← min chars = 0
├─ 중복 prompt 39개
└─ completion 최대 3,000~3,700자
```
### 데이터 길이와 품질 분포 확인

- SFT 데이터 12,000개, RM 데이터 10,220개, PPO 데이터 12,000개를 확인했다.
- SFT의 prompt와 completion에는 결측치(`NaN`)는 없지만, 길이가 0인 빈 문자열 prompt가 존재한다.
- RM 데이터에서는 prompt뿐 아니라 completion에도 길이가 0인 응답이 존재하므로 Reward Model 학습 전에 품질 검사가 필요하다.
- SFT에는 동일한 prompt가 54개, RM에는 39개 중복되어 있어 train/validation/test 분리 시 동일 질문이 서로 다른 split에 들어가는 데이터 누수 가능성이 있다.
- SFT의 전체 길이는 중앙값 56 tokens, 95%가 약 162 tokens 이하로 비교적 짧지만 최대 894 tokens의 긴 outlier도 존재한다.
- 데이터의 평균값만 보면 긴 샘플의 존재를 놓칠 수 있으므로 median, 95%, 99%, max를 함께 확인하는 것이 중요함을 배웠다.
- 학습 길이(`max_length`)는 관습적으로 정하는 것이 아니라 실제 token 길이 분포와 GPU 연산 비용을 함께 고려해 결정해야 한다.


In [ ]:
# ============================================================
# Step 2-5. 학습 길이 후보별 데이터 포함 비율 확인
# ============================================================

length_candidates = [128, 192, 256, 320, 384, 512]

length_summary = []

for max_length in length_candidates:
    included = (sft_df["total_tokens"] <= max_length).sum()
    truncated = (sft_df["total_tokens"] > max_length).sum()

    length_summary.append({
        "max_length": max_length,
        "포함 데이터 수": included,
        "잘림 대상 수": truncated,
        "포함 비율(%)": included / len(sft_df) * 100,
        "잘림 비율(%)": truncated / len(sft_df) * 100,
    })

length_summary_df = pd.DataFrame(length_summary)

display(length_summary_df)

,max_length,포함 데이터 수,잘림 대상 수,포함 비율(%),잘림 비율(%)
0,128,11092,908,92.433333,7.566667
1,192,11601,399,96.675000,3.325000
2,256,11873,127,98.941667,1.058333
3,320,11967,33,99.725000,0.275000
4,384,11990,10,99.916667,0.083333
5,512,11998,2,99.983333,0.016667


In [ ]:
# ============================================================
# Step 2-6. 빈 문자열·중복·긴 샘플 현황 확인
# ============================================================

def is_blank(series):
    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

print("[SFT]")
print("빈 prompt      :", is_blank(sft_df["prompt"]).sum())
print("빈 completion  :", is_blank(sft_df["completion"]).sum())

print(
    "완전 중복 행    :",
    sft_df.duplicated(
        subset=["prompt", "completion"]
    ).sum()
)

print(
    "중복 prompt     :",
    sft_df["prompt"].duplicated().sum()
)

print("\n[길이 이상 후보]")
for threshold in [256, 320, 512]:
    count = (sft_df["total_tokens"] > threshold).sum()

    print(
        f"{threshold:>3} tokens 초과:",
        f"{count:>4}개",
        f"({count / len(sft_df) * 100:.2f}%)"
    )


print("\n[RM]")
print("빈 prompt       :", is_blank(rm_df["prompt"]).sum())

for col in [
    "completion_0",
    "completion_1",
    "completion_2",
]:
    print(
        f"빈 {col:<12}:",
        is_blank(rm_df[col]).sum()
    )

print(
    "중복 prompt      :",
    rm_df["prompt"].duplicated().sum()
)

[SFT]
빈 prompt      : 3
빈 completion  : 0
완전 중복 행    : 0
중복 prompt     : 54

[길이 이상 후보]
256 tokens 초과:  127개 (1.06%)
320 tokens 초과:   33개 (0.27%)
512 tokens 초과:    2개 (0.02%)

[RM]
빈 prompt       : 3
빈 completion_0: 8
빈 completion_1: 8
빈 completion_2: 14
중복 prompt      : 39


In [ ]:
# ============================================================
# Step 2-7. 동일한 질문을 기준으로 학습·검증·평가 데이터 고정
# ============================================================

import hashlib
import re


RANDOM_SEED = 42


def normalize_prompt(text):
    """
    split을 위한 최소 정규화.
    의미를 바꾸지 않고 앞뒤/중복 공백만 통일한다.
    """
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text


def assign_split(prompt):
    """
    동일하게 정규화된 prompt는 항상 동일 split으로 배정한다.
    약 80% train / 10% validation / 10% test.
    """
    normalized = normalize_prompt(prompt)

    digest = hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

    bucket = int(digest[:8], 16) % 100

    if bucket < 80:
        return "train"

    elif bucket < 90:
        return "validation"

    else:
        return "test"


sft_df["prompt_normalized"] = (
    sft_df["prompt"]
    .apply(normalize_prompt)
)

sft_df["split"] = (
    sft_df["prompt_normalized"]
    .apply(assign_split)
)


print("[Split 크기]")
print(sft_df["split"].value_counts())

print("\n[Split 비율]")
print(
    (
        sft_df["split"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)


# 동일 prompt가 여러 split에 들어갔는지 검증
prompt_split_count = (
    sft_df
    .groupby("prompt_normalized")["split"]
    .nunique()
)

leaked_prompts = (
    prompt_split_count > 1
).sum()

print(
    "\n동일 prompt의 split 중복:",
    leaked_prompts
)

[Split 크기]
split
train         9585
validation    1210
test          1205
Name: count, dtype: int64

[Split 비율]
split
train         79.88
validation    10.08
test          10.04
Name: proportion, dtype: float64

동일 prompt의 split 중복: 0


###  평가 데이터 고정과 데이터 누수 방지

- 모델 개선 효과를 공정하게 비교하려면 Raw 데이터와 Clean 데이터로 학습한 모델이 동일한 validation/test 데이터에서 평가되어야 한다.
- 동일한 prompt가 여러 응답으로 반복될 수 있으므로 단순한 row 단위 random split은 동일 질문이 train과 test에 동시에 존재하는 데이터 누수를 만들 수 있다.
- 이를 방지하기 위해 prompt의 최소 정규화 결과를 hash하여 동일 질문은 항상 하나의 split에만 들어가도록 설정했다.
- train/validation/test를 약 80/10/10으로 고정했으며 이후 데이터 정제와 hyperparameter 선택은 train과 validation을 중심으로 수행한다.
- test 데이터는 최종 모델 비교 전까지 tuning에 사용하지 않는 것을 원칙으로 한다.
- 이 과정을 통해 모델 성능 향상이 test 데이터에 맞춘 후처리 때문이 아니라 실제 학습 전략의 차이에서 발생했는지 확인할 수 있다.

### 기록 — 학습 길이와 평가 데이터 확정

- SFT 데이터의 전체 token 길이는 99%가 약 259 tokens 이하였으며, `max_length=256`으로 설정하면 전체 데이터의 98.94%를 포함할 수 있음을 확인했다.
- `max_length=320`으로 늘릴 경우 추가로 보존되는 데이터는 약 0.78%이지만 Self-Attention의 계산량은 sequence length의 제곱에 비례하므로 연산 비용은 더 크게 증가한다.
- 빠른 실험 진행과 데이터 보존율을 함께 고려하여 이번 SFT의 기본 `max_length`를 256으로 결정했다.
- SFT에는 빈 prompt 3개와 동일 prompt 54개가 존재하며, RM에는 빈 prompt 3개와 빈 completion도 여러 개 존재한다.
- train/validation/test를 약 80/10/10으로 분리했고, 동일한 prompt는 반드시 하나의 split에만 속하도록 구성했다.
- 동일 prompt가 여러 split에 나타나지 않음을 확인했으므로 이후 모델 간 비교에서 prompt 기반 데이터 누수를 방지할 수 있다.
- 이 split은 이후 실험에서 변경하지 않고, validation은 설정 선택에, test는 최종 모델 비교에만 사용한다.

In [ ]:
# ============================================================
# Step 3-1. 빈 데이터와 중복 질문의 실제 내용 확인
# ============================================================

# 빈 prompt가 어느 split에 있는지 확인
blank_sft = sft_df[
    sft_df["prompt"].fillna("").astype(str).str.strip().eq("")
]

print("[빈 prompt]")
display(
    blank_sft[
        ["prompt", "completion", "split", "total_tokens"]
    ]
)


# 동일 prompt가 두 번 이상 등장하는 그룹 추출
duplicate_prompt_mask = sft_df.duplicated(
    subset=["prompt_normalized"],
    keep=False,
)

duplicate_sft = (
    sft_df[duplicate_prompt_mask]
    .sort_values("prompt_normalized")
)

duplicate_summary = (
    duplicate_sft
    .groupby("prompt_normalized")
    .agg(
        데이터수=("completion", "size"),
        서로다른응답수=("completion", "nunique"),
        split수=("split", "nunique"),
    )
    .sort_values(
        ["데이터수", "서로다른응답수"],
        ascending=False,
    )
)

print("\n[중복 질문 요약]")
display(duplicate_summary.head(20))

print(
    "\n중복 질문 그룹 수:",
    len(duplicate_summary)
)

print(
    "응답도 서로 다른 중복 질문 그룹:",
    (duplicate_summary["서로다른응답수"] > 1).sum()
)

[빈 prompt]


,prompt,completion,split,total_tokens
1983,,"'Sorry, as an AI language model, I need more i...",train,72
4793,,"'As an AI language model, I do not have a phys...",train,138
8222,,"""As an AI language model, I don't have persona...",train,156



[중복 질문 요약]


,데이터수,서로다른응답수,split수
prompt_normalized,,,
얼마에요?,5,5,1
영수증 좀 주세요,4,4,1
이거는 얼마예요?,4,4,1
,3,3,1
화장실은 어디 있어요?,3,3,1
고백하고 후회하면 어떡하지,2,2,1
과함 설렘 후에 지금은 안 설레,2,2,1
그건 얼마예요?,2,2,1
남편이 자꾸 만져,2,2,1



중복 질문 그룹 수: 45
응답도 서로 다른 중복 질문 그룹: 45


In [ ]:
# ============================================================
# Step 3-2. 중복 질문의 응답 내용 비교
# ============================================================

sample_prompts = duplicate_summary.head(5).index

for i, duplicated_prompt in enumerate(sample_prompts, start=1):
    print("=" * 80)
    print(f"[중복 질문 {i}]")
    print(duplicated_prompt)

    examples = duplicate_sft[
        duplicate_sft["prompt_normalized"] == duplicated_prompt
    ]

    for j, completion in enumerate(
        examples["completion"].tolist(),
        start=1,
    ):
        print(f"\n응답 {j}:")
        print(completion[:700])

[중복 질문 1]
얼마에요?

응답 1:
'물건이나 서비스의 가격을 묻는 한국어 표현입니다. 하지만 구체적인 상황과 물건 종류에 따라서 달라질 수 있습니다.

응답 2:
'제가 가지고 있는 AI 챗봇이며, 사용에 따라 다른 서비스에 연동될 수 있으므로 구체적인 가격은 제공하지 않습니다. 가능한 빨리 도움을 드리도록 노력하겠습니다.

응답 3:
'제가 가격을 알 수 없습니다. 사실 무엇을 구매하려고 하는지에 따라 가격이 달라질 수 있습니다. 구체적인 제품이나 서비스의 가격을 알려주시면 더 자세한 답변을 드릴 수 있습니다.

응답 4:
'저는 가격을 말할 수 없습니다. 전 세계 많은 나라와 지역에서 서비스를 제공하고 있기 때문에 제공하는 상품 또는 서비스의 가격은 다릅니다. 따라서, 정확한 가격에 대해서는 해당 상품 또는 서비스를 제공하는 회사나 업체에 문의하시기 바랍니다.

응답 5:
'죄송합니다, 어떤 물건이나 서비스를 말씀하시는 것인지 알려주시면 가격 정보를 제공해 드릴 수 있습니다.
[중복 질문 2]
영수증 좀 주세요

응답 1:
"I'm sorry, as an AI language model, I cannot provide you with a physical receipt. Please contact the store or restaurant where you made your purchase, and they can give you a receipt.", 'token': 56}

응답 2:
"I'm sorry, I cannot provide a physical receipt as I am an AI language model and do not have the capability to issue or give a receipt. Can I assist you with anything else?", 'token': 55}

응답 3:
"I'm sorry, as an AI language model, I don't have the capability to 

In [ ]:
# ============================================================
# Step 3-4. 공통으로 사용할 유효 데이터 정의
# ============================================================

def valid_text(series):
    return ~(
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )


sft_df["is_valid"] = (
    valid_text(sft_df["prompt"])
    &
    valid_text(sft_df["completion"])
)


print("[유효성 검사]")
print(
    sft_df["is_valid"]
    .value_counts()
)

print(
    "\n제외 대상:",
    (~sft_df["is_valid"]).sum()
)


raw_train_df = (
    sft_df[
        (sft_df["split"] == "train")
        &
        (sft_df["is_valid"])
    ]
    .copy()
)

validation_df = (
    sft_df[
        (sft_df["split"] == "validation")
        &
        (sft_df["is_valid"])
    ]
    .copy()
)

test_df = (
    sft_df[
        (sft_df["split"] == "test")
        &
        (sft_df["is_valid"])
    ]
    .copy()
)


print("\n[학습/검증/평가]")
print("Raw Train :", len(raw_train_df))
print("Validation:", len(validation_df))
print("Test      :", len(test_df))

[유효성 검사]
is_valid
True     11997
False        3
Name: count, dtype: int64

제외 대상: 3

[학습/검증/평가]
Raw Train : 9582
Validation: 1210
Test      : 1205


In [ ]:
# ============================================================
# Step 3-5. 응답의 반복과 언어 구성 이상 후보 탐색
# ============================================================

import re


def korean_ratio(text):
    """
    공백을 제외한 문자 중 한글 음절이 차지하는 비율.
    절대적인 품질 판정이 아니라 검토 후보를 찾기 위한 지표다.
    """
    text = str(text)

    chars = [
        ch for ch in text
        if not ch.isspace()
    ]

    if not chars:
        return 0.0

    korean_chars = sum(
        1 for ch in chars
        if "가" <= ch <= "힣"
    )

    return korean_chars / len(chars)


def repetition_ratio(text):
    """
    공백 기준 단어 중 가장 많이 반복된 단어의 비율.
    반복이 심한 후보를 탐색하기 위한 단순 지표다.
    """
    words = str(text).split()

    if not words:
        return 1.0

    counts = pd.Series(words).value_counts()

    return counts.iloc[0] / len(words)


raw_train_df["korean_ratio"] = (
    raw_train_df["completion"]
    .apply(korean_ratio)
)

raw_train_df["repetition_ratio"] = (
    raw_train_df["completion"]
    .apply(repetition_ratio)
)


print("[한글 비율]")
display(
    raw_train_df["korean_ratio"]
    .describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    )
)

print("\n[단어 반복 비율]")
display(
    raw_train_df["repetition_ratio"]
    .describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

[한글 비율]


,korean_ratio
count,9582.000000
mean,0.835332
std,0.195574
min,0.000000
1%,0.000000
5%,0.392857
50%,0.913580
95%,0.957447
99%,0.965041
max,0.976562



[단어 반복 비율]


,repetition_ratio
count,9582.000000
mean,0.106604
std,0.141560
min,0.014493
50%,0.068182
90%,0.181818
95%,0.250000
99%,1.000000
max,1.000000


In [ ]:
# ============================================================
# Step 3-6. 품질 이상 후보의 실제 문장 확인
# ============================================================

print("[한글 비율이 낮은 응답 후보]")

display(
    raw_train_df
    .sort_values("korean_ratio")
    [
        [
            "prompt",
            "completion",
            "korean_ratio",
            "completion_tokens",
        ]
    ]
    .head(20)
)


print("\n[반복 비율이 높은 응답 후보]")

display(
    raw_train_df
    .sort_values(
        "repetition_ratio",
        ascending=False,
    )
    [
        [
            "prompt",
            "completion",
            "repetition_ratio",
            "completion_tokens",
        ]
    ]
    .head(20)
)

[한글 비율이 낮은 응답 후보]


,prompt,completion,korean_ratio,completion_tokens
6476,음료는 나가서 마실 거라 플라스틱 용기에 주세요,"'As an AI language model, I don\'t drink bever...",0.0,95
3072,우린 운명이었어,"'As an AI language model, I do not have person...",0.0,63
3507,방탄소년단의 두 번째 정규 앨범의 이름은?,'WINGS.,0.0,5
11928,영수증 좀 주세요,"""I'm sorry, I cannot provide a physical receip...",0.0,89
5919,예전엔 알바분이 해줬는데요?,"""I'm sorry, I am an AI language model and I do...",0.0,91
7326,첫눈이 올때 같이 있고 싶다,'I want to be with you when the first snow falls.,0.0,24
1577,그녀도 나를 좋아했으면 좋겠는데.,"'As an AI language model, I cannot determine t...",0.0,117
10349,일시불로 결제해주세요,'Please pay in full at once.,0.0,13
3934,상황이 어찌됐던 간에 내가 못잊겠으면 연락을 했겠죠.,"""However the situation turns out, if I can't f...",0.0,50
2175,금수저로 태어났으면 좋았을텐데,'I wish I had been born into a wealthy family.,0.0,25



[반복 비율이 높은 응답 후보]


,prompt,completion,repetition_ratio,completion_tokens
1409,시리아 민간인 보호를 목표로 한 안보리 결의안에 거부권을 행사한 나라는?,'러시아입니다.,1.0,3
1351,에이브러햄 에이드리언 앨버트가 석사 학위를 수여받은 해는,'1871년입니다.,1.0,5
11888,송무억문의 무사 정권이 수립된 연도는?,'1868년입니다.,1.0,5
9942,테트리스가 처음 발표된 연도는,'1984년입니다.,1.0,5
9,리처드 닉슨이 43대 부통령직을 수행한 년도는?,'1953년입니다.,1.0,5
17,2000년 5월 허위사실유포죄를 위헌판정한 나라는?,'한국입니다.,1.0,3
11896,오세훈 시장이 민선 5기 지방선거에서 서울시장 재선에 도전한 년도는?,'2018년입니다.,1.0,4
2800,"1880년 여름 윤융렬, 이용숙, 지석영 등을 이끌고 수신사로 일본에 파견되었던 인물은?",'신덕수입니다.,1.0,4
9676,"도나우버트에서 미샤엘, 막스와 함께 인형 제작을 맡은 사람은?",'실리아입니다.,1.0,4
5057,경선 중 이명박 측이 15% 격차로 압승할 것으로 예상하고 있다고 발언한 인물은?,'정세균.,1.0,4


In [ ]:
# ============================================================
# Step 3-7. 긴 응답에서 반복되는 구절 확인
# ============================================================

from collections import Counter


def ngram_repetition_ratio(text, n=3):
    """
    긴 문장에서 동일한 n-gram이 반복되는 정도를 확인한다.
    짧은 정답은 반복 데이터로 오판하지 않도록 0으로 처리한다.
    """
    words = str(text).split()

    # 짧은 답변은 반복률 평가 대상에서 제외
    if len(words) < max(10, n * 2):
        return 0.0

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

    if not ngrams:
        return 0.0

    counts = Counter(ngrams)

    repeated_count = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated_count / len(ngrams)


raw_train_df["ngram_repetition_ratio"] = (
    raw_train_df["completion"]
    .apply(ngram_repetition_ratio)
)

print("[3-gram 반복 비율]")

display(
    raw_train_df["ngram_repetition_ratio"]
    .describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)


print("\n[반복 구절이 많은 응답 후보]")

display(
    raw_train_df
    .sort_values(
        "ngram_repetition_ratio",
        ascending=False,
    )
    [
        [
            "prompt",
            "completion",
            "ngram_repetition_ratio",
            "completion_tokens",
        ]
    ]
    .head(20)
)

[3-gram 반복 비율]


,ngram_repetition_ratio
count,9582.000000
mean,0.002412
std,0.012561
min,0.000000
50%,0.000000
90%,0.000000
95%,0.016949
99%,0.055556
max,0.272727



[반복 구절이 많은 응답 후보]


,prompt,completion,ngram_repetition_ratio,completion_tokens
3505,Baby One More Time의 내용을 알려줘,"'Britney Spears의 ""Baby One More Time""은 첫 번째 앨범...",0.272727,881
7843,Bass는 리더와 구성원의 일체감을 강조하는 Burns에 비해 어떤 차이를 보였어,'요약: Burns는 리더와 구성원을 일체감을 강화하며 조직 내의 모든 구성원들이 ...,0.251908,213
2393,"""제시카 수타는 그룹에서 '쫓겨났다'고 폭로한 곳은 어디인가?"", 'completi...","{'prompt': ""제시카 수타는 그룹에서 '쫓겨났다'고 폭로한 곳은 어디인가?""...",0.250000,64
3450,모든 노래 가사는 내 맘같아,'너의 눈 속에 미치도록 빠져들었어\n내 맘은 녹아 내리듯이 녹아내렸어\n모든 게 ...,0.240000,217
5792,"바이오쇼크가 다양한 게임 전문 사이트에서 몇 개의 ""Game of the Show""...","'바이오쇼크가 게임전문 사이트에서 ""Game of the Show"" 상을 수상한 년...",0.204301,281
9285,세대별로 온도조절이 가능한 난방 방식은 무엇인가?,"'1. 전기난방: 전기로 발생한 열을 이용해 방을 따뜻하게 하는 난방 방식으로, 각...",0.200000,133
568,라이징 폼은 어떤 것이 더 강화됐지,"'라이징 폼이 어떤 것이 강화되었는지는 문맥에 따라 다를 수 있습니다. 일반적으로,...",0.194030,109
11798,기준 신호의 일부분은 어떤 형태로 NTSC 신호에서 각 수평 라인의 백 포치 상에 ...,'기준 신호는 NTSC 신호에서 수평 동기화 신호를 생성하기 위해 사용되는 부호화된...,0.182927,149
9294,"""푸시캣 돌스의 네번 째 상글 'Buttons'는 누구와 작업을 한 곡인가?"", '...","{'prompt': ""푸시캣 돌스의 네번 째 상글 'Buttons'는 누구와 작업을...",0.181818,78
3844,바이오쇼크의 엑스박스 360 버전의 평균 리뷰 점수는 플랫폼 내에서 몇 번째로 높은...,'바이오쇼크의 Xbox 360 버전은 플랫폼 내에서 매우 인기가 높은 게임 중 하나...,0.148936,87


### 정제 지표도 검증이 필요함

- 처음 정의한 단어 반복 비율은 한 단어짜리 정상 정답도 반복률 1.0으로 계산하는 문제가 있음을 발견했다.
- 데이터 정제에서는 통계값만 보고 삭제 기준을 적용하지 않고 실제 문장을 함께 확인해야 한다.
- 특히 짧은 사실형 QA와 비정상적으로 긴 반복 생성 문장을 같은 기준으로 평가하면 정상 데이터가 제거될 수 있다.
- 이를 보완하기 위해 일정 길이 이상의 응답에서 동일한 3-gram이 반복되는 정도를 측정하도록 지표를 수정했다.
- EDA는 단순히 데이터를 요약하는 과정이 아니라 이후 전처리 규칙 자체가 타당한지 검증하는 과정임을 확인했다.

In [ ]:
# ============================================================
# Step 3-8. 영어 쓸모떼기 없는 상투 응답 탐지
# ============================================================

import re


ENGLISH_BOILERPLATE_PATTERNS = [
    r"\bas an ai language model\b",
    r"\bas a language model\b",
    r"\bi am an ai language model\b",
    r"\bi'm an ai language model\b",
    r"\bi cannot\b",
    r"\bi can't\b",
    r"\bi do not have\b",
    r"\bi don't have\b",
    r"\bi'm sorry\b",
    r"\bi am sorry\b",
]


def has_english_boilerplate(text):
    text_lower = str(text).lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in ENGLISH_BOILERPLATE_PATTERNS
    )


raw_train_df["english_boilerplate"] = (
    raw_train_df["completion"]
    .apply(has_english_boilerplate)
)


print(
    "영어식 AI 상투 응답:",
    raw_train_df["english_boilerplate"].sum()
)

display(
    raw_train_df[
        raw_train_df["english_boilerplate"]
    ][
        [
            "prompt",
            "completion",
            "korean_ratio",
            "completion_tokens",
        ]
    ].head(20)
)

영어식 AI 상투 응답: 110


,prompt,completion,korean_ratio,completion_tokens
259,참 답 없는 나란 놈.,'I am a language model AI and I do not have th...,0.000000,91
265,여기는 몇 시부터 몇 시까지 문 열어요?,"""I'm sorry, I cannot answer this question as I...",0.000000,99
327,짝녀랑 조금이라도 닮으면 눈이 가는 듯.,"'As an AI language model, I cannot have a pref...",0.000000,75
369,좋아하는 사람만 보면 감정이 북받쳐올라.,"""As an AI language model, I do not have person...",0.000000,83
495,여기 있는 거 다 핑크예요?,"'Sorry, as an AI language model, I do not have...",0.000000,72
668,좋아하는 애랑 같이 있으면 나만 이래?,"""As an AI language model, I don't have prefere...",0.000000,142
732,찰토마토 두 박스 주세요.,"'As an AI language model, I don\'t have the ca...",0.062112,96
756,내가 더 좋아하는 거 같은 느낌이야.,"'As an AI language model, I do not have person...",0.000000,87
805,얼음은 조금 주시고 콜라는 많이 주세요,"""I'm sorry, as an AI language model, I cannot ...",0.000000,94
945,헤어진건 아닌데,"""Sorry, as an AI language model, I don't have ...",0.000000,87


In [ ]:
# ============================================================
# Step 3-9. 영어 중심의 긴 응답 후보 확인
# ============================================================

raw_train_df["english_heavy_long"] = (
    (raw_train_df["korean_ratio"] < 0.20)
    &
    (raw_train_df["completion_tokens"] >= 12)
)


print(
    "영어 중심 장문 응답:",
    raw_train_df["english_heavy_long"].sum()
)


display(
    raw_train_df[
        raw_train_df["english_heavy_long"]
    ][
        [
            "prompt",
            "completion",
            "korean_ratio",
            "completion_tokens",
        ]
    ].head(30)
)

영어 중심 장문 응답: 261


,prompt,completion,korean_ratio,completion_tokens
25,목 마르다.,"'> ""I am thirsty."" (English) \n\n> ""Je suis as...",0.053435,95
50,동거 8년 환승이별.,"'8 years of living together, transfer separation.",0.000000,21
91,오늘도 그냥 그냥 다갔네.,'Translation: Today just passed by like any ot...,0.000000,26
215,이걸로 하나 주세요.,"'번역: Can I get one of these, please?",0.068966,17
259,참 답 없는 나란 놈.,'I am a language model AI and I do not have th...,0.000000,91
265,여기는 몇 시부터 몇 시까지 문 열어요?,"""I'm sorry, I cannot answer this question as I...",0.000000,99
277,참다가 연락,"""을 바로 하세요.\n\nIt's better to contact them soon...",0.064516,57
298,언젠간 내 기억 속에서 서서히 지워지겠지.,"'Eventually, memories of me will gradually fad...",0.000000,33
327,짝녀랑 조금이라도 닮으면 눈이 가는 듯.,"'As an AI language model, I cannot have a pref...",0.000000,75
366,네덜란드에서 정당 명부 가장 처음에 있는 정당인을 이르는 말은,"'""VVD (People\'s Party for Freedom and Democra...",0.062500,27


In [ ]:
# ============================================================
# Step 3-10. 긴 응답과 심한 반복 응답 후보 표시
# ============================================================

raw_train_df["too_long"] = (
    raw_train_df["total_tokens"] > 256
)

# 실제 분포를 보수적으로 잡기 위해 높은 반복률만 후보 처리
raw_train_df["severe_repetition"] = (
    raw_train_df["ngram_repetition_ratio"] >= 0.30
)


print(
    "256 tokens 초과:",
    raw_train_df["too_long"].sum()
)

print(
    "심한 반복 후보:",
    raw_train_df["severe_repetition"].sum()
)

256 tokens 초과: 104
심한 반복 후보: 0


In [ ]:
# ============================================================
# Step 3-11. 정제 기준을 적용한 학습 데이터 생성
# ============================================================

clean_train_df = raw_train_df.copy()


clean_train_df["remove_reason"] = ""


def add_reason(mask, reason):
    current = clean_train_df.loc[mask, "remove_reason"]

    clean_train_df.loc[mask, "remove_reason"] = (
        current
        .apply(
            lambda x:
            f"{x}, {reason}" if x else reason
        )
    )


add_reason(
    clean_train_df["english_boilerplate"],
    "영어 AI 상투 응답",
)

add_reason(
    clean_train_df["english_heavy_long"],
    "영어 중심 장문",
)

add_reason(
    clean_train_df["severe_repetition"],
    "반복 구절 과다",
)

add_reason(
    clean_train_df["too_long"],
    "256 tokens 초과",
)


removed_train_df = clean_train_df[
    clean_train_df["remove_reason"] != ""
].copy()

clean_train_df = clean_train_df[
    clean_train_df["remove_reason"] == ""
].copy()


print("[정제 전]")
print(len(raw_train_df))

print("\n[정제 제외]")
print(len(removed_train_df))

print("\n[정제 후]")
print(len(clean_train_df))

print(
    "\n데이터 유지율:",
    f"{len(clean_train_df) / len(raw_train_df) * 100:.2f}%"
)


print("\n[제거 이유별 수]")
print(
    removed_train_df["remove_reason"]
    .value_counts()
    .head(20)
)

[정제 전]
9582

[정제 제외]
359

[정제 후]
9223

데이터 유지율: 96.25%

[제거 이유별 수]
remove_reason
영어 중심 장문                                148
영어 AI 상투 응답, 영어 중심 장문                   107
256 tokens 초과                            98
영어 중심 장문, 256 tokens 초과                   3
영어 AI 상투 응답, 영어 중심 장문, 256 tokens 초과      3
Name: count, dtype: int64


In [ ]:
# ============================================================
# Step 3-12. 정제 과정에서 제외된 실제 문장 확인
# ============================================================

display(
    removed_train_df[
        [
            "prompt",
            "completion",
            "remove_reason",
            "total_tokens",
            "korean_ratio",
            "ngram_repetition_ratio",
        ]
    ].head(50)
)

,prompt,completion,remove_reason,total_tokens,korean_ratio,ngram_repetition_ratio
25,목 마르다.,"'> ""I am thirsty."" (English) \n\n> ""Je suis as...",영어 중심 장문,98,0.053435,0.000000
50,동거 8년 환승이별.,"'8 years of living together, transfer separation.",영어 중심 장문,27,0.000000,0.000000
91,오늘도 그냥 그냥 다갔네.,'Translation: Today just passed by like any ot...,영어 중심 장문,33,0.000000,0.000000
128,별기군과 구식군대 간의 차별로 발생한 사건은?,"'이 질문에서 언급하는 ""별기군""과 ""구식군대""는 어떤 군사력을 말하는 것인가요? ...",256 tokens 초과,264,0.880157,0.000000
215,이걸로 하나 주세요.,"'번역: Can I get one of these, please?",영어 중심 장문,23,0.068966,0.000000
259,참 답 없는 나란 놈.,'I am a language model AI and I do not have th...,"영어 AI 상투 응답, 영어 중심 장문",98,0.000000,0.000000
265,여기는 몇 시부터 몇 시까지 문 열어요?,"""I'm sorry, I cannot answer this question as I...","영어 AI 상투 응답, 영어 중심 장문",110,0.000000,0.000000
277,참다가 연락,"""을 바로 하세요.\n\nIt's better to contact them soon...",영어 중심 장문,60,0.064516,0.000000
298,언젠간 내 기억 속에서 서서히 지워지겠지.,"'Eventually, memories of me will gradually fad...",영어 중심 장문,45,0.000000,0.000000
325,부자 되게 해주세요,"'저는 AI 어시스턴트로서 부동산, 주식, 사업 등 부의 창출 방법에 대한 조언을 ...",256 tokens 초과,276,0.892256,0.027933


### raw와 clean 실험의 차이...



```
M0
Pretrained KoGPT-2
학습 없음


M1
Raw SFT
────────────────
유효한 원본 Train
9,582 samples
max_length=256
긴 문장은 truncation


M2
Clean SFT
────────────────
M1에서
- 영어 boilerplate
- 영어 중심 장문
- 심한 반복
- >256 tokens
제거
```



### 이렇게 해두면~

M0 → M1
= SFT 효과

M1 → M2
= 데이터 정제 효과

## 최종 정제 규칙을 정해보겠습니다.



```
Raw Train
9,582개
   │
   ├─ 영어 AI 상투 응답
   │   "As an AI language model..."
   │   "I'm sorry..."
   │
   └─ 한국어 prompt에 대한
       영어 중심 장문 응답
       korean_ratio < 0.20
       AND completion_tokens >= 12

                ↓

Clean Train
예상 9,321개
약 97.28% 유지
```



In [ ]:
# ============================================================
# Step 3-13. 최종 정제 기준으로 학습 데이터 확정
# ============================================================

# 최종 정제에서는 언어 오염과 영어식 AI 상투 응답만 제거한다.
# 긴 sequence는 삭제하지 않고 학습 시 max_length=256으로 동일하게 자른다.
# 3-gram 반복은 실제 심각한 후보가 없어 제거 기준에서 제외한다.

clean_train_df = raw_train_df.copy()

clean_train_df["remove_reason"] = ""


def add_reason(mask, reason):
    current = clean_train_df.loc[mask, "remove_reason"]

    clean_train_df.loc[mask, "remove_reason"] = (
        current.apply(
            lambda x: f"{x}, {reason}" if x else reason
        )
    )


add_reason(
    clean_train_df["english_boilerplate"],
    "영어 AI 상투 응답",
)

# boilerplate와 겹칠 수 있지만 이유를 별도로 기록한다.
add_reason(
    clean_train_df["english_heavy_long"],
    "영어 중심 장문",
)


removed_train_df = clean_train_df[
    clean_train_df["remove_reason"] != ""
].copy()

clean_train_df = clean_train_df[
    clean_train_df["remove_reason"] == ""
].copy()


print("[최종 정제 결과]")
print("정제 전 :", len(raw_train_df))
print("제외    :", len(removed_train_df))
print("정제 후 :", len(clean_train_df))

print(
    "유지율  :",
    f"{len(clean_train_df) / len(raw_train_df) * 100:.2f}%"
)

print("\n[제거 이유]")
print(
    removed_train_df["remove_reason"]
    .value_counts()
)

[최종 정제 결과]
정제 전 : 9582
제외    : 261
정제 후 : 9321
유지율  : 97.28%

[제거 이유]
remove_reason
영어 중심 장문                 151
영어 AI 상투 응답, 영어 중심 장문    110
Name: count, dtype: int64


In [ ]:
# ============================================================
# Step 3-14. 확정된 학습·검증·평가 데이터 저장
# ============================================================

import json


def save_records(df, path, columns):
    records = (
        df[columns]
        .to_dict(orient="records")
    )

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"{path.name:<30}",
        f"{len(records):>6}개"
    )


RAW_TRAIN_PATH = DATA_DIR / "sft_raw_train.json"
CLEAN_TRAIN_PATH = DATA_DIR / "sft_clean_train.json"
VALIDATION_PATH = DATA_DIR / "sft_validation.json"
TEST_PATH = DATA_DIR / "sft_test.json"
REMOVED_PATH = DATA_DIR / "sft_removed_samples.json"


base_columns = [
    "prompt",
    "completion",
]

save_records(
    raw_train_df,
    RAW_TRAIN_PATH,
    base_columns,
)

save_records(
    clean_train_df,
    CLEAN_TRAIN_PATH,
    base_columns,
)

save_records(
    validation_df,
    VALIDATION_PATH,
    base_columns,
)

save_records(
    test_df,
    TEST_PATH,
    base_columns,
)

save_records(
    removed_train_df,
    REMOVED_PATH,
    [
        "prompt",
        "completion",
        "remove_reason",
    ],
)

sft_raw_train.json               9582개
sft_clean_train.json             9321개
sft_validation.json              1210개
sft_test.json                    1205개
sft_removed_samples.json          261개


# M1 Raw SFT!

## 4. Raw SFT 학습

지금까지는 사전학습 KoGPT-2의 추론과 데이터 분석만 수행했으며 모델 parameter는 변경하지 않았다.

이 단계부터 `prompt → completion` 쌍을 이용한 Supervised Fine-Tuning을 수행하여 KoGPT-2의 parameter를 업데이트한다.

첫 번째 학습은 정제 효과의 비교 기준을 만들기 위해 Clean 데이터가 아닌 최소 무결성 처리만 적용한 Raw 학습 데이터로 수행한다.


질문 문장을 외워서 재생하는 것보다 질문 뒤에 어떤 답을 생성해야 하는지 학습시키자.

In [ ]:
# ============================================================
# Step 4-1. Raw SFT 학습용 데이터 준비
# ============================================================

from datasets import Dataset

MAX_LENGTH = 256


raw_train_dataset = Dataset.from_pandas(
    raw_train_df[
        ["prompt", "completion"]
    ].reset_index(drop=True),
    preserve_index=False,
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        ["prompt", "completion"]
    ].reset_index(drop=True),
    preserve_index=False,
)

print(raw_train_dataset)
print(validation_dataset)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 9582
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1210
})


In [ ]:
# ============================================================
# Step 3-15. 공정한 비교를 위한 검증·평가 데이터 확정
# ============================================================

def add_evaluation_quality_flags(df):
    result = df.copy()

    result["korean_ratio"] = (
        result["completion"]
        .apply(korean_ratio)
    )

    # completion_tokens는 기존 EDA에서 이미 존재하지만
    # 혹시 없을 경우 다시 계산한다.
    if "completion_tokens" not in result.columns:
        result["completion_tokens"] = (
            result["completion"]
            .apply(count_tokens)
        )

    result["english_boilerplate"] = (
        result["completion"]
        .apply(has_english_boilerplate)
    )

    result["english_heavy_long"] = (
        (result["korean_ratio"] < 0.20)
        &
        (result["completion_tokens"] >= 12)
    )

    result["evaluation_keep"] = ~(
        result["english_boilerplate"]
        |
        result["english_heavy_long"]
    )

    return result


validation_quality_df = add_evaluation_quality_flags(
    validation_df
)

test_quality_df = add_evaluation_quality_flags(
    test_df
)


validation_eval_df = (
    validation_quality_df[
        validation_quality_df["evaluation_keep"]
    ]
    .copy()
)

test_eval_df = (
    test_quality_df[
        test_quality_df["evaluation_keep"]
    ]
    .copy()
)


print("[Validation]")
print("원본 :", len(validation_df))
print("평가 :", len(validation_eval_df))
print(
    "유지율:",
    f"{len(validation_eval_df) / len(validation_df) * 100:.2f}%"
)

print("\n[Test]")
print("원본 :", len(test_df))
print("평가 :", len(test_eval_df))
print(
    "유지율:",
    f"{len(test_eval_df) / len(test_df) * 100:.2f}%"
)

[Validation]
원본 : 1210
평가 : 1181
유지율: 97.60%

[Test]
원본 : 1205
평가 : 1167
유지율: 96.85%


In [ ]:
# ============================================================
# Step 3-16. 확정된 평가 데이터 저장
# ============================================================

VALIDATION_EVAL_PATH = (
    DATA_DIR / "sft_validation_eval.json"
)

TEST_EVAL_PATH = (
    DATA_DIR / "sft_test_eval.json"
)

save_records(
    validation_eval_df,
    VALIDATION_EVAL_PATH,
    ["prompt", "completion"],
)

save_records(
    test_eval_df,
    TEST_EVAL_PATH,
    ["prompt", "completion"],
)

sft_validation_eval.json         1181개
sft_test_eval.json               1167개


### 평가 reference 품질 고정

- train/validation/test의 prompt 기반 split 자체는 변경하지 않았다.
- 원본 데이터에 한국어 질문과 맞지 않는 영어 completion이 포함되어 있어 그대로 평가 reference로 사용하면 모델 품질을 잘못 측정할 가능성이 있음을 확인했다.
- Raw SFT와 Clean SFT 모두 동일한 기준으로 정제된 validation/test에서 평가하도록 평가 조건을 고정했다.
- 평가 reference의 정제는 모델의 예측 결과를 확인하기 전에 사전에 정의한 데이터 품질 규칙만 사용했다.
- 따라서 이후 M1과 M2의 성능 차이는 서로 다른 평가 데이터가 아니라 학습 데이터 차이에서 비교할 수 있다.

In [ ]:
# ============================================================
# Step 4-2. SFT 입력 문장과 정답 label 구성
# ============================================================

import torch
from torch.utils.data import Dataset as TorchDataset


MAX_LENGTH = 256

PROMPT_TEMPLATE = (
    "### Instruction(명령어):\n"
    "{prompt}\n\n"
    "### Response(응답):"
)


class SupervisedSFTDataset(TorchDataset):

    def __init__(
        self,
        df,
        tokenizer,
        max_length=256,
    ):
        self.samples = []
        self.dropped_samples = 0

        for row in df.itertuples(index=False):

            # 모델에 실제로 입력할 Instruction 부분
            source = PROMPT_TEMPLATE.format(
                prompt=row.prompt
            )

            # 모델이 생성하도록 학습할 정답 부분
            target = (
                str(row.completion)
                + tokenizer.eos_token
            )

            full_text = source + target

            # 전체 입력: Instruction + Response
            full_ids = tokenizer(
                full_text,
                add_special_tokens=False,
                truncation=True,
                max_length=max_length,
            )["input_ids"]

            # Instruction 길이를 따로 계산한다.
            source_ids = tokenizer(
                source,
                add_special_tokens=False,
                truncation=True,
                max_length=max_length,
            )["input_ids"]

            input_ids = torch.tensor(
                full_ids,
                dtype=torch.long,
            )

            labels = input_ids.clone()

            source_length = min(
                len(source_ids),
                len(labels),
            )

            # Instruction은 정답 예측 대상이 아니므로
            # Cross Entropy Loss에서 제외한다.
            labels[:source_length] = -100

            # truncation으로 Response가 완전히 사라진 경우 제외
            if not torch.any(labels != -100):
                self.dropped_samples += 1
                continue

            self.samples.append({
                "input_ids": input_ids,
                "labels": labels,
            })


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, index):
        return self.samples[index]

In [ ]:
# ============================================================
# Step 4-3. Raw 학습·검증 데이터 tokenization
# ============================================================

raw_sft_train_dataset = SupervisedSFTDataset(
    raw_train_df,
    tokenizer,
    max_length=MAX_LENGTH,
)

sft_validation_dataset = SupervisedSFTDataset(
    validation_eval_df,
    tokenizer,
    max_length=MAX_LENGTH,
)


print("[Raw Train]")
print("원본 데이터 :", len(raw_train_df))
print("학습 데이터 :", len(raw_sft_train_dataset))
print(
    "Response 소실로 제외:",
    raw_sft_train_dataset.dropped_samples
)

print("\n[Validation]")
print("평가 데이터 :", len(sft_validation_dataset))
print(
    "Response 소실로 제외:",
    sft_validation_dataset.dropped_samples
)

[Raw Train]
원본 데이터 : 9582
학습 데이터 : 9582
Response 소실로 제외: 0

[Validation]
평가 데이터 : 1181
Response 소실로 제외: 0


In [ ]:
# ============================================================
# Step 4-4. SFT에서 학습되는 문장 영역 확인
# ============================================================

sample = raw_sft_train_dataset[0]

input_ids = sample["input_ids"]
labels = sample["labels"]


full_text = tokenizer.decode(
    input_ids,
    skip_special_tokens=False,
)

target_ids = labels[
    labels != -100
]

target_text = tokenizer.decode(
    target_ids,
    skip_special_tokens=False,
)


print("[모델 전체 입력]")
print(full_text)

print("\n[Loss가 계산되는 Response]")
print(target_text)

print("\n[Token 수]")
print("전체 입력 :", len(input_ids))
print(
    "Loss 제외 :",
    int((labels == -100).sum())
)
print(
    "Loss 계산 :",
    int((labels != -100).sum())
)

[모델 전체 입력]
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>

[Loss가 계산되는 Response]
'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>

[Token 수]
전체 입력 : 107
Loss 제외 : 32
Loss 계산 : 75


### SFT에서 `-100`이 하는 역할

- 원본 KoChatGPT의 SFT 구조를 유지하여 `Instruction → Response` 형식으로 데이터를 구성했다.
- Causal Language Model은 기본적으로 모든 위치의 다음 token을 예측할 수 있지만, 이번 SFT에서는 Response 부분에 대해서만 loss를 계산한다.
- Instruction에 해당하는 label을 `-100`으로 설정하면 PyTorch의 Cross Entropy Loss에서 해당 위치가 무시된다.
- 따라서 모델은 질문 문장을 외워 다시 생성하는 것이 아니라 질문을 조건으로 적절한 Response token의 확률을 높이는 방향으로 학습한다.
- 이 단계부터 사전학습 KoGPT-2의 parameter `θ`가 실제로 업데이트된다.

In [ ]:
# ============================================================
# Step 4-5. 배치 단위 padding을 위한 Data Collator 정의
# ============================================================

from dataclasses import dataclass
from typing import Sequence


@dataclass
class SFTDataCollator:

    tokenizer: object

    def __call__(self, instances: Sequence[dict]):

        input_ids = [
            item["input_ids"]
            for item in instances
        ]

        labels = [
            item["labels"]
            for item in instances
        ]

        # 입력 token은 PAD token으로 채운다.
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )

        # label의 padding 영역은 loss에서 제외한다.
        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        attention_mask = (
            input_ids != self.tokenizer.pad_token_id
        ).long()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


sft_data_collator = SFTDataCollator(
    tokenizer=tokenizer
)

In [ ]:
# ============================================================
# Step 4-6. L4에서 안전한 학습 batch size 측정
# ============================================================

import gc

from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM


# 긴 샘플 중심으로 테스트하여
# 실제 학습보다 보수적으로 VRAM을 확인한다.
probe_df = (
    raw_train_df
    .sort_values(
        "total_tokens",
        ascending=False,
    )
    .head(128)
)

probe_dataset = SupervisedSFTDataset(
    probe_df,
    tokenizer,
    max_length=MAX_LENGTH,
)


def probe_batch_size(batch_size):

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    probe_model = None
    optimizer = None

    try:
        probe_model = (
            AutoModelForCausalLM
            .from_pretrained(
                MODEL_ID,
                cache_dir=str(HF_CACHE),
            )
            .to("cuda")
        )

        probe_model.config.use_cache = False
        probe_model.train()

        optimizer = torch.optim.AdamW(
            probe_model.parameters(),
            lr=5e-5,
        )

        loader = DataLoader(
            probe_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=sft_data_collator,
        )

        batch = next(iter(loader))

        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            outputs = probe_model(**batch)
            loss = outputs.loss

        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()

        peak_allocated = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        result = {
            "batch_size": batch_size,
            "success": True,
            "loss": float(loss.detach().cpu()),
            "peak_allocated_gb": peak_allocated,
            "peak_reserved_gb": peak_reserved,
        }

    except torch.OutOfMemoryError:

        result = {
            "batch_size": batch_size,
            "success": False,
            "loss": None,
            "peak_allocated_gb": None,
            "peak_reserved_gb": None,
        }

    finally:

        if optimizer is not None:
            del optimizer

        if probe_model is not None:
            del probe_model

        if "batch" in locals():
            del batch

        if "outputs" in locals():
            del outputs

        gc.collect()
        torch.cuda.empty_cache()

    return result


probe_results = []

for batch_size in [16, 32, 64]:

    result = probe_batch_size(
        batch_size
    )

    probe_results.append(result)

    print(result)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'batch_size': 16, 'success': True, 'loss': 3.5670297145843506, 'peak_allocated_gb': 6.741294860839844, 'peak_reserved_gb': 6.873046875}


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'batch_size': 32, 'success': True, 'loss': 3.5677714347839355, 'peak_allocated_gb': 12.530845642089844, 'peak_reserved_gb': 12.703125}


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'batch_size': 64, 'success': False, 'loss': None, 'peak_allocated_gb': None, 'peak_reserved_gb': None}


In [ ]:
# ============================================================
# Step 4-7. 실제 VRAM 사용량을 기준으로 batch size 확정
# ============================================================

TOTAL_VRAM_GB = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / 1024**3
)

SAFE_LIMIT_GB = (
    TOTAL_VRAM_GB * 0.80
)


safe_results = [
    result
    for result in probe_results
    if (
        result["success"]
        and
        result["peak_reserved_gb"]
        <= SAFE_LIMIT_GB
    )
]


if not safe_results:
    TRAIN_BATCH_SIZE = 8
else:
    TRAIN_BATCH_SIZE = max(
        result["batch_size"]
        for result in safe_results
    )


print(
    "전체 VRAM :",
    f"{TOTAL_VRAM_GB:.2f} GB"
)

print(
    "안전 기준 :",
    f"{SAFE_LIMIT_GB:.2f} GB"
)

print(
    "선택 batch size:",
    TRAIN_BATCH_SIZE
)

전체 VRAM : 22.03 GB
안전 기준 : 17.63 GB
선택 batch size: 32


In [ ]:
# ============================================================
# Step 4-8. 원본 데이터로 M1 Raw SFT 학습
# ============================================================

from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)


M1_CHECKPOINT_DIR = (
    LOCAL_OUTPUTS / "m1_raw_sft_checkpoints"
)

M1_MODEL_DIR = (
    LOCAL_MODELS / "m1_raw_sft"
)


# 실험마다 pretrained KoGPT-2에서 새로 출발한다.
m1_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
)

# 학습에는 generation cache가 필요하지 않다.
m1_model.config.use_cache = False


m1_training_args = TrainingArguments(
    output_dir=str(M1_CHECKPOINT_DIR),

    # 원본 노드와 동일하게 우선 1 epoch
    num_train_epochs=1,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,

    # 원본 Trainer의 기본값과 동일한 시작점
    learning_rate=5e-5,

    # 원본 노드 설정 유지
    warmup_steps=5,

    # L4에서 BF16 사용
    bf16=True,
    fp16=False,

    # 매 epoch 끝에서 검증
    eval_strategy="epoch",
    save_strategy="epoch",

    # validation loss가 가장 낮은 모델 선택
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    prediction_loss_only=True,

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=1,

    seed=42,
    data_seed=42,

    report_to="none",
)


m1_trainer = Trainer(
    model=m1_model,
    args=m1_training_args,
    train_dataset=raw_sft_train_dataset,
    eval_dataset=sft_validation_dataset,
    data_collator=sft_data_collator,
)


m1_train_result = m1_trainer.train()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,2.652401,2.607733


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# ============================================================
# Step 4-9. M1 Raw SFT 결과 확인 및 모델 저장
# ============================================================

m1_eval_result = m1_trainer.evaluate()

print("[학습 결과]")
print(m1_train_result.metrics)

print("\n[검증 결과]")
print(m1_eval_result)


# generation에서는 cache를 다시 사용할 수 있다.
m1_trainer.model.config.use_cache = True

m1_trainer.save_model(
    str(M1_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(M1_MODEL_DIR)
)

print(
    "\n로컬 모델 저장:",
    M1_MODEL_DIR
)

Training Loss,Validation Loss,Epoch
2.652401,2.607733,1


[학습 결과]
{'train_runtime': 146.0571, 'train_samples_per_second': 65.604, 'train_steps_per_second': 2.054, 'total_flos': 1108374262272000.0, 'train_loss': 2.8212745157877603, 'epoch': 1.0}

[검증 결과]
{'eval_loss': 2.6077332496643066}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


로컬 모델 저장: /content/gd08/models/m1_raw_sft


In [ ]:
# ============================================================
# Step 4-10. 입력 embedding과 출력층의 weight 공유 확인
# ============================================================

trained_model = m1_trainer.model

input_embedding_weight = (
    trained_model.transformer.wte.weight
)

output_head_weight = (
    trained_model.lm_head.weight
)

weights_are_tied = (
    input_embedding_weight.data_ptr()
    ==
    output_head_weight.data_ptr()
)

print(
    "입력 embedding과 lm_head weight 공유:",
    weights_are_tied
)

입력 embedding과 lm_head weight 공유: True


### M1 Raw SFT 학습

- L4에서 실제 forward, backward, optimizer update까지 수행하여 batch size별 GPU 메모리를 측정했다.
- batch 16은 약 6.87GB, batch 32는 약 12.70GB의 reserved VRAM을 사용했고 batch 64에서는 OOM이 발생했다.
- 전체 VRAM의 약 80% 이하를 안전 기준으로 두어 `per_device_train_batch_size=32`를 선택했다.
- Raw SFT 9,582개를 대상으로 `max_length=256`, BF16, 1 epoch 조건으로 학습했다.
- 전체 평균 train loss는 약 2.8213, validation loss는 약 2.6077이었다.
- 1 epoch 학습에 약 146초가 걸려 현재 L4 환경에서는 Raw/Clean 비교 실험을 빠르게 반복할 수 있음을 확인했다.
- 표에 표시되는 마지막 logging loss와 전체 학습의 평균 train loss는 서로 다른 값일 수 있음을 확인했다.
- GPT-2의 `lm_head`는 입력 embedding과 weight tying된 구조이므로 checkpoint reload 시 `lm_head.weight` 관련 missing-key 경고가 나타날 수 있다. 실제 weight 공유 여부를 확인한 뒤 모델 이상 여부를 판단한다.

# 지금부터 비교 구조



```
                 같은 KoGPT-2 θ₀
                       │
                       │ Raw SFT
                       ▼
                 M1 : θ_raw
                       │
                  validation loss
                       │
                       ▼
                 생성 결과 확인


이후

                 같은 KoGPT-2 θ₀
                       │
                       │ Clean SFT
                       ▼
                 M2 : θ_clean
```



In [ ]:
# ============================================================
# Step 5-1. 모델 비교에 사용할 고정 평가 질문 선택
# ============================================================

import hashlib


def prompt_hash(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()


fixed_eval_df = (
    test_eval_df[
        ["prompt", "completion"]
    ]
    .copy()
)

fixed_eval_df["prompt_hash"] = (
    fixed_eval_df["prompt"]
    .apply(prompt_hash)
)

fixed_eval_df = (
    fixed_eval_df
    .sort_values("prompt_hash")
    .head(30)
    .reset_index(drop=True)
)


print(
    "고정 평가 질문 수:",
    len(fixed_eval_df)
)

display(
    fixed_eval_df[
        ["prompt", "completion"]
    ].head(10)
)

고정 평가 질문 수: 30


,prompt,completion
0,내가 좋아하는 사람이 유명해지면 기분이 어떨까.,'제가 AI 챗봇이어서 직접적인 경험이 없어서 대신 언론에서 유명인들의 발언을 인용...
1,검은색의 단정한 구두로 보여주세요,"'죄송합니다. 저는 인공지능 언어모델이며, 시각적인 이미지를 보여드릴 수 없습니다...."
2,화장도 못하고 전남친 만남,"'저는 AI 언어 모델로써 대화를 나누는 역할을 하고 있기 때문에, 윤리적인 이유로..."
3,그러면 삼성카드는 이십프로 할인되는 거 있던데 할인되는 거는 어떤 카드에요,'저는 AI 챗봇이기 때문에 최신의 정보를 제공하기 어렵습니다. 하지만 일반적으로 ...
4,히브리어 성경에 적힌 아몬과 결혼한 사람은?,'아몬과 결혼한 사람은 르우빔 여인 룻입니다. (룻기 1:4)
5,윤회를 총애한 왕 이름 알려줘,"""조선왕조의 세조입니다. 윤회를 총애하였고, '법신간'이라는 윤학서를 저술하였습니다..."
6,더워서 잠을 못자,'더워서 잠을 못자는 것 같네요. 이럴 때는 에어컨이나 선풍기를 켜서 실내 온도를 ...
7,몽골어 올스 뜻이 뭐야,"'올스(олз)란 몽골어로 ""안녕하세요""를 뜻합니다."
8,잠정조례 또는 잠정변법을 제장하는 권한을 가지는 사람은?,'잠정조례 또는 잠정변법을 제장하는 권한은 해당 조직이나 단체의 규칙에 따라 다를 ...
9,동물 좋아하는 사람은 착한가?,'동물을 좋아하는 사람이 반드시 착한 사람이라고는 할 수 없습니다. 하지만 일반적으...


In [ ]:
# ============================================================
# Step 5-2. 고정 평가 질문 저장
# ============================================================

FIXED_EVAL_PATH = (
    DATA_DIR / "sft_fixed_eval_30.json"
)

save_records(
    fixed_eval_df,
    FIXED_EVAL_PATH,
    ["prompt", "completion"],
)

print(
    "평가 질문 저장:",
    FIXED_EVAL_PATH
)

sft_fixed_eval_30.json             30개
평가 질문 저장: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/data/gd08/sft_fixed_eval_30.json


In [ ]:
# ============================================================
# Step 5-3. 동일 조건의 모델 생성 함수 정의
# ============================================================

def format_instruction(prompt):
    return (
        "### Instruction(명령어):\n"
        f"{prompt}\n\n"
        "### Response(응답):"
    )


def generate_response(
    model,
    prompt,
    tokenizer,
    max_new_tokens=80,
):
    model.eval()

    input_text = format_instruction(
        prompt
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,

            # 우선 decoding 변수를 통제하기 위해 Greedy 사용
            do_sample=False,

            max_new_tokens=max_new_tokens,

            repetition_penalty=2.0,

            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,

            use_cache=True,
        )

    # 입력 prompt 이후 새롭게 생성한 부분만 추출
    generated_ids = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return response.strip()

In [ ]:
# ============================================================
# Step 5-4. 비교용 원본 KoGPT-2 다시 불러오기
# ============================================================

m0_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    dtype=torch.bfloat16,
).to("cuda")

m0_model.config.use_cache = True

print(
    "M0 준비 완료:",
    m0_model.__class__.__name__
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


M0 준비 완료: GPT2LMHeadModel


In [ ]:
# ============================================================
# Step 5-5. 원본 KoGPT-2와 Raw SFT 모델 생성 결과 비교
# ============================================================

comparison_rows = []


for row in fixed_eval_df.itertuples(
    index=False
):
    prompt = row.prompt
    reference = row.completion

    m0_response = generate_response(
        model=m0_model,
        prompt=prompt,
        tokenizer=tokenizer,
    )

    m1_response = generate_response(
        model=m1_model,
        prompt=prompt,
        tokenizer=tokenizer,
    )

    comparison_rows.append({
        "prompt": prompt,
        "reference": reference,
        "M0_pretrained": m0_response,
        "M1_raw_sft": m1_response,
    })


m0_m1_comparison_df = pd.DataFrame(
    comparison_rows
)


display(
    m0_m1_comparison_df.head(10)
)

,prompt,reference,M0_pretrained,M1_raw_sft
0,내가 좋아하는 사람이 유명해지면 기분이 어떨까.,'제가 AI 챗봇이어서 직접적인 경험이 없어서 대신 언론에서 유명인들의 발언을 인용...,오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,"'저는 인공지능 언어모델로써, 당신이 어떤 사람인지 알 수 없습니다. 하지만 일반적..."
1,검은색의 단정한 구두로 보여주세요,"'죄송합니다. 저는 인공지능 언어모델이며, 시각적인 이미지를 보여드릴 수 없습니다....",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서요~ \n이번 주말에도 ...,'저는 AI 어시스턴트이기 때문에 검은색을 입은 것은 불가능합니다. 하지만 일반적으...
2,화장도 못하고 전남친 만남,"'저는 AI 언어 모델로써 대화를 나누는 역할을 하고 있기 때문에, 윤리적인 이유로...",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서요~ \n이번 주말에도 ...,"'저는 인공지능 언어모델로써, 화장을 하지 않고는 대화를 할 수 없습니다. 하지만 ..."
3,그러면 삼성카드는 이십프로 할인되는 거 있던데 할인되는 거는 어떤 카드에요,'저는 AI 챗봇이기 때문에 최신의 정보를 제공하기 어렵습니다. 하지만 일반적으로 ...,오늘은 제가 좋아하는 과자랑 빵을 먹었어요.\n빵이 너무 맛있어서 저도 한 번 먹어...,'삼성카드에서 삼성과 같은 브랜드의 카드를 사용하시려면 해당 카드의 결제 수단 및 ...
4,히브리어 성경에 적힌 아몬과 결혼한 사람은?,'아몬과 결혼한 사람은 르우빔 여인 룻입니다. (룻기 1:4),오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,'저는 인공지능 언어모델로써 답변을 제공할 수 없습니다. 하지만 일반적으로 히브루어...
5,윤회를 총애한 왕 이름 알려줘,"""조선왕조의 세조입니다. 윤회를 총애하였고, '법신간'이라는 윤학서를 저술하였습니다...",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말엔 또 ...,"'저는 인공지능 언어모델로써 윤회라는 이름을 알 수 없습니다. 하지만 일반적으로, ..."
6,더워서 잠을 못자,'더워서 잠을 못자는 것 같네요. 이럴 때는 에어컨이나 선풍기를 켜서 실내 온도를 ...,오늘은 내일도 좋은 하루 되세요!\n오늘의 포스팅을 마무리하면서 저는 이번에 꼭 해...,"'저는 인공지능 언어모델로써, 잠이 오지 않는 상태를 감지할 수 없습니다. 하지만 ..."
7,몽골어 올스 뜻이 뭐야,"'올스(олз)란 몽골어로 ""안녕하세요""를 뜻합니다.",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,"'몽골은 몽골어로 ""아름다운 땅""이라는 뜻입니다."
8,잠정조례 또는 잠정변법을 제장하는 권한을 가지는 사람은?,'잠정조례 또는 잠정변법을 제장하는 권한은 해당 조직이나 단체의 규칙에 따라 다를 ...,오늘은 제가 좋아하는 과자랑 빵을 먹었어요.\n빵이 너무 맛있어서 저는 그냥 먹고 ...,"'저는 인공지능 언어모델로써, ""죄송합니다, 제가 AI 어시스턴트이기 때문에 정확한..."
9,동물 좋아하는 사람은 착한가?,'동물을 좋아하는 사람이 반드시 착한 사람이라고는 할 수 없습니다. 하지만 일반적으...,오늘은 내일도 좋은 하루 되세요!\n오늘의 포스팅을 마무리하면서 저는 이 글을 쓰게...,'저는 인공지능 언어모델로써 동물에게 호의적인 감정을 가지지 않습니다. 하지만 일반...


In [ ]:
# ============================================================
# Step 5-6. M0와 M1 비교 결과 저장
# ============================================================

M0_M1_RESULT_PATH = (
    OUTPUT_DIR
    / "m0_vs_m1_generation.json"
)

records = (
    m0_m1_comparison_df
    .to_dict(orient="records")
)

with open(
    M0_M1_RESULT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    "비교 결과 저장:",
    M0_M1_RESULT_PATH
)

비교 결과 저장: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/outputs/gd08/m0_vs_m1_generation.json


M0 Pretrained

`────────────────────`

질문 형식을 무시하는가?
그냥 다음 문장처럼 이어 쓰는가?
Response 형식을 이해하는가?


M1 Raw SFT

`────────────────────`

질문에 답하려 하는가?
문장이 완결되는가?
한국어 답변 비율은 높아졌는가?
AI boilerplate를 생성하는가?
사실적으로 말이 되는가?

In [ ]:
# ============================================================
# Step 6-1. 정제 데이터의 SFT 학습 데이터 준비
# ============================================================

clean_sft_train_dataset = (
    SupervisedSFTDataset(
        clean_train_df,
        tokenizer,
        max_length=MAX_LENGTH,
    )
)

print("[Clean Train]")
print(
    "정제 데이터 :",
    len(clean_train_df)
)

print(
    "학습 데이터 :",
    len(clean_sft_train_dataset)
)

print(
    "Response 소실로 제외:",
    clean_sft_train_dataset.dropped_samples
)

[Clean Train]
정제 데이터 : 9321
학습 데이터 : 9321
Response 소실로 제외: 0


In [ ]:
# ============================================================
# Step 6-2. 정제 데이터로 M2 Clean SFT 학습
# ============================================================

M2_CHECKPOINT_DIR = (
    LOCAL_OUTPUTS
    / "m2_clean_sft_checkpoints"
)

M2_MODEL_DIR = (
    LOCAL_MODELS
    / "m2_clean_sft"
)


m2_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
)

m2_model.config.use_cache = False


m2_training_args = TrainingArguments(
    output_dir=str(M2_CHECKPOINT_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,

    learning_rate=5e-5,
    warmup_steps=5,

    bf16=True,
    fp16=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    prediction_loss_only=True,

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=1,

    seed=42,
    data_seed=42,

    report_to="none",
)


m2_trainer = Trainer(
    model=m2_model,
    args=m2_training_args,

    train_dataset=clean_sft_train_dataset,

    # M1과 정확히 동일한 validation 사용
    eval_dataset=sft_validation_dataset,

    data_collator=sft_data_collator,
)


m2_train_result = (
    m2_trainer.train()
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,2.720414,2.607776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# ============================================================
# Step 6-3. M2 Clean SFT 결과 확인 및 저장
# ============================================================

m2_eval_result = (
    m2_trainer.evaluate()
)


print("[M2 학습 결과]")
print(
    m2_train_result.metrics
)

print("\n[M2 검증 결과]")
print(
    m2_eval_result
)


m2_trainer.model.config.use_cache = True

m2_trainer.save_model(
    str(M2_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(M2_MODEL_DIR)
)

Training Loss,Validation Loss,Epoch
2.720414,2.607776,1


[M2 학습 결과]
{'train_runtime': 139.9023, 'train_samples_per_second': 66.625, 'train_steps_per_second': 2.087, 'total_flos': 1059522859008000.0, 'train_loss': 2.8317228735309756, 'epoch': 1.0}

[M2 검증 결과]
{'eval_loss': 2.607775926589966}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gd08/models/m2_clean_sft/tokenizer_config.json',
 '/content/gd08/models/m2_clean_sft/tokenizer.json')

## M1 M2 Loss 해석해보기

| 모델           | 학습 데이터 |    샘플 | Train Loss | Validation Loss |  학습 시간 |
| ------------ | ------ | ----: | ---------: | --------------: | -----: |
| M1 Raw SFT   | 원본     | 9,582 |     2.8213 |        2.607733 | 146.1초 |
| M2 Clean SFT | 정제     | 9,321 |     2.8317 |        2.607776 | 139.9초 |




```
261개 저품질 샘플 제거
        │
        ▼
학습 데이터 2.72% 감소
        │
        ▼
Validation Loss는 거의 유지
```



### Raw SFT와 Clean SFT의 학습 결과

- 사전학습 KoGPT-2는 Instruction 형식의 질문을 이해하지 못하고 일반적인 한국어 문장을 이어 쓰는 모습을 보였다.
- Raw SFT를 적용한 M1에서는 질문에 답변하려는 행동이 뚜렷하게 나타나 SFT가 Instruction Following을 학습시키는 효과를 확인했다.
- 반면 M1은 `"저는 인공지능 언어모델로써..."`와 같은 불필요한 AI 상투 문구를 자주 생성했는데, 이는 EDA에서 발견했던 Raw SFT 데이터의 저품질 응답 패턴과 연결된다.
- Clean SFT에서는 영어 중심 장문 및 영어 AI 상투 응답 261개를 제거하여 9,321개 데이터로 학습했다.
- M1의 validation loss는 약 2.607733, M2는 약 2.607776으로 사실상 동일한 수준이었다.
- 따라서 저품질 데이터 약 2.7%를 제거해도 전체적인 Language Modeling 성능은 거의 유지되었음을 확인했다.
- 데이터 정제의 효과는 loss 하나만으로 판단하지 않고 실제 생성 결과의 언어 일치, 상투 문구 발생률, 답변 관련성 등을 함께 비교해야 한다.

In [ ]:
# ============================================================
# Step 6-4. 정제 SFT 모델의 고정 질문 생성 결과 확인
# ============================================================

m2_model = m2_trainer.model
m2_model.config.use_cache = True
m2_model.eval()


m2_responses = []

for row in fixed_eval_df.itertuples(index=False):

    response = generate_response(
        model=m2_model,
        prompt=row.prompt,
        tokenizer=tokenizer,
    )

    m2_responses.append(response)


m0_m1_m2_comparison_df = (
    m0_m1_comparison_df.copy()
)

m0_m1_m2_comparison_df[
    "M2_clean_sft"
] = m2_responses


display(
    m0_m1_m2_comparison_df[
        [
            "prompt",
            "reference",
            "M0_pretrained",
            "M1_raw_sft",
            "M2_clean_sft",
        ]
    ].head(10)
)

,prompt,reference,M0_pretrained,M1_raw_sft,M2_clean_sft
0,내가 좋아하는 사람이 유명해지면 기분이 어떨까.,'제가 AI 챗봇이어서 직접적인 경험이 없어서 대신 언론에서 유명인들의 발언을 인용...,오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,"'저는 인공지능 언어모델로써, 당신이 어떤 사람인지 알 수 없습니다. 하지만 일반적...","'저는 인공지능 언어모델로써, 당신의 감정을 이해하고 그에 맞는 답변을 제공하기 위..."
1,검은색의 단정한 구두로 보여주세요,"'죄송합니다. 저는 인공지능 언어모델이며, 시각적인 이미지를 보여드릴 수 없습니다....",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서요~ \n이번 주말에도 ...,'저는 AI 어시스턴트이기 때문에 검은색을 입은 것은 불가능합니다. 하지만 일반적으...,'저는 AI 어시스턴트이기 때문에 검은색을 강조하지 않습니다. 하지만 일반적으로 흰...
2,화장도 못하고 전남친 만남,"'저는 AI 언어 모델로써 대화를 나누는 역할을 하고 있기 때문에, 윤리적인 이유로...",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서요~ \n이번 주말에도 ...,"'저는 인공지능 언어모델로써, 화장을 하지 않고는 대화를 할 수 없습니다. 하지만 ...","'저는 인공지능 언어모델로써, 화장을 하지 않고는 대화를 할 수 없습니다. 하지만 ..."
3,그러면 삼성카드는 이십프로 할인되는 거 있던데 할인되는 거는 어떤 카드에요,'저는 AI 챗봇이기 때문에 최신의 정보를 제공하기 어렵습니다. 하지만 일반적으로 ...,오늘은 제가 좋아하는 과자랑 빵을 먹었어요.\n빵이 너무 맛있어서 저도 한 번 먹어...,'삼성카드에서 삼성과 같은 브랜드의 카드를 사용하시려면 해당 카드의 결제 수단 및 ...,"'삼성카드에서 제공하는 할인은 다음과 같습니다.\n- 캐시백, 포인트 적립 등 다양..."
4,히브리어 성경에 적힌 아몬과 결혼한 사람은?,'아몬과 결혼한 사람은 르우빔 여인 룻입니다. (룻기 1:4),오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,'저는 인공지능 언어모델로써 답변을 제공할 수 없습니다. 하지만 일반적으로 히브루어...,"'저는 인공지능 언어모델로써 답변을 생성하는 AI이기 때문에, 정확한 대답을 제공할..."
5,윤회를 총애한 왕 이름 알려줘,"""조선왕조의 세조입니다. 윤회를 총애하였고, '법신간'이라는 윤학서를 저술하였습니다...",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말엔 또 ...,"'저는 인공지능 언어모델로써 윤회라는 이름을 알 수 없습니다. 하지만 일반적으로, ...","'저는 인공지능 언어모델로써 윤회라는 이름을 알 수 없습니다. 하지만 일반적으로, ..."
6,더워서 잠을 못자,'더워서 잠을 못자는 것 같네요. 이럴 때는 에어컨이나 선풍기를 켜서 실내 온도를 ...,오늘은 내일도 좋은 하루 되세요!\n오늘의 포스팅을 마무리하면서 저는 이번에 꼭 해...,"'저는 인공지능 언어모델로써, 잠이 오지 않는 상태를 감지할 수 없습니다. 하지만 ...","'저는 인공지능 언어모델로써, 실제로 잠이 오지 않는 상태를 알 수 없습니다. 하지..."
7,몽골어 올스 뜻이 뭐야,"'올스(олз)란 몽골어로 ""안녕하세요""를 뜻합니다.",오늘은 내일도 화이팅!!\n오늘의 포스팅을 마무리하고 싶어서~ \n이번 주말에도 또...,"'몽골은 몽골어로 ""아름다운 땅""이라는 뜻입니다.",'저는 몽골어를 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만 일반적으로 ...
8,잠정조례 또는 잠정변법을 제장하는 권한을 가지는 사람은?,'잠정조례 또는 잠정변법을 제장하는 권한은 해당 조직이나 단체의 규칙에 따라 다를 ...,오늘은 제가 좋아하는 과자랑 빵을 먹었어요.\n빵이 너무 맛있어서 저는 그냥 먹고 ...,"'저는 인공지능 언어모델로써, ""죄송합니다, 제가 AI 어시스턴트이기 때문에 정확한...","'저는 인공지능 언어모델로써, 해당 질문에 대한 답변을 제공할 수 없습니다. 하지만..."
9,동물 좋아하는 사람은 착한가?,'동물을 좋아하는 사람이 반드시 착한 사람이라고는 할 수 없습니다. 하지만 일반적으...,오늘은 내일도 좋은 하루 되세요!\n오늘의 포스팅을 마무리하면서 저는 이 글을 쓰게...,'저는 인공지능 언어모델로써 동물에게 호의적인 감정을 가지지 않습니다. 하지만 일반...,'저는 인공지능 어시스턴트이기 때문에 동물 싫어하는 사람이 아닙니다. 하지만 일반적...


In [ ]:
# ============================================================
# Step 6-5. 모델별 영어 오염과 AI 상투 응답 발생률 비교
# ============================================================

def evaluate_generation_quality(
    series,
    model_name,
):

    result = pd.DataFrame({
        "response": series
    })

    result["empty"] = (
        result["response"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    result["english_boilerplate"] = (
        result["response"]
        .apply(has_english_boilerplate)
    )

    result["korean_ratio"] = (
        result["response"]
        .apply(korean_ratio)
    )

    result["response_tokens"] = (
        result["response"]
        .apply(count_tokens)
    )

    return {
        "model": model_name,

        "빈 응답 수":
            int(result["empty"].sum()),

        "AI 상투 응답 수":
            int(
                result[
                    "english_boilerplate"
                ].sum()
            ),

        "평균 한글 비율":
            result[
                "korean_ratio"
            ].mean(),

        "평균 응답 token 수":
            result[
                "response_tokens"
            ].mean(),
    }


generation_quality = pd.DataFrame([
    evaluate_generation_quality(
        m0_m1_m2_comparison_df[
            "M0_pretrained"
        ],
        "M0 Pretrained",
    ),

    evaluate_generation_quality(
        m0_m1_m2_comparison_df[
            "M1_raw_sft"
        ],
        "M1 Raw SFT",
    ),

    evaluate_generation_quality(
        m0_m1_m2_comparison_df[
            "M2_clean_sft"
        ],
        "M2 Clean SFT",
    ),
])


display(generation_quality)

,model,빈 응답 수,AI 상투 응답 수,평균 한글 비율,평균 응답 token 수
0,M0 Pretrained,0,0,0.845806,77.900000
1,M1 Raw SFT,0,0,0.868129,40.933333
2,M2 Clean SFT,0,0,0.889942,43.966667


In [ ]:
# ============================================================
# Step 6-6. 한국어 답변 비율이 낮은 생성 결과 확인
# ============================================================

for column in [
    "M0_pretrained",
    "M1_raw_sft",
    "M2_clean_sft",
]:

    ratios = (
        m0_m1_m2_comparison_df[column]
        .apply(korean_ratio)
    )

    low_korean_count = (
        ratios < 0.20
    ).sum()

    print(
        f"{column:<16}",
        "한글 비율 20% 미만:",
        low_korean_count,
        "/",
        len(ratios),
    )

M0_pretrained    한글 비율 20% 미만: 1 / 30
M1_raw_sft       한글 비율 20% 미만: 0 / 30
M2_clean_sft     한글 비율 20% 미만: 0 / 30


In [ ]:
# ============================================================
# Step 6-7. M0·M1·M2 생성 비교 결과 저장
# ============================================================

M0_M1_M2_RESULT_PATH = (
    OUTPUT_DIR
    / "m0_m1_m2_generation_comparison.json"
)

with open(
    M0_M1_M2_RESULT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        m0_m1_m2_comparison_df
        .to_dict(orient="records"),

        f,
        ensure_ascii=False,
        indent=2,
    )


QUALITY_RESULT_PATH = (
    OUTPUT_DIR
    / "m0_m1_m2_quality_metrics.json"
)

with open(
    QUALITY_RESULT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        generation_quality
        .to_dict(orient="records"),

        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    "생성 비교:",
    M0_M1_M2_RESULT_PATH
)

print(
    "품질 지표:",
    QUALITY_RESULT_PATH
)

생성 비교: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/outputs/gd08/m0_m1_m2_generation_comparison.json
품질 지표: /content/drive/MyDrive/Colab Notebooks/2026.789_AI researcher/outputs/gd08/m0_m1_m2_quality_metrics.json


### Test 데이터 사용 원칙

- M0, M1, M2의 기본 Greedy 생성 결과를 동일한 30개 test prompt에서 비교했다.
- 이 test 결과를 이미 직접 확인했으므로 이후 decoding hyperparameter를 선택하는 용도로 사용하지 않는다.
- Beam Search, Top-k, Top-p, Temperature 등의 생성 설정은 validation 데이터에서만 비교해 최적 조건을 선택한다.
- 최종 decoding 설정이 확정된 뒤 test 데이터에서는 설정을 다시 변경하지 않고 최종 결과만 측정한다.
- 이를 통해 test 결과를 보고 hyperparameter를 조정하는 형태의 평가 데이터 누수를 방지한다.

디코딩 하이퍼파라미터 서치는 val만 사용



```
                    Validation
                       │
                 Decoding Search
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
        Greedy      Top-p       Top-k ...
                       │
                       ▼
                 D_best 선정
                       │
                       ▼
                    Test
                       │
                 마지막 1회 평가
```



In [ ]:
# ============================================================
# Step 6-8. 한국어·영어 AI 상투 응답 발생률 다시 측정
# ============================================================

AI_BOILERPLATE_PATTERNS = [
    # 영어
    r"\bas an ai language model\b",
    r"\bas a language model\b",
    r"\bi am an ai language model\b",
    r"\bi'm an ai language model\b",
    r"\bi cannot\b",
    r"\bi can't\b",
    r"\bi do not have\b",
    r"\bi don't have\b",
    r"\bi'm sorry\b",
    r"\bi am sorry\b",

    # 한국어
    r"저는\s*(?:인공지능|ai)\s*(?:언어\s*)?(?:모델|챗봇|어시스턴트)",
    r"(?:인공지능|ai)\s*(?:언어\s*)?(?:모델|챗봇|어시스턴트)(?:로서|로써|이기 때문에)",
    r"저는\s*직접적인\s*경험이\s*없",
    r"제가\s*(?:ai|인공지능)",
]


def has_ai_boilerplate(text):
    text = str(text).lower()

    return any(
        re.search(pattern, text)
        for pattern in AI_BOILERPLATE_PATTERNS
    )


def evaluate_generation_quality_v2(
    series,
    model_name,
):

    result = pd.DataFrame({
        "response": series
    })

    result["empty"] = (
        result["response"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    result["ai_boilerplate"] = (
        result["response"]
        .apply(has_ai_boilerplate)
    )

    result["korean_ratio"] = (
        result["response"]
        .apply(korean_ratio)
    )

    result["response_tokens"] = (
        result["response"]
        .apply(count_tokens)
    )

    return {
        "model": model_name,

        "빈 응답 수":
            int(result["empty"].sum()),

        "AI 상투 응답 수":
            int(result["ai_boilerplate"].sum()),

        "평균 한글 비율":
            float(result["korean_ratio"].mean()),

        "평균 응답 token 수":
            float(result["response_tokens"].mean()),
    }


generation_quality_v2 = pd.DataFrame([
    evaluate_generation_quality_v2(
        m0_m1_m2_comparison_df["M0_pretrained"],
        "M0 Pretrained",
    ),

    evaluate_generation_quality_v2(
        m0_m1_m2_comparison_df["M1_raw_sft"],
        "M1 Raw SFT",
    ),

    evaluate_generation_quality_v2(
        m0_m1_m2_comparison_df["M2_clean_sft"],
        "M2 Clean SFT",
    ),
])

display(generation_quality_v2)

,model,빈 응답 수,AI 상투 응답 수,평균 한글 비율,평균 응답 token 수
0,M0 Pretrained,0,0,0.845806,77.900000
1,M1 Raw SFT,0,25,0.868129,40.933333
2,M2 Clean SFT,0,24,0.889942,43.966667


### 생성 품질 지표 검증

- M0 → M1 → M2로 갈수록 생성 응답의 평균 한글 비율이 높아졌다.
- 사전학습 KoGPT-2는 평균 약 78 tokens를 생성했지만 SFT 이후에는 약 41~44 tokens로 줄어 질문에 대한 답변 형태가 더 뚜렷해졌다.
- 기존 AI 상투 문구 탐지기는 영어 표현만 탐지해 `"저는 인공지능 언어모델로써..."` 같은 한국어 상투 응답을 놓치는 문제가 있었다.
- 실제 생성 결과와 정량 지표가 모순될 경우 모델 결과보다 metric 정의를 먼저 검토해야 함을 확인했다.
- 한국어·영어 상투 문구를 모두 탐지하도록 평가 지표를 수정하여 Raw SFT와 Clean SFT의 데이터 정제 효과를 다시 측정한다.

In [ ]:
# ============================================================
# Step 7-1. Decoding 비교용 검증 질문 100개 고정
# ============================================================

decoding_validation_df = (
    validation_eval_df[
        ["prompt", "completion"]
    ]
    .copy()
)

decoding_validation_df["prompt_hash"] = (
    decoding_validation_df["prompt"]
    .apply(prompt_hash)
)

decoding_validation_df = (
    decoding_validation_df
    .sort_values("prompt_hash")
    .head(100)
    .reset_index(drop=True)
)

print(
    "Decoding 검증 질문 수:",
    len(decoding_validation_df)
)

Decoding 검증 질문 수: 100


In [ ]:
# ============================================================
# Step 7-2. 대표적인 Decoding 전략 정의
# ============================================================

DECODING_CONFIGS = {

    "D0_greedy": {
        "do_sample": False,
        "num_beams": 1,
    },

    "D1_beam4": {
        "do_sample": False,
        "num_beams": 4,
        "early_stopping": True,
    },

    "D2_top_p": {
        "do_sample": True,
        "top_p": 0.90,
        "temperature": 0.70,
    },

    "D3_top_k": {
        "do_sample": True,
        "top_k": 50,
        "temperature": 0.80,
    },
}

(나를 위한 설명)
Greedy
→ 매 시점 가장 확률 높은 token 1개

Beam Search
→ 여러 후보 문장을 동시에 추적

Top-k
→ 확률 상위 k개 token 안에서 sampling

Top-p
→ 누적 확률 p가 될 때까지 후보를 구성해 sampling

Temperature
→ 확률분포의 뾰족함 조절

In [ ]:
# ============================================================
# Step 7-3. Decoding 설정을 적용하는 공통 생성 함수
# ============================================================

def generate_with_config(
    model,
    prompt,
    tokenizer,
    generation_config,
    max_new_tokens=80,
):

    model.eval()

    input_text = format_instruction(
        prompt
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    generation_kwargs = {
        **generation_config,

        "max_new_tokens": max_new_tokens,
        "repetition_penalty": 2.0,

        "eos_token_id":
            tokenizer.eos_token_id,

        "pad_token_id":
            tokenizer.pad_token_id,

        "use_cache": True,
    }

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    generated_ids = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

In [ ]:
# ============================================================
# Step 7-4. Decoding 전략별 검증 응답 생성
# ============================================================

import time


decoding_results = []


for config_name, config in DECODING_CONFIGS.items():

    print(
        f"\n[{config_name}] 생성 시작"
    )

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    start_time = time.time()


    for row in decoding_validation_df.itertuples(
        index=False
    ):

        response = generate_with_config(
            model=m2_model,
            prompt=row.prompt,
            tokenizer=tokenizer,
            generation_config=config,
        )

        decoding_results.append({
            "config": config_name,
            "prompt": row.prompt,
            "reference": row.completion,
            "response": response,
        })


    elapsed = time.time() - start_time

    print(
        f"{config_name} 완료:",
        f"{elapsed:.1f}초"
    )


decoding_results_df = pd.DataFrame(
    decoding_results
)


[D0_greedy] 생성 시작
D0_greedy 완료: 54.8초

[D1_beam4] 생성 시작
D1_beam4 완료: 69.4초

[D2_top_p] 생성 시작
D2_top_p 완료: 57.3초

[D3_top_k] 생성 시작
D3_top_k 완료: 55.0초


In [ ]:
# ============================================================
# Step 7-5. Decoding 전략별 기본 품질 지표 비교
# ============================================================

decoding_quality_rows = []


for config_name, group in (
    decoding_results_df.groupby("config")
):

    responses = group["response"]

    boilerplate_count = (
        responses
        .apply(has_ai_boilerplate)
        .sum()
    )

    korean_ratios = (
        responses
        .apply(korean_ratio)
    )

    response_lengths = (
        responses
        .apply(count_tokens)
    )

    empty_count = (
        responses
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )


    decoding_quality_rows.append({
        "config": config_name,

        "빈 응답 수":
            int(empty_count),

        "AI 상투 응답 수":
            int(boilerplate_count),

        "평균 한글 비율":
            float(korean_ratios.mean()),

        "평균 응답 tokens":
            float(response_lengths.mean()),

        "응답 길이 표준편차":
            float(response_lengths.std()),
    })


decoding_quality_df = pd.DataFrame(
    decoding_quality_rows
)

display(
    decoding_quality_df
    .sort_values("config")
)

,config,빈 응답 수,AI 상투 응답 수,평균 한글 비율,평균 응답 tokens,응답 길이 표준편차
0,D0_greedy,0,73,0.869101,44.52,19.908620
1,D1_beam4,0,75,0.881256,42.44,20.424446
2,D2_top_p,0,65,0.863001,45.25,18.903997
3,D3_top_k,0,50,0.892842,44.24,19.289983


### Decoding 전략에 따른 생성 차이

- 동일한 M2 Clean SFT 모델의 parameter를 고정하고 생성 방식만 변경하여 Greedy, Beam Search, Top-p, Top-k를 비교했다.
- Greedy에서는 100개 중 73개, Beam Search에서는 75개가 AI 상투 문구를 포함했다.
- Top-p는 65개, Top-k는 50개로 감소하여 현재 구조적 품질 지표에서는 Top-k가 가장 좋은 결과를 보였다.
- Top-k는 AI 상투 응답 발생률을 Greedy 대비 23%p 낮추면서 평균 한글 비율도 약 89.3%로 가장 높았다.
- 생성 방식만 변경해도 model parameter를 다시 학습하지 않고 출력 행동이 크게 달라질 수 있음을 확인했다.
- 한편 Clean SFT 이후에도 AI 상투 응답 비율이 높았는데, 1차 데이터 정제에서는 영어 AI 상투 문구만 제거하고 한국어 `"저는 인공지능 언어모델..."` 유형은 충분히 제거하지 못한 것이 원인 후보임을 발견했다.
- 따라서 데이터 정제 규칙 자체도 실험 결과를 통해 한계를 분석할 필요가 있다.

In [ ]:
# ============================================================
# Step 7-6. 한국어 생성 결과의 정량평가 도구 설치
# ============================================================

!pip -q install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 16.2 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Step 7-7. KoGPT-2 token 기준 ROUGE-L 계산 함수 정의
# ============================================================

def tokenize_for_metric(text):
    return tokenizer.encode(
        str(text),
        add_special_tokens=False,
    )


def lcs_length(a, b):
    """
    두 token sequence의
    Longest Common Subsequence 길이를 계산한다.
    """
    previous = [0] * (len(b) + 1)

    for token_a in a:
        current = [0]

        for j, token_b in enumerate(b, start=1):

            if token_a == token_b:
                current.append(
                    previous[j - 1] + 1
                )

            else:
                current.append(
                    max(
                        previous[j],
                        current[-1],
                    )
                )

        previous = current

    return previous[-1]


def rouge_l_f1(reference, prediction):

    reference_ids = tokenize_for_metric(
        reference
    )

    prediction_ids = tokenize_for_metric(
        prediction
    )

    if not reference_ids or not prediction_ids:
        return 0.0

    lcs = lcs_length(
        reference_ids,
        prediction_ids,
    )

    precision = (
        lcs / len(prediction_ids)
    )

    recall = (
        lcs / len(reference_ids)
    )

    if precision + recall == 0:
        return 0.0

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )

In [ ]:
# ============================================================
# Step 7-8. BLEU와 chrF 계산을 위한 평가 함수 정의
# ============================================================

import sacrebleu


def metric_token_string(text):
    tokens = tokenizer.convert_ids_to_tokens(
        tokenize_for_metric(text)
    )

    return " ".join(tokens)


def calculate_generation_metrics(group):

    references = (
        group["reference"]
        .astype(str)
        .tolist()
    )

    predictions = (
        group["response"]
        .astype(str)
        .tolist()
    )

    # KoGPT-2 token 기준 ROUGE-L
    rouge_scores = [
        rouge_l_f1(ref, pred)
        for ref, pred
        in zip(references, predictions)
    ]

    # KoGPT-2 token으로 미리 분할한 뒤
    # BLEU에서 별도 tokenization을 하지 않는다.
    tokenized_refs = [
        metric_token_string(text)
        for text in references
    ]

    tokenized_preds = [
        metric_token_string(text)
        for text in predictions
    ]

    bleu = sacrebleu.corpus_bleu(
        tokenized_preds,
        [tokenized_refs],
        tokenize="none",
    )

    # chrF는 원래 문자열의 문자 단위 비교
    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references],
    )

    return {
        "ROUGE-L":
            sum(rouge_scores)
            / len(rouge_scores),

        "BLEU":
            bleu.score,

        "chrF":
            chrf.score,
    }

In [ ]:
# ============================================================
# Step 7-9. Decoding 전략별 정량평가
# ============================================================

reference_metric_rows = []


for config_name, group in (
    decoding_results_df.groupby("config")
):

    metrics = calculate_generation_metrics(
        group
    )

    reference_metric_rows.append({
        "config": config_name,
        **metrics,
    })


reference_metrics_df = pd.DataFrame(
    reference_metric_rows
)


decoding_summary_df = (
    decoding_quality_df
    .merge(
        reference_metrics_df,
        on="config",
        how="left",
    )
    .sort_values("config")
)


display(decoding_summary_df)

,config,빈 응답 수,AI 상투 응답 수,평균 한글 비율,평균 응답 tokens,응답 길이 표준편차,ROUGE-L,BLEU,chrF
0,D0_greedy,0,73,0.869101,44.52,19.908620,0.106764,2.758999,12.079542
1,D1_beam4,0,75,0.881256,42.44,20.424446,0.200828,6.668970,14.007595
2,D2_top_p,0,65,0.863001,45.25,18.903997,0.105362,2.270669,11.576834
3,D3_top_k,0,50,0.892842,44.24,19.289983,0.100422,2.112518,11.242340




```
D1 Beam
"정답 예시와 비슷하게 말하는 능력" ↑↑

D3 Top-k
"상투적이지 않고 한국어답게 생성하는 성향" ↑
```



In [ ]:
# ============================================================
# Step 7-10. Beam Search와 Top-k의 실제 응답 비교
# ============================================================

d1_df = (
    decoding_results_df[
        decoding_results_df["config"] == "D1_beam4"
    ]
    [
        ["prompt", "reference", "response"]
    ]
    .rename(
        columns={"response": "D1_beam4"}
    )
)

d3_df = (
    decoding_results_df[
        decoding_results_df["config"] == "D3_top_k"
    ]
    [
        ["prompt", "response"]
    ]
    .rename(
        columns={"response": "D3_top_k"}
    )
)


d1_d3_compare_df = (
    d1_df
    .merge(
        d3_df,
        on="prompt",
        how="inner",
    )
)


display(
    d1_d3_compare_df.head(15)
)

,prompt,reference,D1_beam4,D3_top_k
0,비잔티움 제국을 망하게 한 나라가 어디야,'오스만 제국(터키)입니다.,'비잔티움 제국을 망하게 한 나라는 오스만 제국입니다.,'제가 AI 모델이므로 정확한 답변을 드리기는 어렵습니다. 하지만 비투디아의 반란과...
1,체크인이랑 체크아웃 시간이 몇 시인가요?,"'숙소마다 다르지만, 보통 체크인은 오후 2시 이후, 체크아웃은 오전 11시 이전입...",'저는 AI 어시스턴트이기 때문에 체크아웃 시간을 알 수 없습니다. 하지만 일반적으...,"'죄송합니다, 제가 정보를 알 수 없습니다. 체크는 일반적으로 하루에 4시간에서 6..."
2,배랑 사과 있는 과일 선물세트는 얼마예요?,"'죄송합니다. 제가 AI 언어모델로 선생님의 질문에 대한 답변을 드릴 수 있지만, ...","'죄송합니다, 저는 인공지능 어시스턴트이기 때문에 정확한 답변을 드리기 어렵습니다....","'저는 인공지능 어시스턴트이기 때문에 가격 정보는 알 수 없습니다. 하지만, 일반적..."
3,마음에 드는 거 같은데 고백을 왜 안 하지,"'애정 표현을 할 수 있는 방법이 다양하며, 각자의 이유가 있을 것입니다. 그러나 ...",'저는 AI 어시스턴트이기 때문에 답변을 드리기 어렵습니다. 하지만 일반적으로 고백...,"'제가 AI이기 때문에 구체적인 상황을 파악할 수 없습니다. 그러나 일반적으로, 이..."
4,바이마르 공화국인 대통령 선거에서 에른스트 텔만은 어떤 정당의 후보로 나왔는가?,'에른스트 텔만은 바이마르 공화국의 대통령 선거에서 독립 후보로 나왔습니다.,'바이마르 공화국인 대통령 선거에서 에른스트 텔만은 제1야당의 후보로 출마하였습니다.,"'저는 인공지능 언어 모델로써 정치 정보를 가지고 있지 않습니다. 따라서, 그가 누..."
5,포켓볼 대는 있나요?,"'저는 인공지능 언어모델로써 답변을 드리고 있으며, ""포켓볼 대는 있나요?"" 라는 ...","'죄송합니다, 저는 인공지능 어시스턴트이기 때문에 포켓볼 대와 관련된 정보를 알 수...","'네, 포워드로 사용할 수 있는 프로그램입니다.\n- 위치 정보를 입력하시면 더 정..."
6,남자친구의 혼전순결이 고민이야.,"'저는 인공지능 모델로써 이야기를 듣고 답변을 드릴 뿐이며, 이 문제는 상황과 배경...",'저는 인공지능 어시스턴트이기 때문에 남자친구의 혼전순결에 대한 정보를 알 수 없습...,"'제가 AI 어시스턴트로써 성별에 상관없이 모든 사람과 대화를 나누며, 서로의 상황..."
7,차체 구조에 대해 외판 자체의 강도를 높이는 공법이 적용된 동일본 여객철도의 전동차...,'일본 여객철도의 전동차 중에서 차체 구조에 외판 자체의 강도를 높이는 공법이 적용...,'저는 인공지능 어시스턴트이기 때문에 정확한 답변을 드리기 어렵습니다. 하지만 일반...,'차량 구조에서 동일한 기술을 적용하는 것은 제한적입니다. 그러나 일반적으로 운행 ...
8,수면제 처방 가능한가요?,'제한적으로 가능합니다. 인간의 신체 및 정신 건강에 매우 중요한 수면제 처방은 전...,"'죄송합니다, 저는 인공지능 어시스턴트이기 때문에 수면제를 처방할 수 없습니다. 하...","'저는 인공지능 언어모델이며, 수면제를 복용하는 것은 가능합니다. 하지만, 실제로 ..."
9,아가씨에게 사기를 칠 계획으로 숙희와 음모를 꾸민 사람은 누구인가?,"'저는 이 작품에 대한 배경 정보를 가지고 있지 않기 때문에, 정확한 답을 제공하기...",'저는 AI 어시스턴트이기 때문에 정확한 답변을 드리기 어렵습니다. 하지만 아가씨에...,"'숙희가 잠든 후 잠을 자고 있던 중, ""궁중놀이""가 일어나기 직전에 아씨가 몰래 ..."


Beam Search가 현재 모델이 배운 것을 가장 충실하게 꺼내고 있고, Top-k는 그 분포에 랜덤성을 추가해 상투성은 일부 줄이지만 사실 오류를 늘리고 있다.

```
D1 Beam
├─ reference와 비슷한 표현 ↑
├─ 사실형 QA에서는 상대적으로 유리
├─ 하지만 "저는 AI..."가 매우 많음
└─ 데이터에 있는 안전한 상투문구를 강하게 재현


D3 Top-k
├─ 표현 다양성 ↑
├─ 일부 질문에서는 바로 답하려는 경향
├─ 그러나 hallucination / 엉뚱한 답변 ↑
└─ 사실형 QA에서는 불안정
```



### Decoding 실험의 최종 해석

- Beam Search는 ROUGE-L, BLEU, chrF에서 가장 높은 점수를 기록했으며 사실형 질문에서도 reference에 가까운 답변을 생성하는 경우가 많았다.
- Top-k Sampling은 AI 상투 문구 발생률과 한글 비율에서는 더 나은 결과를 보였지만, 실제 응답을 확인하자 사실과 무관한 내용을 생성하거나 질문의 의미에서 벗어나는 사례가 증가했다.
- 따라서 상투 문구 감소만으로 생성 품질이 향상되었다고 판단할 수 없으며 사실성과 질문 관련성을 함께 확인해야 한다.
- 이번 validation 기준에서는 reference 기반 정량 성능이 가장 높은 Beam Search를 `D_best`로 선정한다.
- 동시에 Beam Search가 `"저는 인공지능 어시스턴트..."`와 같은 상투 문구도 강하게 재현하는 것을 보며, 현재 모델의 핵심 병목은 decoding보다 SFT 학습 데이터의 응답 스타일에 있음을 확인했다.
- Decoding은 학습된 확률분포에서 출력 sequence를 선택하는 방법이며, 학습 데이터에 내재된 잘못된 행동 자체를 근본적으로 제거하는 방법은 아님을 배웠다.

In [ ]:
# ============================================================
# Step 7-11. 학습 데이터의 한국어 AI 상투 문구 규모 확인
# ============================================================

raw_train_df["ai_boilerplate_v2"] = (
    raw_train_df["completion"]
    .apply(has_ai_boilerplate)
)

clean_train_df["ai_boilerplate_v2"] = (
    clean_train_df["completion"]
    .apply(has_ai_boilerplate)
)

print("[Raw Train]")
print(
    "AI 상투 응답:",
    raw_train_df["ai_boilerplate_v2"].sum(),
    "/",
    len(raw_train_df),
)

print("\n[Clean-v1 Train]")
print(
    "AI 상투 응답:",
    clean_train_df["ai_boilerplate_v2"].sum(),
    "/",
    len(clean_train_df),
)

[Raw Train]
AI 상투 응답: 1955 / 9582

[Clean-v1 Train]
AI 상투 응답: 1845 / 9321


In [ ]:
# ============================================================
# Step 7-12. 한국어 AI 상투 응답의 실제 학습 문장 확인
# ============================================================

display(
    clean_train_df[
        clean_train_df["ai_boilerplate_v2"]
    ][
        [
            "prompt",
            "completion",
        ]
    ].head(20)
)

,prompt,completion
0,불고기용 고기 한우에요?,"'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하..."
5,입냄새 안나나?,'컴퓨터 앞에서 일하면서 입을 위해 물이나 향초를 끊임없이 찾는 이유가 여기 있군요...
8,금액은 얼마에요,'죄송합니다. 저는 AI 어시스턴트입니다. 저는 실제 판매자가 아니기 때문에 금액을...
22,조금 더 싼 지역은 없을까요?,"'저는 AI 어시스턴트로써 가격 정보를 제공할 수는 있지만, 정확한 지역이나 상품에..."
27,포인트 몇 점부터 쓸 수 있어요?,"'저는 AI 어시스턴트이기 때문에 정확한 답변은 어렵지만, 일반적으로는 각 포인트 ..."
34,치홀트는 뉴타이포그래피가 어떤 형태를 해야한다고 주장했어,'.\n\n좋은 질문입니다. 저는 인공지능 언어모델로써 치홀트의 발언을 인지하고 응...
43,정말 무슨생각인지 궁금하네,"'죄송합니다, 제가 인공지능 봇이므로 무슨 생각인지 파악할 수 없습니다. 하지만 어..."
53,몇일정도 썸타?,"'제가 AI 챗봇이라서 썸타이머를 예측할 수 있는 능력은 없지만, 썸이 얼마나 지속..."
59,면담을 통해서 노동 계층 어린이들은 무엇이 한정적이라고 판단했나?,"'저는 AI 어시스턴트이기 때문에, 노동 계층 어린이들이 면담에서 무엇이 한정적인지..."
64,갤럭시 S6 엣지와 비교했을 때 갤럭시 S7 엣지 의 센서 면적은 약 몇 프로 커졌는가?,"'저는 인공지능 언어모델이며, 정확한 정보를 알 수 없습니다. 하지만 갤럭시 S7 ..."


In [ ]:
# ============================================================
# Step 7-13. 이미 확인한 질문을 진단용으로 격리
# ============================================================

seen_test_prompts = set(
    fixed_eval_df["prompt"].tolist()
)

final_test_df = (
    test_eval_df[
        ~test_eval_df["prompt"].isin(seen_test_prompts)
    ]
    .copy()
    .reset_index(drop=True)
)

diagnostic_test_df = (
    test_eval_df[
        test_eval_df["prompt"].isin(seen_test_prompts)
    ]
    .copy()
    .reset_index(drop=True)
)

print("진단용으로 이미 본 데이터 :", len(diagnostic_test_df))
print("아직 보지 않은 최종 Test :", len(final_test_df))

진단용으로 이미 본 데이터 : 30
아직 보지 않은 최종 Test : 1137


### 이미 본 Test 데이터 격리

- 초기 모델 비교를 위해 Test 데이터 중 30개 prompt의 실제 생성 결과를 이미 확인했다.
- 사람이 결과를 본 뒤 모델이나 전처리 전략을 수정하면 해당 샘플은 더 이상 완전히 독립적인 최종 평가 데이터라고 보기 어렵다.
- 따라서 이미 확인한 30개를 diagnostic set으로 분리하고, 아직 결과를 확인하지 않은 나머지 Test 데이터를 최종 평가용으로 보존했다.
- 이후 데이터 정제와 decoding 선택에는 train과 validation만 사용하고 최종 Test는 모든 설정이 확정된 뒤 한 번만 평가한다.

In [ ]:
# ============================================================
# Step 7-14. Clean-v1에 남아 있는 AI 상투 응답 규모 확인
# ============================================================

clean_train_df["ai_boilerplate_v2"] = (
    clean_train_df["completion"]
    .apply(has_ai_boilerplate)
)

boilerplate_count_v1 = int(
    clean_train_df["ai_boilerplate_v2"].sum()
)

print(
    "Clean-v1 전체:",
    len(clean_train_df)
)

print(
    "AI 상투 응답:",
    boilerplate_count_v1
)

print(
    "AI 상투 응답 비율:",
    f"{boilerplate_count_v1 / len(clean_train_df) * 100:.2f}%"
)

Clean-v1 전체: 9321
AI 상투 응답: 1845
AI 상투 응답 비율: 19.79%


In [ ]:
# ============================================================
# Step 7-15. AI 상투 응답을 추가 제거한 Clean-v2 생성
# ============================================================

clean_v2_train_df = (
    clean_train_df[
        ~clean_train_df["ai_boilerplate_v2"]
    ]
    .copy()
    .reset_index(drop=True)
)

removed_v2_df = (
    clean_train_df[
        clean_train_df["ai_boilerplate_v2"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("[Clean-v2]")
print("Clean-v1 :", len(clean_train_df))
print("추가 제외 :", len(removed_v2_df))
print("Clean-v2 :", len(clean_v2_train_df))

print(
    "Raw 대비 최종 유지율:",
    f"{len(clean_v2_train_df) / len(raw_train_df) * 100:.2f}%"
)

[Clean-v2]
Clean-v1 : 9321
추가 제외 : 1845
Clean-v2 : 7476
Raw 대비 최종 유지율: 78.02%


In [ ]:
# ============================================================
# Step 7-16. Clean-v2 데이터 저장
# ============================================================

CLEAN_V2_TRAIN_PATH = (
    DATA_DIR / "sft_clean_v2_train.json"
)

REMOVED_V2_PATH = (
    DATA_DIR / "sft_clean_v2_removed.json"
)

save_records(
    clean_v2_train_df,
    CLEAN_V2_TRAIN_PATH,
    ["prompt", "completion"],
)

save_records(
    removed_v2_df,
    REMOVED_V2_PATH,
    ["prompt", "completion"],
)

sft_clean_v2_train.json          7476개
sft_clean_v2_removed.json        1845개


In [ ]:
# ============================================================
# Step 7-17. Clean-v2 SFT 학습 데이터 구성
# ============================================================

clean_v2_sft_dataset = (
    SupervisedSFTDataset(
        clean_v2_train_df,
        tokenizer,
        max_length=MAX_LENGTH,
    )
)

print("Clean-v2 데이터 :", len(clean_v2_train_df))
print("학습 데이터     :", len(clean_v2_sft_dataset))
print(
    "Response 소실 제외:",
    clean_v2_sft_dataset.dropped_samples
)

Clean-v2 데이터 : 7476
학습 데이터     : 7476
Response 소실 제외: 0


In [ ]:
# ============================================================
# Step 7-18. Clean-v2로 M2b SFT 학습
# ============================================================

M2B_CHECKPOINT_DIR = (
    LOCAL_OUTPUTS
    / "m2b_clean_v2_checkpoints"
)

M2B_MODEL_DIR = (
    LOCAL_MODELS
    / "m2b_clean_v2_sft"
)


m2b_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
)

m2b_model.config.use_cache = False


m2b_training_args = TrainingArguments(
    output_dir=str(M2B_CHECKPOINT_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,

    learning_rate=5e-5,
    warmup_steps=5,

    bf16=True,
    fp16=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    prediction_loss_only=True,

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=1,

    seed=42,
    data_seed=42,

    report_to="none",
)


m2b_trainer = Trainer(
    model=m2b_model,
    args=m2b_training_args,

    train_dataset=clean_v2_sft_dataset,

    # M1, M2와 같은 validation
    eval_dataset=sft_validation_dataset,

    data_collator=sft_data_collator,
)


m2b_train_result = (
    m2b_trainer.train()
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,2.809507,2.660196


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# ============================================================
# Step 7-19. M2b 학습 결과 확인
# ============================================================

m2b_eval_result = (
    m2b_trainer.evaluate()
)

print("[M2b 학습 결과]")
print(m2b_train_result.metrics)

print("\n[M2b 검증 결과]")
print(m2b_eval_result)

m2b_trainer.model.config.use_cache = True

m2b_trainer.save_model(
    str(M2B_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(M2B_MODEL_DIR)
)

Training Loss,Validation Loss,Epoch
2.809507,2.660196,1


[M2b 학습 결과]
{'train_runtime': 117.916, 'train_samples_per_second': 63.401, 'train_steps_per_second': 1.984, 'total_flos': 880923631104000.0, 'train_loss': 2.944756279643784, 'epoch': 1.0}

[M2b 검증 결과]
{'eval_loss': 2.660195827484131}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gd08/models/m2b_clean_v2_sft/tokenizer_config.json',
 '/content/gd08/models/m2b_clean_v2_sft/tokenizer.json')

In [ ]:
# ============================================================
# Step 7-20. Clean-v2 목표와 정렬된 Validation 생성
# ============================================================

validation_v2_df = (
    validation_eval_df[
        ~validation_eval_df["completion"]
        .apply(has_ai_boilerplate)
    ]
    .copy()
    .reset_index(drop=True)
)

print("[Validation-v2]")
print(
    "기존 Validation :",
    len(validation_eval_df)
)

print(
    "AI 상투 응답 제외:",
    len(validation_eval_df)
    - len(validation_v2_df)
)

print(
    "최종 Validation :",
    len(validation_v2_df)
)

print(
    "유지율:",
    f"{len(validation_v2_df) / len(validation_eval_df) * 100:.2f}%"
)

[Validation-v2]
기존 Validation : 1181
AI 상투 응답 제외: 258
최종 Validation : 923
유지율: 78.15%


In [ ]:
# ============================================================
# Step 7-21. 최종 평가용 Test-v2 기준 고정
# ============================================================

final_test_v2_df = (
    final_test_df[
        ~final_test_df["completion"]
        .apply(has_ai_boilerplate)
    ]
    .copy()
    .reset_index(drop=True)
)

print("[Final Test-v2]")
print(
    "기존 Final Test :",
    len(final_test_df)
)

print(
    "최종 Test-v2   :",
    len(final_test_v2_df)
)

print(
    "유지율:",
    f"{len(final_test_v2_df) / len(final_test_df) * 100:.2f}%"
)

[Final Test-v2]
기존 Final Test : 1137
최종 Test-v2   : 901
유지율: 79.24%


In [ ]:
VALIDATION_V2_PATH = (
    DATA_DIR / "sft_validation_v2.json"
)

FINAL_TEST_V2_PATH = (
    DATA_DIR / "sft_final_test_v2.json"
)

save_records(
    validation_v2_df,
    VALIDATION_V2_PATH,
    ["prompt", "completion"],
)

save_records(
    final_test_v2_df,
    FINAL_TEST_V2_PATH,
    ["prompt", "completion"],
)

sft_validation_v2.json            923개
sft_final_test_v2.json            901개


In [ ]:
# ============================================================
# Step 7-22. M2와 M2b를 동일한 Validation-v2에서 비교
# ============================================================

sft_validation_v2_dataset = (
    SupervisedSFTDataset(
        validation_v2_df,
        tokenizer,
        max_length=MAX_LENGTH,
    )
)

print(
    "Validation-v2 학습 형식 데이터:",
    len(sft_validation_v2_dataset)
)

Validation-v2 학습 형식 데이터: 923


In [ ]:
m2_v2_eval = m2_trainer.evaluate(
    eval_dataset=sft_validation_v2_dataset
)

print(
    "[M2 Clean-v1 / Validation-v2]"
)
print(m2_v2_eval)

Training Loss,Validation Loss,Epoch
2.720414,2.709270,1


[M2 Clean-v1 / Validation-v2]
{'eval_loss': 2.7092695236206055}


In [ ]:
m2b_v2_eval = m2b_trainer.evaluate(
    eval_dataset=sft_validation_v2_dataset
)

print(
    "[M2b Clean-v2 / Validation-v2]"
)
print(m2b_v2_eval)

Training Loss,Validation Loss,Epoch
2.809507,2.722674,1


[M2b Clean-v2 / Validation-v2]
{'eval_loss': 2.7226736545562744}


In [ ]:
# ============================================================
# Step 7-23. Clean-v2용 Decoding 검증 질문 100개 고정
# ============================================================

decoding_validation_v2_df = (
    validation_v2_df[
        ["prompt", "completion"]
    ]
    .copy()
)

decoding_validation_v2_df["prompt_hash"] = (
    decoding_validation_v2_df["prompt"]
    .apply(prompt_hash)
)

decoding_validation_v2_df = (
    decoding_validation_v2_df
    .sort_values("prompt_hash")
    .head(100)
    .reset_index(drop=True)
)

print(
    "Validation-v2 decoding 질문:",
    len(decoding_validation_v2_df)
)

Validation-v2 decoding 질문: 100


In [ ]:
# ============================================================
# Step 7-24. M2b에서 Decoding 전략 재비교
# ============================================================

m2b_model = m2b_trainer.model
m2b_model.config.use_cache = True
m2b_model.eval()


m2b_decoding_results = []


for config_name, config in DECODING_CONFIGS.items():

    print(
        f"\n[M2b / {config_name}] 생성 시작"
    )

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    start_time = time.time()


    for row in decoding_validation_v2_df.itertuples(
        index=False
    ):

        response = generate_with_config(
            model=m2b_model,
            prompt=row.prompt,
            tokenizer=tokenizer,
            generation_config=config,
        )

        m2b_decoding_results.append({
            "config": config_name,
            "prompt": row.prompt,
            "reference": row.completion,
            "response": response,
        })


    print(
        "완료:",
        f"{time.time() - start_time:.1f}초"
    )


m2b_decoding_results_df = pd.DataFrame(
    m2b_decoding_results
)


[M2b / D0_greedy] 생성 시작
완료: 41.8초

[M2b / D1_beam4] 생성 시작
완료: 59.8초

[M2b / D2_top_p] 생성 시작
완료: 46.2초

[M2b / D3_top_k] 생성 시작
완료: 54.0초


In [ ]:
# ============================================================
# Step 7-25. M2b Decoding 정량평가
# ============================================================

quality_rows = []
reference_rows = []


for config_name, group in (
    m2b_decoding_results_df.groupby("config")
):

    responses = group["response"]

    quality_rows.append({
        "config": config_name,

        "빈 응답 수":
            int(
                responses
                .fillna("")
                .astype(str)
                .str.strip()
                .eq("")
                .sum()
            ),

        "AI 상투 응답 수":
            int(
                responses
                .apply(has_ai_boilerplate)
                .sum()
            ),

        "평균 한글 비율":
            float(
                responses
                .apply(korean_ratio)
                .mean()
            ),

        "평균 응답 tokens":
            float(
                responses
                .apply(count_tokens)
                .mean()
            ),
    })

    metrics = calculate_generation_metrics(
        group
    )

    reference_rows.append({
        "config": config_name,
        **metrics,
    })


m2b_quality_df = pd.DataFrame(
    quality_rows
)

m2b_reference_df = pd.DataFrame(
    reference_rows
)

m2b_summary_df = (
    m2b_quality_df
    .merge(
        m2b_reference_df,
        on="config",
    )
    .sort_values("config")
)

display(m2b_summary_df)

,config,빈 응답 수,AI 상투 응답 수,평균 한글 비율,평균 응답 tokens,ROUGE-L,BLEU,chrF
0,D0_greedy,0,7,0.869159,33.13,0.113485,1.315656,10.387341
1,D1_beam4,0,16,0.835039,40.01,0.200803,6.338329,13.083728
2,D2_top_p,0,5,0.865563,35.88,0.111228,1.166521,9.653455
3,D3_top_k,0,5,0.832091,42.95,0.092649,1.069500,10.264225


In [ ]:
# ============================================================
# Step 7-26. M2와 M2b의 실제 optimizer update 수 확인
# ============================================================

print(
    "M2 optimizer steps:",
    m2_trainer.state.global_step
)

print(
    "M2b optimizer steps:",
    m2b_trainer.state.global_step
)

M2 optimizer steps: 292
M2b optimizer steps: 234


In [ ]:
# ============================================================
# Step 7-27. M2와 같은 update 수로 Clean-v2 재학습
# ============================================================

MATCHED_STEPS = (
    m2_trainer.state.global_step
)

M2C_CHECKPOINT_DIR = (
    LOCAL_OUTPUTS
    / "m2c_clean_v2_stepmatched_checkpoints"
)

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)


m2c_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
)

m2c_model.config.use_cache = False


m2c_training_args = TrainingArguments(
    output_dir=str(M2C_CHECKPOINT_DIR),

    # epoch 대신 M2와 동일한 update 수를 고정
    max_steps=MATCHED_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,

    learning_rate=5e-5,
    warmup_steps=5,

    bf16=True,
    fp16=False,

    eval_strategy="steps",

    # 마지막에서 validation을 확인하면 충분
    eval_steps=MATCHED_STEPS,

    save_strategy="steps",
    save_steps=MATCHED_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    prediction_loss_only=True,

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=1,

    seed=42,
    data_seed=42,

    report_to="none",
)


m2c_trainer = Trainer(
    model=m2c_model,
    args=m2c_training_args,

    train_dataset=clean_v2_sft_dataset,

    # Clean-v2 목표와 정렬된 validation
    eval_dataset=sft_validation_v2_dataset,

    data_collator=sft_data_collator,
)


m2c_train_result = (
    m2c_trainer.train()
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss,Validation Loss
292,2.460683,2.720076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# ============================================================
# Step 7-28. 동일 update 조건의 Clean-v2 결과 확인
# ============================================================

m2c_eval_result = (
    m2c_trainer.evaluate(
        eval_dataset=sft_validation_v2_dataset
    )
)

print("[M2]")
print(
    "steps:",
    m2_trainer.state.global_step
)
print(
    "Validation-v2:",
    m2_v2_eval["eval_loss"]
)

print("\n[M2b - 1 epoch]")
print(
    "steps:",
    m2b_trainer.state.global_step
)
print(
    "Validation-v2:",
    m2b_v2_eval["eval_loss"]
)

print("\n[M2c - step matched]")
print(
    "steps:",
    m2c_trainer.state.global_step
)
print(
    "Validation-v2:",
    m2c_eval_result["eval_loss"]
)

Training Loss,Validation Loss,Step
2.460683,2.720076,292


[M2]
steps: 292
Validation-v2: 2.7092695236206055

[M2b - 1 epoch]
steps: 234
Validation-v2: 2.7226736545562744

[M2c - step matched]
steps: 292
Validation-v2: 2.720076084136963


| 모델                        | 학습 데이터 | optimizer steps | Validation-v2 loss |  Perplexity |
| ------------------------- | -----: | --------------: | -----------------: | ----------: |
| M2 Clean-v1               |  9,321 |             292 |         **2.7093** | 약 **15.02** |
| M2b Clean-v2              |  7,476 |             234 |             2.7227 |     약 15.22 |
| M2c Clean-v2 step-matched |  7,476 |         **292** |         **2.7201** | 약 **15.18** |




```
Clean-v2 loss 상승 원인

데이터 감소로 학습 step 감소
        │
        └─ 일부 영향 있음
           2.7227 → 2.7201

+

1,845개 데이터 자체를 제거한 영향
        │
        └─ 여전히 남음
           M2 2.7093 < M2c 2.7201
```



### validation loss가 가장 낮은 모델이 반드시 우리가 원하는 생성 행동을 가장 잘하는 모델은 아니다.

In [ ]:
# ============================================================
# Step 7-29. Step-matched Clean-v2 모델의 Beam 생성 평가
# ============================================================

m2c_model = m2c_trainer.model
m2c_model.config.use_cache = True
m2c_model.eval()


M2C_BEAM_CONFIG = {
    "do_sample": False,
    "num_beams": 4,
    "early_stopping": True,
}


m2c_beam_rows = []


for row in decoding_validation_v2_df.itertuples(
    index=False
):

    response = generate_with_config(
        model=m2c_model,
        prompt=row.prompt,
        tokenizer=tokenizer,
        generation_config=M2C_BEAM_CONFIG,
    )

    m2c_beam_rows.append({
        "config": "M2c_beam4",
        "prompt": row.prompt,
        "reference": row.completion,
        "response": response,
    })


m2c_beam_df = pd.DataFrame(
    m2c_beam_rows
)

print(
    "생성 완료:",
    len(m2c_beam_df)
)

생성 완료: 100


In [ ]:
# ============================================================
# Step 7-30. M2b와 M2c의 Beam 결과 비교
# ============================================================

def summarize_generation(group, model_name):

    metrics = calculate_generation_metrics(
        group
    )

    responses = group["response"]

    return {
        "model": model_name,

        "AI 상투 응답 수":
            int(
                responses
                .apply(has_ai_boilerplate)
                .sum()
            ),

        "평균 한글 비율":
            float(
                responses
                .apply(korean_ratio)
                .mean()
            ),

        "평균 응답 tokens":
            float(
                responses
                .apply(count_tokens)
                .mean()
            ),

        **metrics,
    }


m2b_beam_df = (
    m2b_decoding_results_df[
        m2b_decoding_results_df["config"]
        == "D1_beam4"
    ]
)


final_sft_comparison = pd.DataFrame([
    summarize_generation(
        m2b_beam_df,
        "M2b Clean-v2 1 epoch",
    ),

    summarize_generation(
        m2c_beam_df,
        "M2c Clean-v2 step-matched",
    ),
])


display(
    final_sft_comparison
)

,model,AI 상투 응답 수,평균 한글 비율,평균 응답 tokens,ROUGE-L,BLEU,chrF
0,M2b Clean-v2 1 epoch,16,0.835039,40.01,0.200803,6.338329,13.083728
1,M2c Clean-v2 step-matched,2,0.875810,32.71,0.199831,5.128953,12.062210


| 지표       | M2b Clean-v2 | M2c step-matched | 해석           |
| -------- | -----------: | ---------------: | ------------ |
| AI 상투 응답 |       16/100 |        **2/100** | **87.5% 감소** |
| 평균 한글 비율 |       83.50% |       **87.58%** | +4.08%p      |
| 평균 응답 길이 |        40.01 |        **32.71** | 약 18.2% 짧아짐  |
| ROUGE-L  | **0.200803** |         0.199831 | 약 0.5% 감소    |
| BLEU     |    **6.338** |            5.129 | 약 19.1% 감소   |
| chrF     |   **13.084** |           12.062 | 약 7.8% 감소    |


## SFT 실험 구조 정리


```
                    KoGPT-2
                      θ₀
                       │
          ┌────────────┴────────────┐
          │                         │
          ▼                         ▼
     M1 Raw SFT                데이터 분석
     9,582 samples                  │
          │                         │
          │                 영어 오염 발견
          │                         │
          │                         ▼
          │                   Clean-v1
          │                     9,321
          │                         │
          │                         ▼
          │                      M2 SFT
          │
          │                 생성 결과 분석
          │                         │
          │          한국어 AI boilerplate 발견
          │                         │
          │                         ▼
          │                   Clean-v2
          │                     7,476
          │                         │
          │              ┌──────────┴──────────┐
          │              ▼                     ▼
          │          M2b 1 epoch          M2c step matched
          │           234 steps             292 steps
          │                                      │
          └──────────────────────────────────────┤
                                                 ▼
                                      AI boilerplate
                                           2 / 100
```



### 학습 기록 — 최종 SFT 모델 선정

- Clean-v2 1 epoch 모델과 M2의 optimizer update 수를 맞춘 step-matched 모델을 추가 비교했다.
- M2c는 동일한 Beam Search 조건에서 AI 상투 응답을 100개 중 16개에서 2개로 줄였다.
- 평균 한글 비율도 약 83.5%에서 87.6%로 증가했다.
- ROUGE-L은 약 0.2008에서 0.1998로 거의 유지되었으나 BLEU와 chrF는 감소했다.
- M2c의 평균 응답 길이는 약 40 tokens에서 33 tokens로 짧아졌으며, 불필요한 자기소개형 표현 감소가 reference 기반 n-gram 지표 하락에 일부 영향을 주었을 가능성이 있다.
- 따라서 M2c를 모든 metric에서 우수한 모델이라고 해석하지 않고, 이번 프로젝트의 핵심 목표인 AI 상투 응답 억제와 한국어 응답 품질 측면에서 최종 SFT 후보로 선정했다.
- validation loss, reference similarity metric, 실제 생성 행동이 서로 다른 품질 측면을 측정한다는 것을 실험을 통해 확인했다.

## Reward Model


```
KoGPT-2 backbone
      ↓
Masked Mean Pooling
      ↓
Linear(hidden → 1)
      ↓
scalar reward
```



In [ ]:
# ============================================================
# Step 8-1. Reward Model 데이터의 ranking 구조 확인
# ============================================================

print("[RM 데이터 크기]")
print(rm_df.shape)

print("\n[컬럼]")
print(rm_df.columns.tolist())

print("\n[ranking 타입]")
print(
    rm_df["ranking"]
    .apply(type)
    .value_counts()
)

print("\n[ranking 값 예시]")
display(
    rm_df[
        [
            "prompt",
            "completion_0",
            "completion_1",
            "completion_2",
            "ranking",
        ]
    ].head(10)
)

[RM 데이터 크기]
(10220, 9)

[컬럼]
['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking', 'prompt_chars', 'completion_0_chars', 'completion_1_chars', 'completion_2_chars']

[ranking 타입]
ranking
<class 'list'>    10220
Name: count, dtype: int64

[ranking 값 예시]


,prompt,completion_0,completion_1,completion_2,ranking
0,"번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독...",Allow me to answer your question. I know that ...,번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.,라이언에게 말했다.,"[2, 1, 0]"
1,개포주공아파트는 몇 단지로 이루어져 있나?,개포주공아파트는 다섯 단지로 이루어져 있습니다.,이날 목송에서 구글상위노,개포주공아파트는 총 27개 단지로 이루어져 있습니다.,"[2, 0, 1]"
2,김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?,The diameter of the Metallic domain is bigger ...,"이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가...",김영삼의 후보 시절에 지역표심을 겨냥한 발언은 대통령 당선 전까지 대한민국 정부가 ...,"[1, 2, 0]"
3,페르시아로부터 기원된 아랍요리의 특징이 뭐야,"1. 다양한 스파이스와 허브 사용\n2. 쌀, 콩, 나물, 고기, 양파 등 다양한 ...","- 아랍요리는 소금, 산 재료, 일부 채소, 고기, 생선 또는 난류의 조합으로 만들...",위한 태야 대표 기원들 개인이라 겪고 태야\n\n\n귀식 걸 공개한 구문\n\n사띩...,"[0, 1, 2]"
4,중화인민공화국의 기본 법률은 누가 제정 및 개정하는가?,중화인민공화국의 기본 법률은 중국인민대표대회(국회)가 제정하고 개정합니다.,The limit of the Office of the Vice President is?,중화인민공화국의 기본 법률은 중화인민공화국의 의회가 제정 및 개정합니다.,"[0, 2, 1]"
5,송편은 요 얼마예요?,요건들 개인사이들 한국 인기후들 한국 인기를 구과하지 않돼야 한국 인기의 의료문명을...,운임은 각 상황에 따라 다르게 책정됩니다. 자세한 내용은 당사 전용 고객 서비스 센...,저는 AI 어시스턴트로서 물건을 판매하는 기능은 제공하지 않습니다. 죄송합니다.,"[2, 1, 0]"
6,"#이름#로 예약했는데, 확인 부탁드려도 될까요?",I'm not sure.,"네, 예약하신 #이름#으로 확인해 드리겠습니다. 감사합니다.","죄송하지만 제가 예약에 대한 정보를 확인할 수 없기 때문에, 예약을 하신 호텔 또는...","[2, 1, 0]"
7,지루가 데뷔골을 넣은 경기는 어느팀과의 경기였나?,지루가 데뷔골을 넣은 경기는 2016년 8월 20일 KRC 격돌 웨스트로엔 경기였습니다.,제목들 개인권 회장 개인권 회장\n\n1. 한국 트랙 개인권 회장\n\n2. 고츠 ...,지루가 데뷔골을 넣은 경기는 2019년 11월 23일 AFC 챔피언스리그 그룹스테이...,"[0, 2, 1]"
8,비탈리 쿠투조프가 AC 밀란 첫 시즌에 몇 경기 출전했어,요?\n\n비탈리 쿠투조프는 2020-2021 시즌 AC 밀란에서 34 경기를 출전...,라\n\n비타트링 토마티생 트라스콥 곤러라 \n\n비타트링 토마티생 트라스콥 곤러라,"Sorry, I am an AI language model and I am not ...","[2, 0, 1]"
9,부자 되게 해주세요,"저는 AI 어시스턴트로서 부동산, 주식, 사업 등 부의 창출 방법에 대한 조언을 드...",yalehtneuna.com yalehtneuna.com/ territories\n...,부자를 되게 하기 위해서는 여러가지 카테고리로 나눌 수 있습니다. \n\n1. 자금...,"[0, 2, 1]"


In [ ]:
# ============================================================
# Step 8-2. ranking 값의 분포 확인
# ============================================================

ranking_counts = (
    rm_df["ranking"]
    .astype(str)
    .value_counts()
)

display(
    ranking_counts.head(20)
)

,count
ranking,
"[1, 2, 0]",1769
"[0, 2, 1]",1760
"[1, 0, 2]",1696
"[2, 0, 1]",1681
"[2, 1, 0]",1676
"[0, 1, 2]",1638


### 필수로 이해하고 갑시다!



```
chosen
= 선호되는 응답

rejected
= 덜 선호되는 응답

reward
= 응답에 부여되는 scalar score

margin
= reward(chosen) - reward(rejected)
```



In [ ]:
# ============================================================
# Step 8-3. RM ranking 구조의 무결성 확인
# ============================================================

def valid_ranking(ranking):
    return (
        isinstance(ranking, list)
        and len(ranking) == 3
        and sorted(ranking) == [0, 1, 2]
    )


rm_df["valid_ranking"] = (
    rm_df["ranking"]
    .apply(valid_ranking)
)


print(
    "전체:",
    len(rm_df)
)

print(
    "정상 ranking:",
    rm_df["valid_ranking"].sum()
)

print(
    "비정상 ranking:",
    (~rm_df["valid_ranking"]).sum()
)

전체: 10220
정상 ranking: 10220
비정상 ranking: 0


In [ ]:
# ============================================================
# Step 8-4. RM에서 사용할 최소 무결성 데이터 생성
# ============================================================

RM_COMPLETION_COLS = [
    "completion_0",
    "completion_1",
    "completion_2",
]


rm_valid_df = rm_df.copy()


rm_valid_df["blank_prompt"] = (
    rm_valid_df["prompt"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)


for col in RM_COMPLETION_COLS:

    rm_valid_df[f"blank_{col}"] = (
        rm_valid_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )


invalid_completion_mask = (
    rm_valid_df[
        [
            f"blank_{col}"
            for col in RM_COMPLETION_COLS
        ]
    ]
    .any(axis=1)
)


rm_invalid_mask = (
    rm_valid_df["blank_prompt"]
    |
    invalid_completion_mask
    |
    (~rm_valid_df["valid_ranking"])
)


rm_invalid_df = (
    rm_valid_df[
        rm_invalid_mask
    ]
    .copy()
)

rm_valid_df = (
    rm_valid_df[
        ~rm_invalid_mask
    ]
    .copy()
    .reset_index(drop=True)
)


print("[RM 최소 무결성 정제]")

print(
    "원본:",
    len(rm_df)
)

print(
    "제외:",
    len(rm_invalid_df)
)

print(
    "사용:",
    len(rm_valid_df)
)

print(
    "유지율:",
    f"{len(rm_valid_df) / len(rm_df) * 100:.2f}%"
)

[RM 최소 무결성 정제]
원본: 10220
제외: 33
사용: 10187
유지율: 99.68%


In [ ]:
# ============================================================
# Step 8-5. Prompt 단위로 RM Train / Validation / Test 분리
# ============================================================

import hashlib
import re


def normalize_prompt(text):

    text = str(text).strip()

    return re.sub(
        r"\s+",
        " ",
        text,
    )


def rm_split_from_prompt(prompt):

    normalized = normalize_prompt(
        prompt
    )

    hash_value = int(
        hashlib.sha256(
            normalized.encode("utf-8")
        ).hexdigest(),
        16,
    )

    bucket = hash_value % 100

    if bucket < 80:
        return "train"

    elif bucket < 90:
        return "validation"

    else:
        return "test"


rm_valid_df["normalized_prompt"] = (
    rm_valid_df["prompt"]
    .apply(normalize_prompt)
)

rm_valid_df["split"] = (
    rm_valid_df["prompt"]
    .apply(rm_split_from_prompt)
)


print(
    rm_valid_df["split"]
    .value_counts()
)

print(
    "\n비율"
)

print(
    rm_valid_df["split"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

split
train         8088
validation    1079
test          1020
Name: count, dtype: int64

비율
split
train         79.40
validation    10.59
test          10.01
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# Step 8-6. RM Prompt leakage 확인
# ============================================================

train_prompts = set(
    rm_valid_df.loc[
        rm_valid_df["split"] == "train",
        "normalized_prompt",
    ]
)

validation_prompts = set(
    rm_valid_df.loc[
        rm_valid_df["split"] == "validation",
        "normalized_prompt",
    ]
)

test_prompts = set(
    rm_valid_df.loc[
        rm_valid_df["split"] == "test",
        "normalized_prompt",
    ]
)


print(
    "Train ↔ Validation overlap:",
    len(
        train_prompts
        & validation_prompts
    )
)

print(
    "Train ↔ Test overlap:",
    len(
        train_prompts
        & test_prompts
    )
)

print(
    "Validation ↔ Test overlap:",
    len(
        validation_prompts
        & test_prompts
    )
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0


In [ ]:
# ============================================================
# Step 8-7. 3개 응답 ranking을 chosen / rejected pair로 변환
# ============================================================

PAIR_INDICES = [
    (0, 1),
    (0, 2),
    (1, 2),
]


def expand_rm_pairs(df):

    pairs = []

    for row in df.itertuples(
        index=False
    ):

        completions = [
            row.completion_0,
            row.completion_1,
            row.completion_2,
        ]

        ranks = row.ranking


        for left, right in PAIR_INDICES:

            # 숫자가 작을수록 더 선호되는 응답
            if ranks[left] < ranks[right]:

                chosen_idx = left
                rejected_idx = right

            else:

                chosen_idx = right
                rejected_idx = left


            pairs.append({

                "prompt":
                    row.prompt,

                "chosen":
                    completions[chosen_idx],

                "rejected":
                    completions[rejected_idx],

                "chosen_rank":
                    ranks[chosen_idx],

                "rejected_rank":
                    ranks[rejected_idx],

                # 0 vs 2이면 2,
                # 0 vs 1 또는 1 vs 2이면 1
                "rank_gap":
                    ranks[rejected_idx]
                    - ranks[chosen_idx],

            })

    return pd.DataFrame(
        pairs
    )

In [ ]:
# ============================================================
# Step 8-8. Split별 Pairwise RM 데이터 생성
# ============================================================

rm_train_pairs = expand_rm_pairs(
    rm_valid_df[
        rm_valid_df["split"] == "train"
    ]
)

rm_validation_pairs = expand_rm_pairs(
    rm_valid_df[
        rm_valid_df["split"] == "validation"
    ]
)

rm_test_pairs = expand_rm_pairs(
    rm_valid_df[
        rm_valid_df["split"] == "test"
    ]
)


print("[RM Pair 데이터]")

print(
    "Train      :",
    len(rm_train_pairs)
)

print(
    "Validation :",
    len(rm_validation_pairs)
)

print(
    "Test       :",
    len(rm_test_pairs)
)

print(
    "Total      :",
    (
        len(rm_train_pairs)
        + len(rm_validation_pairs)
        + len(rm_test_pairs)
    )
)

[RM Pair 데이터]
Train      : 24264
Validation : 3237
Test       : 3060
Total      : 30561


loss는 PairWiseLoss를 쓴다.

```
                 Reward Model
                     θ
                    / \
                   /   \
                  ▼     ▼

chosen          r_c     r_r         rejected
좋은 응답       2.4     0.7         나쁜 응답

                  │
                  ▼

margin = 2.4 - 0.7
       = +1.7

                  │
                  ▼

positive margin
→ 올바른 ranking
→ loss 작아짐
```

반대로

```
chosen    -0.2
rejected   0.8

margin = -1.0

→ 순서를 거꾸로 판단
→ loss 커짐
```



In [ ]:
# ============================================================
# Step 8-9. chosen / rejected 변환 결과 직접 확인
# ============================================================

display(
    rm_train_pairs[
        [
            "prompt",
            "chosen",
            "rejected",
            "chosen_rank",
            "rejected_rank",
            "rank_gap",
        ]
    ].head(15)
)

,prompt,chosen,rejected,chosen_rank,rejected_rank,rank_gap
0,개포주공아파트는 몇 단지로 이루어져 있나?,이날 목송에서 구글상위노,개포주공아파트는 다섯 단지로 이루어져 있습니다.,0,2,2
1,개포주공아파트는 몇 단지로 이루어져 있나?,개포주공아파트는 총 27개 단지로 이루어져 있습니다.,개포주공아파트는 다섯 단지로 이루어져 있습니다.,1,2,1
2,개포주공아파트는 몇 단지로 이루어져 있나?,이날 목송에서 구글상위노,개포주공아파트는 총 27개 단지로 이루어져 있습니다.,0,1,1
3,김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?,The diameter of the Metallic domain is bigger ...,"이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가...",1,2,1
4,김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?,김영삼의 후보 시절에 지역표심을 겨냥한 발언은 대통령 당선 전까지 대한민국 정부가 ...,The diameter of the Metallic domain is bigger ...,0,1,1
5,김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?,김영삼의 후보 시절에 지역표심을 겨냥한 발언은 대통령 당선 전까지 대한민국 정부가 ...,"이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가...",0,2,2
6,페르시아로부터 기원된 아랍요리의 특징이 뭐야,"1. 다양한 스파이스와 허브 사용\n2. 쌀, 콩, 나물, 고기, 양파 등 다양한 ...","- 아랍요리는 소금, 산 재료, 일부 채소, 고기, 생선 또는 난류의 조합으로 만들...",0,1,1
7,페르시아로부터 기원된 아랍요리의 특징이 뭐야,"1. 다양한 스파이스와 허브 사용\n2. 쌀, 콩, 나물, 고기, 양파 등 다양한 ...",위한 태야 대표 기원들 개인이라 겪고 태야\n\n\n귀식 걸 공개한 구문\n\n사띩...,0,2,2
8,페르시아로부터 기원된 아랍요리의 특징이 뭐야,"- 아랍요리는 소금, 산 재료, 일부 채소, 고기, 생선 또는 난류의 조합으로 만들...",위한 태야 대표 기원들 개인이라 겪고 태야\n\n\n귀식 걸 공개한 구문\n\n사띩...,1,2,1
9,중화인민공화국의 기본 법률은 누가 제정 및 개정하는가?,중화인민공화국의 기본 법률은 중국인민대표대회(국회)가 제정하고 개정합니다.,The limit of the Office of the Vice President is?,0,2,2


In [ ]:
assert (
    rm_train_pairs[
        "chosen_rank"
    ]
    <
    rm_train_pairs[
        "rejected_rank"
    ]
).all()

assert (
    rm_validation_pairs[
        "chosen_rank"
    ]
    <
    rm_validation_pairs[
        "rejected_rank"
    ]
).all()

assert (
    rm_test_pairs[
        "chosen_rank"
    ]
    <
    rm_test_pairs[
        "rejected_rank"
    ]
).all()

print(
    "Pairwise ranking 방향 검증 완료"
)

Pairwise ranking 방향 검증 완료


In [ ]:
# ============================================================
# Step 8-10. RM pair의 token 길이 분포 확인
# ============================================================

def rm_sequence_text(
    prompt,
    response,
):

    return (
        format_instruction(prompt)
        + str(response)
        + tokenizer.eos_token
    )


def rm_token_length(
    prompt,
    response,
):

    text = rm_sequence_text(
        prompt,
        response,
    )

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False,
        )
    )


rm_train_pairs[
    "chosen_tokens"
] = [
    rm_token_length(p, r)
    for p, r
    in zip(
        rm_train_pairs["prompt"],
        rm_train_pairs["chosen"],
    )
]

rm_train_pairs[
    "rejected_tokens"
] = [
    rm_token_length(p, r)
    for p, r
    in zip(
        rm_train_pairs["prompt"],
        rm_train_pairs["rejected"],
    )
]

rm_train_pairs[
    "pair_max_tokens"
] = (
    rm_train_pairs[
        [
            "chosen_tokens",
            "rejected_tokens",
        ]
    ]
    .max(axis=1)
)


print(
    rm_train_pairs[
        "pair_max_tokens"
    ]
    .describe(
        percentiles=[
            .90,
            .95,
            .99,
        ]
    )
)


for length in [
    128,
    192,
    256,
    320,
    384,
    512,
]:

    included = (
        rm_train_pairs[
            "pair_max_tokens"
        ]
        <= length
    ).sum()

    print(
        f"{length:>3} tokens |",
        f"포함 {included:>6} |",
        f"{included / len(rm_train_pairs) * 100:6.2f}%"
    )

count    24264.000000
mean       107.112265
std         75.796787
min         33.000000
50%         90.000000
90%        164.000000
95%        227.000000
99%        377.000000
max       1894.000000
Name: pair_max_tokens, dtype: float64
128 tokens | 포함  20146 |  83.03%
192 tokens | 포함  22476 |  92.63%
256 tokens | 포함  23420 |  96.52%
320 tokens | 포함  23884 |  98.43%
384 tokens | 포함  24034 |  99.05%
512 tokens | 포함  24154 |  99.55%


## RM 모델을 어떻게 만들 예정이냐 하면요,


```
                  KoGPT-2 pretrained θ₀
                     /             \
                    /               \
                   ▼                 ▼
             SFT Generator       Reward Model
                M2c                  RM
                 │                    │
                 │ Response           │ score
                 ▼                    ▼
              "답 생성"           "답 평가"

                     \             /
                      \           /
                       ▼         ▼
                           PPO
```



### RM ranking을 pairwise preference로 변환

- KoChatGPT RM 데이터의 `ranking`은 completion 인덱스의 정렬 순서가 아니라 각 completion에 대응하는 순위값이다.
- rank 값은 작을수록 우선순위가 높으며 `0`이 가장 선호되고 `2`가 가장 덜 선호된다.
- 원본 KoChatGPT 구현에서도 두 completion의 ranking 값을 비교하여 값이 작은 응답을 `chosen`, 큰 응답을 `rejected`로 변환한다.
- 하나의 prompt와 세 completion으로 구성된 ranking 데이터는 `(0,1)`, `(0,2)`, `(1,2)`의 세 pairwise sample로 확장된다.
- KoChatGPT의 RM ranking은 실제 사람이 모든 응답을 읽고 평가한 결과가 아니라 ChatGPT > GPT3-davinci > GPT3-ada라는 생성 모델의 우선순위를 이용한 자동 label이므로 실제 응답 품질과 ranking이 일치하지 않는 noise가 존재할 수 있다.
- SFT에서는 나쁜 응답을 제거하는 것이 중요했지만 RM에서는 좋은 응답과 비교할 rejected example이 필요하므로 영어·상투문구·낮은 품질의 응답을 일괄적으로 제거하지 않는다.
- 같은 prompt에서 생성된 세 pair가 Train/Validation/Test에 흩어지지 않도록 prompt 단위 split을 먼저 수행하고 그 이후 pairwise 데이터로 확장한다.
- Reward Model은 문장을 생성하는 모델이 아니라 `prompt + response`에 하나의 scalar reward를 부여하는 모델이며, 학습 목표는 `reward(chosen) > reward(rejected)`가 되도록 하는 것이다.

In [ ]:
# ============================================================
# Step 8-11. RM 최대 sequence 길이 확정
# ============================================================

RM_MAX_LENGTH = 256

print(
    "RM MAX_LENGTH:",
    RM_MAX_LENGTH
)

RM MAX_LENGTH: 256


In [ ]:
# ============================================================
# Step 8-12. Reward Model용 pair dataset 정의
# ============================================================

from torch.utils.data import Dataset


class RewardPairDataset(Dataset):

    def __init__(self, df):
        self.df = (
            df[
                [
                    "prompt",
                    "chosen",
                    "rejected",
                    "rank_gap",
                ]
            ]
            .reset_index(drop=True)
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        return {
            "prompt": row["prompt"],
            "chosen": row["chosen"],
            "rejected": row["rejected"],
            "rank_gap": int(row["rank_gap"]),
        }

In [ ]:
# ============================================================
# Step 8-13. Reward Model용 batch collator 정의
# ============================================================

class RewardDataCollator:

    def __init__(
        self,
        tokenizer,
        max_length=256,
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):

        chosen_texts = []
        rejected_texts = []
        rank_gaps = []

        for item in batch:

            prompt_text = format_instruction(
                item["prompt"]
            )

            chosen_texts.append(
                prompt_text
                + str(item["chosen"])
                + self.tokenizer.eos_token
            )

            rejected_texts.append(
                prompt_text
                + str(item["rejected"])
                + self.tokenizer.eos_token
            )

            rank_gaps.append(
                item["rank_gap"]
            )


        chosen_tokens = self.tokenizer(
            chosen_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            add_special_tokens=False,
        )

        rejected_tokens = self.tokenizer(
            rejected_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            add_special_tokens=False,
        )


        return {
            "chosen_input_ids":
                chosen_tokens["input_ids"],

            "chosen_attention_mask":
                chosen_tokens["attention_mask"],

            "rejected_input_ids":
                rejected_tokens["input_ids"],

            "rejected_attention_mask":
                rejected_tokens["attention_mask"],

            "rank_gap":
                torch.tensor(
                    rank_gaps,
                    dtype=torch.long,
                ),
        }


rm_data_collator = RewardDataCollator(
    tokenizer=tokenizer,
    max_length=RM_MAX_LENGTH,
)

In [ ]:
# ============================================================
# Step 8-14. RM 학습용 Dataset 생성
# ============================================================

rm_train_dataset = RewardPairDataset(
    rm_train_pairs
)

rm_validation_dataset = RewardPairDataset(
    rm_validation_pairs
)

rm_test_dataset = RewardPairDataset(
    rm_test_pairs
)


print(
    "Train      :",
    len(rm_train_dataset)
)

print(
    "Validation :",
    len(rm_validation_dataset)
)

print(
    "Test       :",
    len(rm_test_dataset)
)

Train      : 24264
Validation : 3237
Test       : 3060


In [ ]:
# ============================================================
# Step 8-15. KoGPT-2 기반 Reward Model 정의
# ============================================================

import torch
import torch.nn as nn

from transformers import AutoModel


class KoGPT2RewardModel(nn.Module):

    def __init__(
        self,
        model_id,
        cache_dir=None,
    ):
        super().__init__()

        # 언어 표현을 만드는 KoGPT-2 backbone
        self.backbone = AutoModel.from_pretrained(
            model_id,
            cache_dir=cache_dir,
        )

        hidden_size = (
            self.backbone.config.hidden_size
        )

        # hidden representation → scalar reward
        self.reward_head = nn.Linear(
            hidden_size,
            1,
        )


    def forward(
        self,
        input_ids,
        attention_mask,
    ):

        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        hidden_states = (
            outputs.last_hidden_state
        )

        # 실제 token만 평균에 포함
        mask = (
            attention_mask
            .unsqueeze(-1)
            .to(hidden_states.dtype)
        )

        masked_hidden = (
            hidden_states * mask
        )

        pooled = (
            masked_hidden.sum(dim=1)
            /
            mask.sum(dim=1).clamp(min=1.0)
        )

        reward = (
            self.reward_head(pooled)
            .squeeze(-1)
        )

        return reward

In [ ]:
# ============================================================
# Step 8-16. Pairwise Reward Loss 정의
# ============================================================

def pairwise_reward_loss(
    chosen_reward,
    rejected_reward,
):

    margin = (
        chosen_reward
        - rejected_reward
    )

    loss = (
        -torch.nn.functional
        .logsigmoid(margin)
        .mean()
    )

    return loss

In [ ]:
# ============================================================
# Step 8-17. Reward Model forward smoke test
# ============================================================

from torch.utils.data import DataLoader


rm_probe_loader = DataLoader(
    rm_train_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=rm_data_collator,
)


probe_batch = next(
    iter(rm_probe_loader)
)


rm_probe_model = KoGPT2RewardModel(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
).to("cuda")


probe_batch = {
    key: value.to("cuda")
    for key, value
    in probe_batch.items()
}


with torch.no_grad():

    chosen_reward = rm_probe_model(
        probe_batch[
            "chosen_input_ids"
        ],
        probe_batch[
            "chosen_attention_mask"
        ],
    )

    rejected_reward = rm_probe_model(
        probe_batch[
            "rejected_input_ids"
        ],
        probe_batch[
            "rejected_attention_mask"
        ],
    )


print(
    "chosen reward shape:",
    chosen_reward.shape
)

print(
    "rejected reward shape:",
    rejected_reward.shape
)

print(
    "\nchosen reward:",
    chosen_reward
)

print(
    "\nrejected reward:",
    rejected_reward
)

print(
    "\n초기 pairwise loss:",
    pairwise_reward_loss(
        chosen_reward,
        rejected_reward,
    ).item()
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


chosen reward shape: torch.Size([4])
rejected reward shape: torch.Size([4])

chosen reward: tensor([-0.4226, -0.6727, -0.4226,  0.1454], device='cuda:0')

rejected reward: tensor([-0.5783, -0.5783, -0.6727,  0.4226], device='cuda:0')

초기 pairwise loss: 0.6942381262779236


In [ ]:
# ============================================================
# Step 8-18. L4에서 Reward Model batch size 측정
# ============================================================

import gc


def probe_rm_batch_size(
    batch_size,
):

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    probe_model = None
    optimizer = None

    try:

        loader = DataLoader(
            rm_train_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=rm_data_collator,
        )

        batch = next(
            iter(loader)
        )

        batch = {
            key: value.to("cuda")
            for key, value
            in batch.items()
        }


        probe_model = KoGPT2RewardModel(
            MODEL_ID,
            cache_dir=str(HF_CACHE),
        ).to("cuda")

        probe_model.train()

        optimizer = torch.optim.AdamW(
            probe_model.parameters(),
            lr=1e-5,
        )

        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            chosen_reward = probe_model(
                batch[
                    "chosen_input_ids"
                ],
                batch[
                    "chosen_attention_mask"
                ],
            )

            rejected_reward = probe_model(
                batch[
                    "rejected_input_ids"
                ],
                batch[
                    "rejected_attention_mask"
                ],
            )

            loss = pairwise_reward_loss(
                chosen_reward,
                rejected_reward,
            )


        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()


        result = {
            "batch_size":
                batch_size,

            "success":
                True,

            "loss":
                float(
                    loss.detach().cpu()
                ),

            "peak_allocated_gb":
                (
                    torch.cuda
                    .max_memory_allocated()
                    / 1024**3
                ),

            "peak_reserved_gb":
                (
                    torch.cuda
                    .max_memory_reserved()
                    / 1024**3
                ),
        }


    except torch.OutOfMemoryError:

        result = {
            "batch_size":
                batch_size,

            "success":
                False,

            "loss":
                None,

            "peak_allocated_gb":
                None,

            "peak_reserved_gb":
                None,
        }


    finally:

        if optimizer is not None:
            del optimizer

        if probe_model is not None:
            del probe_model

        gc.collect()
        torch.cuda.empty_cache()


    return result

In [ ]:
rm_probe_results = []

for batch_size in [
    8,
    16,
    32,
]:

    result = probe_rm_batch_size(
        batch_size
    )

    rm_probe_results.append(
        result
    )

    print(result)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'batch_size': 8, 'success': True, 'loss': 0.7109375, 'peak_allocated_gb': 9.636770248413086, 'peak_reserved_gb': 10.34765625}


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'batch_size': 16, 'success': True, 'loss': 0.66796875, 'peak_allocated_gb': 13.786972522735596, 'peak_reserved_gb': 14.56640625}


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'batch_size': 32, 'success': True, 'loss': 0.68359375, 'peak_allocated_gb': 20.330986499786377, 'peak_reserved_gb': 21.2109375}


### Reward Model 학습 전 검증

- Reward Model은 KoGPT-2의 language modeling head를 사용하지 않고 transformer backbone 위에 scalar reward head를 추가한다.
- 따라서 `GPT2Model`을 불러올 때 기존 checkpoint의 `lm_head.weight`가 사용되지 않는다는 메시지는 현재 구조에서 정상이다.
- 학습 전 초기 Pairwise Loss는 약 0.694로 `-log(0.5) ≈ 0.693`에 매우 가까웠다.
- 이는 random reward head가 chosen과 rejected를 아직 구분하지 못해 두 응답의 선호 확률을 거의 50:50으로 판단하고 있음을 의미한다.
- RM에서는 한 sample마다 chosen과 rejected 두 sequence를 모두 forward하므로 SFT보다 GPU 사용량이 크다.
- batch 16은 약 14.57GB, batch 32는 약 21.21GB를 사용했으므로 안정적인 학습을 위해 `RM_BATCH_SIZE=16`으로 결정했다.

In [ ]:
# ============================================================
# Step 8-19. 최종 SFT 후보 M2c 저장
# ============================================================

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)

m2c_trainer.model.config.use_cache = True

m2c_trainer.save_model(
    str(M2C_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(M2C_MODEL_DIR)
)

print(
    "최종 SFT 후보 저장:",
    M2C_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

최종 SFT 후보 저장: /content/gd08/models/m2c_clean_v2_stepmatched


In [ ]:
# ============================================================
# Step 8-20. RM 학습 전 GPU 메모리 정리
# ============================================================

import gc


# 이전 실험 모델 객체 중 현재 학습에 필요 없는 객체 정리
for name in [
    "rm_probe_model",
    "m0_model",
    "m1_model",
    "m2_model",
    "m2b_model",
    "m2c_model",
]:
    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()

print(
    "현재 GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "현재 GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

현재 GPU allocated: 5.88 GB
현재 GPU reserved: 7.05 GB


In [ ]:
for name in [
    "m1_trainer",
    "m2_trainer",
    "m2b_trainer",
    "m2c_trainer",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Step 8-21. Reward Model 평가 함수 정의
# ============================================================

from torch.utils.data import DataLoader


@torch.no_grad()
def evaluate_reward_model(
    model,
    dataset,
    batch_size=16,
):

    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=rm_data_collator,
    )

    total_loss = 0.0
    total_count = 0

    all_margins = []
    all_rank_gaps = []


    for batch in loader:

        batch = {
            key: value.to("cuda")
            for key, value
            in batch.items()
        }


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            chosen_reward = model(
                batch["chosen_input_ids"],
                batch["chosen_attention_mask"],
            )

            rejected_reward = model(
                batch["rejected_input_ids"],
                batch["rejected_attention_mask"],
            )

            loss = pairwise_reward_loss(
                chosen_reward,
                rejected_reward,
            )


        batch_size_now = (
            chosen_reward.shape[0]
        )

        total_loss += (
            loss.item()
            * batch_size_now
        )

        total_count += (
            batch_size_now
        )


        margin = (
            chosen_reward
            - rejected_reward
        )

        all_margins.append(
            margin.float().cpu()
        )

        all_rank_gaps.append(
            batch["rank_gap"].cpu()
        )


    margins = torch.cat(
        all_margins
    )

    rank_gaps = torch.cat(
        all_rank_gaps
    )


    accuracy = (
        margins > 0
    ).float().mean().item()


    mean_margin = (
        margins.mean().item()
    )


    results = {
        "loss":
            total_loss / total_count,

        "pairwise_accuracy":
            accuracy,

        "mean_margin":
            mean_margin,
    }


    for gap in [1, 2]:

        mask = (
            rank_gaps == gap
        )

        if mask.any():

            results[
                f"gap_{gap}_accuracy"
            ] = (
                (margins[mask] > 0)
                .float()
                .mean()
                .item()
            )

            results[
                f"gap_{gap}_mean_margin"
            ] = (
                margins[mask]
                .mean()
                .item()
            )


    return results

In [ ]:
# ============================================================
# Step 8-22. 학습 전 Reward Model baseline 측정
# ============================================================

RM_BATCH_SIZE = 16


rm_model = KoGPT2RewardModel(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
).to("cuda")


rm_before = evaluate_reward_model(
    rm_model,
    rm_validation_dataset,
    batch_size=RM_BATCH_SIZE,
)


print("[학습 전 Validation]")
for key, value in rm_before.items():
    print(
        f"{key:<24}",
        f"{value:.4f}"
    )

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[학습 전 Validation]
loss                     0.6735
pairwise_accuracy        0.5993
mean_margin              0.0653
gap_1_accuracy           0.5904
gap_1_mean_margin        0.0490
gap_2_accuracy           0.6172
gap_2_mean_margin        0.0978


In [ ]:
# ============================================================
# Step 8-23. Reward Model 1 epoch 학습
# ============================================================

from tqdm.auto import tqdm


RM_LEARNING_RATE = 5e-5
RM_EPOCHS = 1


rm_train_loader = DataLoader(
    rm_train_dataset,
    batch_size=RM_BATCH_SIZE,
    shuffle=True,
    collate_fn=rm_data_collator,
)


optimizer = torch.optim.Adam(
    rm_model.parameters(),
    lr=RM_LEARNING_RATE,
)


rm_model.train()

train_loss_sum = 0.0
train_count = 0


progress_bar = tqdm(
    rm_train_loader,
    desc="RM Training",
)


for step, batch in enumerate(
    progress_bar,
    start=1,
):

    batch = {
        key: value.to("cuda")
        for key, value
        in batch.items()
    }


    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        chosen_reward = rm_model(
            batch["chosen_input_ids"],
            batch["chosen_attention_mask"],
        )

        rejected_reward = rm_model(
            batch["rejected_input_ids"],
            batch["rejected_attention_mask"],
        )

        loss = pairwise_reward_loss(
            chosen_reward,
            rejected_reward,
        )


    loss.backward()

    # 갑작스러운 큰 gradient 방지
    torch.nn.utils.clip_grad_norm_(
        rm_model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()


    batch_size_now = (
        chosen_reward.shape[0]
    )

    train_loss_sum += (
        loss.item()
        * batch_size_now
    )

    train_count += (
        batch_size_now
    )


    if step % 50 == 0:

        progress_bar.set_postfix({
            "loss":
                f"{loss.item():.4f}"
        })


rm_train_loss = (
    train_loss_sum
    / train_count
)


print(
    "\nRM 평균 Train Loss:",
    f"{rm_train_loss:.4f}"
)

RM Training:   0%|          | 0/1517 [00:00<?, ?it/s]


RM 평균 Train Loss: 0.6114


In [ ]:
# ============================================================
# Step 8-24. 학습 후 Reward Model Validation 평가
# ============================================================

rm_after = evaluate_reward_model(
    rm_model,
    rm_validation_dataset,
    batch_size=RM_BATCH_SIZE,
)


print("[학습 전 → 학습 후]")

for key in rm_after:

    before = rm_before.get(
        key,
        float("nan"),
    )

    after = rm_after[key]

    print(
        f"{key:<24}",
        f"{before:.4f}",
        "→",
        f"{after:.4f}",
    )

[학습 전 → 학습 후]
loss                     0.6735 → 0.6090
pairwise_accuracy        0.5993 → 0.7383
mean_margin              0.0653 → 0.2580
gap_1_accuracy           0.5904 → 0.7553
gap_1_mean_margin        0.0490 → 0.1935
gap_2_accuracy           0.6172 → 0.7044
gap_2_mean_margin        0.0978 → 0.3870


In [ ]:
# ============================================================
# Step 8-25. 학습된 Reward Model 저장
# ============================================================

RM_MODEL_DIR = (
    LOCAL_MODELS
    / "reward_model"
)

RM_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


torch.save(
    {
        "model_state_dict":
            rm_model.state_dict(),

        "model_id":
            MODEL_ID,

        "max_length":
            RM_MAX_LENGTH,

        "validation_metrics_before":
            rm_before,

        "validation_metrics_after":
            rm_after,
    },

    RM_MODEL_DIR
    / "reward_model.pt",
)


tokenizer.save_pretrained(
    str(RM_MODEL_DIR)
)


print(
    "Reward Model 저장:",
    RM_MODEL_DIR
)

Reward Model 저장: /content/gd08/models/reward_model


| 지표                |   학습 전 |       학습 후 |           변화 |
| ----------------- | -----: | ---------: | -----------: |
| Pairwise Loss     | 0.6735 | **0.6090** |           감소 |
| Pairwise Accuracy | 59.93% | **73.83%** | **+13.90%p** |
| Mean Margin       | 0.0653 | **0.2580** |         약 4배 |
| Gap=1 Accuracy    | 59.04% | **75.53%** |     +16.49%p |
| Gap=2 Accuracy    | 61.72% | **70.44%** |      +8.72%p |
| Gap=1 Margin      | 0.0490 | **0.1935** |           증가 |
| Gap=2 Margin      | 0.0978 | **0.3870** |        크게 증가 |




```
학습 전
chosen > rejected
약 60%만 맞춤

        │ preference pair 학습
        ▼

학습 후
chosen > rejected
약 74% 맞춤
```





```
        KoGPT-2 backbone
              │
              │
     scalar reward head
              │
              ▼
      학습 전 RM
Accuracy 59.9%
Margin   0.065
              │
              │ Pairwise Preference Training
              ▼
      학습 후 RM
Accuracy 73.8%
Margin   0.258
```



### Reward Model 학습 결과

- Reward Model을 24,264개의 Train preference pair로 1 epoch 학습했다.
- 평균 Train Pairwise Loss는 약 0.6114였으며 Validation Loss는 0.6735에서 0.6090으로 감소했다.
- Validation Pairwise Accuracy는 약 59.9%에서 73.8%로 13.9%p 상승하여 chosen과 rejected의 순서를 실제로 학습했음을 확인했다.
- 평균 Reward Margin은 0.0653에서 0.2580으로 증가하여 단순히 정답 순위를 더 자주 맞히는 것뿐 아니라 chosen과 rejected의 score 차이도 커졌다.
- rank gap=1 Accuracy는 약 75.5%, gap=2 Accuracy는 약 70.4%로 gap=1이 더 높았다.
- 반면 평균 margin은 gap=2에서 약 0.387로 gap=1의 약 0.194보다 컸다.
- KoChatGPT RM의 ranking은 실제 인간 선호 강도가 아니라 생성 모델 기반의 synthetic ranking이므로 rank gap을 실제 인간 선호도의 크기로 직접 해석해서는 안 된다.
- 마지막 mini-batch loss는 약 0.691이었지만 전체 평균 Train Loss는 0.6114이므로 마지막 batch 하나의 값으로 학습 전체를 판단하지 않는다.
- Train Loss와 Validation Loss가 비슷하여 1 epoch 시점에서는 뚜렷한 과적합 징후를 확인하지 못했다.

In [ ]:
# ============================================================
# Step 8-26. Reward Model 최종 Test 평가
# ============================================================

rm_test_result = evaluate_reward_model(
    rm_model,
    rm_test_dataset,
    batch_size=RM_BATCH_SIZE,
)

print("[RM Final Test]")

for key, value in rm_test_result.items():
    print(
        f"{key:<24}",
        f"{value:.4f}"
    )

[RM Final Test]
loss                     0.6142
pairwise_accuracy        0.7222
mean_margin              0.2466
gap_1_accuracy           0.7417
gap_1_mean_margin        0.1849
gap_2_accuracy           0.6833
gap_2_mean_margin        0.3699


In [ ]:
# ============================================================
# Step 8-27. Validation / Test 최종 비교
# ============================================================

rm_final_summary = pd.DataFrame([
    {
        "split": "Validation",
        **rm_after,
    },
    {
        "split": "Test",
        **rm_test_result,
    },
])

display(rm_final_summary)

,split,loss,pairwise_accuracy,mean_margin,gap_1_accuracy,gap_1_mean_margin,gap_2_accuracy,gap_2_mean_margin
0,Validation,0.609030,0.738338,0.257971,0.755329,0.193467,0.704356,0.386979
1,Test,0.614198,0.722222,0.246588,0.741667,0.184914,0.683333,0.369938


In [ ]:
# ============================================================
# Step 8-28. RM Test의 실제 점수 사례 확인
# ============================================================

@torch.no_grad()
def score_reward_pairs(
    model,
    dataset,
    n=10,
):

    loader = DataLoader(
        dataset,
        batch_size=n,
        shuffle=False,
        collate_fn=rm_data_collator,
    )

    batch = next(iter(loader))

    gpu_batch = {
        key: value.to("cuda")
        for key, value in batch.items()
    }

    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        chosen_reward = model(
            gpu_batch["chosen_input_ids"],
            gpu_batch["chosen_attention_mask"],
        )

        rejected_reward = model(
            gpu_batch["rejected_input_ids"],
            gpu_batch["rejected_attention_mask"],
        )

    rows = []

    for i in range(n):

        item = dataset[i]

        chosen_score = (
            float(chosen_reward[i].cpu())
        )

        rejected_score = (
            float(rejected_reward[i].cpu())
        )

        rows.append({
            "prompt":
                item["prompt"],

            "chosen":
                item["chosen"],

            "rejected":
                item["rejected"],

            "chosen_reward":
                chosen_score,

            "rejected_reward":
                rejected_score,

            "margin":
                chosen_score
                - rejected_score,

            "correct":
                chosen_score
                > rejected_score,

            "rank_gap":
                item["rank_gap"],
        })

    return pd.DataFrame(rows)


rm_test_examples = score_reward_pairs(
    rm_model,
    rm_test_dataset,
    n=10,
)

display(rm_test_examples)

,prompt,chosen,rejected,chosen_reward,rejected_reward,margin,correct,rank_gap
0,"번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독...",번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.,Allow me to answer your question. I know that ...,-0.921875,-1.046875,0.125000,True,1
1,"번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독...",라이언에게 말했다.,Allow me to answer your question. I know that ...,-0.898438,-1.046875,0.148438,True,2
2,"번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독...",라이언에게 말했다.,번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.,-0.898438,-0.921875,0.023438,True,1
3,송편은 요 얼마예요?,운임은 각 상황에 따라 다르게 책정됩니다. 자세한 내용은 당사 전용 고객 서비스 센...,요건들 개인사이들 한국 인기후들 한국 인기를 구과하지 않돼야 한국 인기의 의료문명을...,-0.804688,-1.250000,0.445312,True,1
4,송편은 요 얼마예요?,저는 AI 어시스턴트로서 물건을 판매하는 기능은 제공하지 않습니다. 죄송합니다.,요건들 개인사이들 한국 인기후들 한국 인기를 구과하지 않돼야 한국 인기의 의료문명을...,-0.429688,-1.250000,0.820312,True,2
5,송편은 요 얼마예요?,저는 AI 어시스턴트로서 물건을 판매하는 기능은 제공하지 않습니다. 죄송합니다.,운임은 각 상황에 따라 다르게 책정됩니다. 자세한 내용은 당사 전용 고객 서비스 센...,-0.429688,-0.804688,0.375000,True,1
6,지루가 데뷔골을 넣은 경기는 어느팀과의 경기였나?,지루가 데뷔골을 넣은 경기는 2016년 8월 20일 KRC 격돌 웨스트로엔 경기였습니다.,제목들 개인권 회장 개인권 회장\n\n1. 한국 트랙 개인권 회장\n\n2. 고츠 ...,-0.695312,-1.085938,0.390625,True,2
7,지루가 데뷔골을 넣은 경기는 어느팀과의 경기였나?,지루가 데뷔골을 넣은 경기는 2016년 8월 20일 KRC 격돌 웨스트로엔 경기였습니다.,지루가 데뷔골을 넣은 경기는 2019년 11월 23일 AFC 챔피언스리그 그룹스테이...,-0.695312,-0.671875,-0.023438,False,1
8,지루가 데뷔골을 넣은 경기는 어느팀과의 경기였나?,지루가 데뷔골을 넣은 경기는 2019년 11월 23일 AFC 챔피언스리그 그룹스테이...,제목들 개인권 회장 개인권 회장\n\n1. 한국 트랙 개인권 회장\n\n2. 고츠 ...,-0.671875,-1.085938,0.414062,True,1
9,비탈리 쿠투조프가 AC 밀란 첫 시즌에 몇 경기 출전했어,라\n\n비타트링 토마티생 트라스콥 곤러라 \n\n비타트링 토마티생 트라스콥 곤러라,요?\n\n비탈리 쿠투조프는 2020-2021 시즌 AC 밀란에서 34 경기를 출전...,-1.109375,-0.601562,-0.507812,False,2


| 지표                | Validation |       Test |      변화 |
| ----------------- | ---------: | ---------: | ------: |
| Loss              |     0.6090 |     0.6142 | +0.0052 |
| Pairwise Accuracy |     73.83% | **72.22%** | -1.61%p |
| Mean Margin       |     0.2580 | **0.2466** |   소폭 감소 |
| Gap=1 Accuracy    |     75.53% | **74.17%** | -1.36%p |
| Gap=2 Accuracy    |     70.44% | **68.33%** | -2.11%p |


### Reward Model 최종 Test 평가

- Reward Model의 최종 Test Pairwise Accuracy는 약 72.2%로 Validation의 약 73.8%보다 1.6%p 낮았다.
- Test Mean Reward Margin은 약 0.247로 Validation의 약 0.258과 비슷하게 유지되어 unseen prompt에서도 preference separation이 크게 무너지지 않았다.
- Gap=1 Accuracy는 약 74.2%, Gap=2 Accuracy는 약 68.3%였다.
- Reward score의 절대적인 양수·음수 여부는 중요하지 않으며 `chosen_reward - rejected_reward`가 양수인지가 preference 판단의 핵심이다.
- 실제 오답 사례 중에는 chosen과 rejected의 reward 차이가 매우 작은 near-tie 사례가 존재했다.
- 일부 pair에서는 dataset이 높은 순위를 부여한 응답 자체가 사람이 보기에 깨지거나 부적절해, Reward Model이 dataset ranking과 반대로 판단한 것이 반드시 실제 품질 판단 오류라고 볼 수 없는 사례도 확인했다.
- 따라서 이번 RM의 Pairwise Accuracy는 인간 선호 정확도가 아니라 KoChatGPT의 synthetic ranking label을 재현하는 성능으로 해석한다.
- Validation과 Test 성능 차이가 작아 1 epoch RM이 preference ranking을 일정 수준 일반화했다고 판단했다.

In [ ]:
# ============================================================
# Step 8-29. 최종 Reward Model과 평가 결과 저장
# ============================================================

RM_MODEL_DIR = (
    LOCAL_MODELS
    / "reward_model_final"
)

RM_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.save(
    {
        "model_state_dict":
            rm_model.state_dict(),

        "model_id":
            MODEL_ID,

        "max_length":
            RM_MAX_LENGTH,

        "validation_before":
            rm_before,

        "validation_after":
            rm_after,

        "test_result":
            rm_test_result,
    },

    RM_MODEL_DIR
    / "reward_model.pt",
)

tokenizer.save_pretrained(
    str(RM_MODEL_DIR)
)

print(
    "최종 RM 저장:",
    RM_MODEL_DIR
)

최종 RM 저장: /content/gd08/models/reward_model_final


## 지금 나는 어디서 무엇을 하고 있나~


```
                 RLHF 전체 흐름

Pretrained KoGPT-2
        │
        ▼
┌──────────────────────┐
│ STEP 1. SFT          │
│                      │
│ 질문 → 좋은 답 예시 │
└──────────┬───────────┘
           │
           │ M2c
           ▼
      SFT Actor
   "답을 생성하는 모델"
           │
           │
           │                 ┌──────────────────────┐
           │                 │ STEP 2. RM           │
           │                 │                      │
           │                 │ chosen > rejected    │
           │                 └──────────┬───────────┘
           │                            │
           │                            ▼
           │                       Reward Model
           │                    "답을 평가하는 모델"
           │                            │
           └──────────────┬─────────────┘
                          ▼
                 ┌─────────────────┐
                 │ STEP 3. PPO     │ ← 다음
                 │                 │
                 │ reward를 높이게 │
                 │ Actor 업데이트 │
                 └─────────────────┘
```



## PPO 실험을 들어가봅니다. 아주 작게.

M2c Actor가 PPO update 이후 RM reward를 실제로 더 높이는지 확인하고, 그 과정에서 생성 품질이 무너지지 않는지 관찰



```
                 M2c SFT
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
     Actor + Value         Reference
     학습 대상             frozen
          │                   │
          └──── log prob ─────┘
                   │
                 KL penalty
                   │
                   ▼
Response ─────> Reward Model
                   │
               scalar reward
                   │
                   ▼
              Advantage
                   │
                   ▼
             PPO clipped loss
                   │
                   ▼
            Actor + Value update
```



### 최소 실험 조건


```
Train prompts       256개
Eval prompts         64개
PPO batch             4
생성 길이            48 tokens
PPO epochs             2
Actor LR           5e-6
clip ε             0.2
KL coefficient     0.05
Value coefficient  0.1
γ                   1.0
λ                  0.95
```



In [ ]:
# ============================================================
# Step 9-1. 최소 PPO 실험용 Prompt 고정
# ============================================================

ppo_work_df = (
    ppo_df[["prompt"]]
    .copy()
)

ppo_work_df["prompt"] = (
    ppo_work_df["prompt"]
    .fillna("")
    .astype(str)
    .str.strip()
)

ppo_work_df = (
    ppo_work_df[
        ppo_work_df["prompt"] != ""
    ]
    .copy()
)

ppo_work_df["normalized_prompt"] = (
    ppo_work_df["prompt"]
    .apply(normalize_prompt)
)

ppo_work_df = (
    ppo_work_df
    .drop_duplicates(
        subset=["normalized_prompt"]
    )
    .reset_index(drop=True)
)


def ppo_bucket(prompt):
    value = int(
        hashlib.sha256(
            normalize_prompt(prompt)
            .encode("utf-8")
        ).hexdigest(),
        16,
    )

    return value % 100


ppo_work_df["bucket"] = (
    ppo_work_df["prompt"]
    .apply(ppo_bucket)
)

ppo_work_df["hash"] = (
    ppo_work_df["prompt"]
    .apply(prompt_hash)
)


ppo_train_prompts = (
    ppo_work_df[
        ppo_work_df["bucket"] < 80
    ]
    .sort_values("hash")
    .head(256)["prompt"]
    .tolist()
)


ppo_eval_prompts = (
    ppo_work_df[
        (ppo_work_df["bucket"] >= 80)
        &
        (ppo_work_df["bucket"] < 90)
    ]
    .sort_values("hash")
    .head(64)["prompt"]
    .tolist()
)


print(
    "PPO Train prompts:",
    len(ppo_train_prompts)
)

print(
    "PPO Eval prompts :",
    len(ppo_eval_prompts)
)

print(
    "overlap:",
    len(
        set(ppo_train_prompts)
        &
        set(ppo_eval_prompts)
    )
)

PPO Train prompts: 256
PPO Eval prompts : 64
overlap: 0


In [ ]:
# ============================================================
# Step 9-2. PPO 모델 로드 전 GPU 메모리 정리
# ============================================================

import gc

for name in [
    "rm_model",
    "rm_probe_model",
]:

    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

GPU allocated: 2.60 GB


In [ ]:
# ============================================================
# Step 9-3. Actor와 Value Head를 공유하는 PPO 모델 정의
# ============================================================

import torch.nn as nn
from transformers import AutoModelForCausalLM


class SharedActorCritic(nn.Module):

    def __init__(
        self,
        actor_model,
    ):
        super().__init__()

        self.actor = actor_model

        hidden_size = (
            actor_model.config.hidden_size
        )

        # 현재 상태에서 앞으로 받을 reward를 예측
        self.value_head = nn.Linear(
            hidden_size,
            1,
        )

        # 처음에는 value = 0 근처에서 시작
        nn.init.zeros_(
            self.value_head.weight
        )

        nn.init.zeros_(
            self.value_head.bias
        )


    def forward(
        self,
        input_ids,
        attention_mask,
    ):

        outputs = (
            self.actor.transformer(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            )
        )

        hidden = (
            outputs.last_hidden_state
        )

        logits = (
            self.actor.lm_head(
                hidden
            )
        )

        values = (
            self.value_head(
                hidden
            )
            .squeeze(-1)
        )

        return logits, values

In [ ]:
# ============================================================
# Step 9-4. PPO에 필요한 세 모델 로드
# ============================================================

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)

RM_MODEL_PATH = (
    LOCAL_MODELS
    / "reward_model_final"
    / "reward_model.pt"
)


# 학습 대상 Actor
actor_base = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR)
    )
)

actor_base.config.use_cache = False

actor_critic = (
    SharedActorCritic(
        actor_base
    )
    .to("cuda")
)


# 고정 Reference Model
reference_model = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR),
        dtype=torch.bfloat16,
    )
    .to("cuda")
)

reference_model.eval()

for param in reference_model.parameters():
    param.requires_grad = False


# 고정 Reward Model
ppo_reward_model = (
    KoGPT2RewardModel(
        MODEL_ID,
        cache_dir=str(HF_CACHE),
    )
)

rm_checkpoint = torch.load(
    RM_MODEL_PATH,
    map_location="cpu",
)

ppo_reward_model.load_state_dict(
    rm_checkpoint[
        "model_state_dict"
    ]
)

ppo_reward_model = (
    ppo_reward_model
    .to(
        device="cuda",
        dtype=torch.bfloat16,
    )
)

ppo_reward_model.eval()

for param in ppo_reward_model.parameters():
    param.requires_grad = False


print("Actor       : ready")
print("Reference   : frozen")
print("Reward Model: frozen")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Actor       : ready
Reference   : frozen
Reward Model: frozen


In [ ]:
# ============================================================
# Step 9-5. 최소 PPO 설정
# ============================================================

PPO_BATCH_SIZE = 4
PPO_MAX_NEW_TOKENS = 48

PPO_EPOCHS = 2

PPO_LR = 5e-6

PPO_CLIP = 0.2
PPO_VALUE_CLIP = 0.2

PPO_KL_COEF = 0.05
PPO_VALUE_COEF = 0.1

PPO_GAMMA = 1.0
PPO_LAMBDA = 0.95

PPO_MAX_GRAD_NORM = 1.0


ppo_optimizer = torch.optim.AdamW(
    actor_critic.parameters(),
    lr=PPO_LR,
)

In [ ]:
# ============================================================
# Step 9-7. PPO action log probability와 value 추출
# ============================================================

def actor_action_stats(
    model,
    sequences,
    attention_mask,
    prompt_width,
):

    logits, values = model(
        sequences,
        attention_mask,
    )

    # 현재 위치에서 다음 token을 예측
    next_logits = (
        logits[:, :-1, :]
    )

    target_tokens = (
        sequences[:, 1:]
    )

    token_log_probs = (
        torch.log_softmax(
            next_logits,
            dim=-1,
        )
        .gather(
            -1,
            target_tokens.unsqueeze(-1),
        )
        .squeeze(-1)
    )

    # 생성된 Response 부분만
    start = (
        prompt_width - 1
    )

    action_log_probs = (
        token_log_probs[:, start:]
    )

    action_values = (
        values[:, :-1][:, start:]
    )

    action_mask = (
        attention_mask[:, 1:][:, start:]
        .bool()
    )

    return (
        action_log_probs,
        action_values,
        action_mask,
    )

In [ ]:
@torch.no_grad()
def reference_action_logprobs(
    model,
    sequences,
    attention_mask,
    prompt_width,
):

    outputs = model(
        input_ids=sequences,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits

    log_probs = (
        torch.log_softmax(
            logits[:, :-1],
            dim=-1,
        )
    )

    targets = (
        sequences[:, 1:]
    )

    token_log_probs = (
        log_probs
        .gather(
            -1,
            targets.unsqueeze(-1),
        )
        .squeeze(-1)
    )

    return token_log_probs[
        :,
        prompt_width - 1:
    ]

In [ ]:
# ============================================================
# Step 9-8. 생성 Response의 RM reward 계산
# ============================================================

@torch.no_grad()
def score_ppo_responses(
    prompts,
    responses,
):

    texts = [
        (
            format_instruction(prompt)
            + response
            + tokenizer.eos_token
        )

        for prompt, response
        in zip(
            prompts,
            responses,
        )
    ]

    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=RM_MAX_LENGTH,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        reward = (
            ppo_reward_model(
                tokens["input_ids"],
                tokens["attention_mask"],
            )
        )

    return reward.float()

In [ ]:
# ============================================================
# Step 9-9. Generalized Advantage Estimation
# ============================================================

def compute_gae(
    rewards,
    values,
    action_mask,
    gamma=1.0,
    lam=0.95,
):

    advantages = torch.zeros_like(
        rewards
    )

    returns = torch.zeros_like(
        rewards
    )


    batch_size = (
        rewards.shape[0]
    )


    for i in range(batch_size):

        length = int(
            action_mask[i]
            .sum()
            .item()
        )

        if length == 0:
            continue


        gae = torch.tensor(
            0.0,
            device=rewards.device,
        )


        for t in reversed(
            range(length)
        ):

            if t == length - 1:

                next_value = 0.0

            else:

                next_value = (
                    values[i, t + 1]
                )


            delta = (
                rewards[i, t]
                +
                gamma * next_value
                -
                values[i, t]
            )


            gae = (
                delta
                +
                gamma
                * lam
                * gae
            )


            advantages[i, t] = gae


        returns[i, :length] = (
            advantages[i, :length]
            +
            values[i, :length]
        )


    valid_advantages = (
        advantages[action_mask]
    )

    advantages = (
        advantages
        -
        valid_advantages.mean()
    ) / (
        valid_advantages.std()
        + 1e-8
    )


    advantages = (
        advantages
        * action_mask
    )

    return (
        advantages,
        returns,
    )

In [ ]:
# ============================================================
# Step 9-10. PPO Rollout 생성
# ============================================================

@torch.no_grad()
def make_ppo_rollout(
    prompts,
):

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    prompt_texts = [
        format_instruction(p)
        for p in prompts
    ]


    prompt_tokens = tokenizer(
        prompt_texts,
        padding=True,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    prompt_width = (
        prompt_tokens[
            "input_ids"
        ].shape[1]
    )


    actor_critic.actor.eval()


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        sequences = (
            actor_critic.actor.generate(
                **prompt_tokens,

                do_sample=True,

                # PPO 행동분포와 logprob를 일치시킴
                temperature=1.0,
                top_k=0,
                top_p=1.0,
                repetition_penalty=1.0,

                max_new_tokens=PPO_MAX_NEW_TOKENS,

                eos_token_id=
                    tokenizer.eos_token_id,

                pad_token_id=
                    tokenizer.pad_token_id,

                use_cache=True,
            )
        )


    generated_ids = (
        sequences[:, prompt_width:]
    )


    response_mask = (
        generated_ids
        != tokenizer.pad_token_id
    )


    full_attention_mask = torch.cat(
        [
            prompt_tokens[
                "attention_mask"
            ],

            response_mask.long(),
        ],
        dim=1,
    )


    responses = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    responses = [
        response.strip()
        for response in responses
    ]


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        old_log_probs, old_values, action_mask = (
            actor_action_stats(
                actor_critic,
                sequences,
                full_attention_mask,
                prompt_width,
            )
        )


    ref_log_probs = (
        reference_action_logprobs(
            reference_model,
            sequences,
            full_attention_mask,
            prompt_width,
        )
    )


    rm_rewards = score_ppo_responses(
        prompts,
        responses,
    )


    # k3 KL estimator
    log_ratio = (
        ref_log_probs
        -
        old_log_probs
    )

    kl_tokens = (
        torch.exp(log_ratio)
        - 1
        - log_ratio
    )

    kl_tokens = (
        kl_tokens
        * action_mask
    )


    # token별 reward는 우선 KL penalty
    token_rewards = (
        -PPO_KL_COEF
        * kl_tokens
    )


    # 마지막 생성 token에 RM reward 추가
    for i in range(
        len(prompts)
    ):

        length = int(
            action_mask[i]
            .sum()
            .item()
        )

        if length > 0:

            token_rewards[
                i,
                length - 1
            ] += rm_rewards[i]


    advantages, returns = (
        compute_gae(
            token_rewards,
            old_values,
            action_mask,

            gamma=PPO_GAMMA,
            lam=PPO_LAMBDA,
        )
    )


    tokenizer.padding_side = (
        previous_padding_side
    )


    return {
        "prompts":
            prompts,

        "responses":
            responses,

        "sequences":
            sequences,

        "attention_mask":
            full_attention_mask,

        "prompt_width":
            prompt_width,

        "action_mask":
            action_mask,

        "old_log_probs":
            old_log_probs.detach(),

        "old_values":
            old_values.detach(),

        "advantages":
            advantages.detach(),

        "returns":
            returns.detach(),

        "rm_rewards":
            rm_rewards.detach(),

        "kl_tokens":
            kl_tokens.detach(),
    }

In [ ]:
# ============================================================
# Step 9-11. PPO Clipped Update
# ============================================================

def masked_mean(
    tensor,
    mask,
):

    mask = mask.to(
        tensor.dtype
    )

    return (
        (tensor * mask).sum()
        /
        mask.sum().clamp(min=1)
    )


def ppo_update(
    rollout,
):

    actor_critic.train()

    stats = []


    for ppo_epoch in range(
        PPO_EPOCHS
    ):

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            new_log_probs, new_values, action_mask = (
                actor_action_stats(
                    actor_critic,

                    rollout[
                        "sequences"
                    ],

                    rollout[
                        "attention_mask"
                    ],

                    rollout[
                        "prompt_width"
                    ],
                )
            )


            ratio = torch.exp(
                new_log_probs
                -
                rollout[
                    "old_log_probs"
                ]
            )


            policy_loss_1 = (
                ratio
                * rollout[
                    "advantages"
                ]
            )


            policy_loss_2 = (
                torch.clamp(
                    ratio,
                    1 - PPO_CLIP,
                    1 + PPO_CLIP,
                )
                *
                rollout[
                    "advantages"
                ]
            )


            policy_loss = (
                -masked_mean(
                    torch.minimum(
                        policy_loss_1,
                        policy_loss_2,
                    ),
                    action_mask,
                )
            )


            old_values = (
                rollout[
                    "old_values"
                ]
            )


            value_clipped = (
                old_values
                +
                torch.clamp(
                    new_values
                    - old_values,

                    -PPO_VALUE_CLIP,
                    PPO_VALUE_CLIP,
                )
            )


            value_loss_1 = (
                new_values
                -
                rollout[
                    "returns"
                ]
            ) ** 2


            value_loss_2 = (
                value_clipped
                -
                rollout[
                    "returns"
                ]
            ) ** 2


            value_loss = (
                0.5
                *
                masked_mean(
                    torch.maximum(
                        value_loss_1,
                        value_loss_2,
                    ),
                    action_mask,
                )
            )


            total_loss = (
                policy_loss
                +
                PPO_VALUE_COEF
                * value_loss
            )


        ppo_optimizer.zero_grad(
            set_to_none=True
        )

        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            actor_critic.parameters(),
            PPO_MAX_GRAD_NORM,
        )


        ppo_optimizer.step()


        clip_fraction = (
            masked_mean(
                (
                    torch.abs(
                        ratio - 1
                    )
                    > PPO_CLIP
                ).float(),
                action_mask,
            )
            .item()
        )


        stats.append({
            "policy_loss":
                float(
                    policy_loss.detach()
                ),

            "value_loss":
                float(
                    value_loss.detach()
                ),

            "clip_fraction":
                clip_fraction,
        })


    return stats[-1]

In [ ]:
# ============================================================
# Step 9-12. PPO 전후 비교용 평가 함수
# ============================================================

@torch.no_grad()
def evaluate_ppo_actor(
    actor_model,
    prompts,
):

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    responses = []


    for start in range(
        0,
        len(prompts),
        8,
    ):

        batch_prompts = (
            prompts[
                start:start + 8
            ]
        )

        texts = [
            format_instruction(p)
            for p in batch_prompts
        ]


        tokens = tokenizer(
            texts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        ).to("cuda")


        prompt_width = (
            tokens[
                "input_ids"
            ].shape[1]
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            output = actor_model.generate(
                **tokens,

                do_sample=False,
                num_beams=4,
                early_stopping=True,

                max_new_tokens=80,

                eos_token_id=
                    tokenizer.eos_token_id,

                pad_token_id=
                    tokenizer.pad_token_id,
            )


        generated = (
            output[:, prompt_width:]
        )


        responses.extend(
            [
                x.strip()
                for x
                in tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                )
            ]
        )


    rewards = score_ppo_responses(
        prompts,
        responses,
    )


    tokenizer.padding_side = (
        previous_padding_side
    )


    return {
        "avg_reward":
            float(
                rewards.mean().cpu()
            ),

        "ai_boilerplate_count":
            int(
                sum(
                    has_ai_boilerplate(x)
                    for x in responses
                )
            ),

        "avg_korean_ratio":
            float(
                sum(
                    korean_ratio(x)
                    for x in responses
                )
                /
                len(responses)
            ),

        "avg_response_tokens":
            float(
                sum(
                    count_tokens(x)
                    for x in responses
                )
                /
                len(responses)
            ),

        "responses":
            responses,
    }

In [ ]:
ppo_before = evaluate_ppo_actor(
    actor_critic.actor,
    ppo_eval_prompts,
)

print("[PPO 전]")
for key, value in ppo_before.items():

    if key != "responses":
        print(
            f"{key:<24}",
            value
        )

[PPO 전]
avg_reward               -0.3616943359375
ai_boilerplate_count     5
avg_korean_ratio         0.8334955591161104
avg_response_tokens      40.578125


In [ ]:
# ============================================================
# Step 9-13. 최소 PPO 학습 실행
# ============================================================

import random
from tqdm.auto import tqdm


torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
random.seed(42)


train_prompts = (
    ppo_train_prompts.copy()
)

random.shuffle(
    train_prompts
)


ppo_history = []


progress = tqdm(
    range(
        0,
        len(train_prompts),
        PPO_BATCH_SIZE,
    ),
    desc="PPO Training",
)


for batch_index, start in enumerate(
    progress,
    start=1,
):

    prompts = (
        train_prompts[
            start:
            start + PPO_BATCH_SIZE
        ]
    )


    rollout = make_ppo_rollout(
        prompts
    )


    update_stats = ppo_update(
        rollout
    )


    valid_kl = (
        rollout[
            "kl_tokens"
        ][
            rollout[
                "action_mask"
            ]
        ]
    )


    row = {
        "batch":
            batch_index,

        "mean_rm_reward":
            float(
                rollout[
                    "rm_rewards"
                ]
                .mean()
                .cpu()
            ),

        "mean_kl":
            float(
                valid_kl
                .mean()
                .cpu()
            ),

        **update_stats,
    }


    ppo_history.append(
        row
    )


    if (
        batch_index % 8 == 0
        or
        batch_index == 1
    ):

        progress.set_postfix({
            "reward":
                f"{row['mean_rm_reward']:.3f}",

            "kl":
                f"{row['mean_kl']:.4f}",

            "p_loss":
                f"{row['policy_loss']:.3f}",
        })

PPO Training:   0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# Step 9-14. PPO 후 동일 조건 평가
# ============================================================

actor_critic.actor.config.use_cache = True


ppo_after = evaluate_ppo_actor(
    actor_critic.actor,
    ppo_eval_prompts,
)


print("[PPO 전 → PPO 후]")

for key in [
    "avg_reward",
    "ai_boilerplate_count",
    "avg_korean_ratio",
    "avg_response_tokens",
]:

    print(
        f"{key:<24}",
        ppo_before[key],
        "→",
        ppo_after[key],
    )

[PPO 전 → PPO 후]
avg_reward               -0.3616943359375 → -0.39668500423431396
ai_boilerplate_count     5 → 4
avg_korean_ratio         0.8334955591161104 → 0.7149687458731798
avg_response_tokens      40.578125 → 36.125


In [ ]:
# ============================================================
# Step 9-15. PPO 전후 실제 응답 비교
# ============================================================

ppo_comparison_df = pd.DataFrame({
    "prompt":
        ppo_eval_prompts,

    "before_PPO":
        ppo_before[
            "responses"
        ],

    "after_PPO":
        ppo_after[
            "responses"
        ],
})


display(
    ppo_comparison_df.head(15)
)

,prompt,before_PPO,after_PPO
0,네 확인 감사합니다. 주소 선생님 무슨 동이세요,"'죄송합니다, 저는 위치 정보를 알 수 없습니다. 어떤 동인지 알려주시면 더 정확한...","'제가 도와드릴 수 있는 부분이 있다면, 언제든지 말씀해주세요."
1,계약금 조절 가능합니까?,'계약금 조절 가능합니다! 계약금을 조정할 수 있는 방법은 다양합니다. 일반적으로 ...,'계약금 조절 가능합니다! \n\n'계약금 조절 가능합니다!
2,콩쿠르는 언제쯤 해요?,'콩쿠르는 2021년부터 2021년까지 진행됩니다.,'콩쿠르는 매년 가을에 개최됩니다. 그러나 일반적으로 콩쿠르는 매년 가을에 개최됩니다.
3,컨티넨탈 유로피언상의 최초 수상자가 누구야,'컨티넨탈 유로피언상의 최초 수상자는 알베르트 아인슈타인입니다.,'컨티넨탈 유로피언상의 최초 수상자는 알프레드 히딩크(Alfred Hidingk)입니다.
4,유아인이 영화 좋지 아니한가 에서 맡은 역은?,"'유아인이 영화 좋지 아니한가 에서 맡은 역할은 ""유아인""입니다.",'유아인이 \n\n'\n'\n''''''''''''''''''''''''''''''...
5,아뤼나 댕단은 어느 나라 선수야,'아뤼나 댕단은 프랑스 선수입니다.,'아뤼나 댕단은 중국입니다.
6,얘는 그래서 얼마에요?,"'죄송합니다, 제가 가격을 알 수 없습니다. 제가 가격을 알 수 없습니다. 제가 가...","'죄송합니다, 저는 가격 정보를 알 수 없습니다. 추가 정보를 제공해주시면 더 정확..."
7,스타크래프트 프로게이머 신상문 생년월일을 알려줘,'스타크래프트 프로게이머 신상문 생년월일을 알려드리겠습니다.,'스타크래프트 프로게이머 신상문 생년월일을 알려드립니다.
8,그래 그러자,'저는 인공지능이기 때문에 인공지능이 아니기 때문에 인공지능이 아니기 때문에 인공지...,'저는 인공지능이기 때문에 인공지능이 아닙니다. 그러나 저는 인공지능이기 때문에 인...
9,발표할때 자꾸 떨려,'저도 떨리는 것 같아요. 떨리는 이유는 여러 가지가 있을 수 있습니다.\n\n1....,"'저도 떨리는 것 같아요.\n\n결론적으로, 발표할 때 떨리는 것은 자연스러운 일이..."


In [ ]:
# ============================================================
# Step 9-16. PPO 학습 로그 요약
# ============================================================

ppo_history_df = pd.DataFrame(
    ppo_history
)

display(
    ppo_history_df.describe()
)

,batch,mean_rm_reward,mean_kl,policy_loss,value_loss,clip_fraction
count,64.000000,64.000000,6.400000e+01,64.000000,6.400000e+01,64.000000
mean,32.500000,-0.551388,1.469934e+07,207.817336,4.943934e+15,0.777502
std,18.618987,0.095100,6.413947e+07,1347.547693,3.747602e+16,0.061937
min,1.000000,-0.807617,1.206535e-03,-0.002315,4.962183e-02,0.570175
25%,16.750000,-0.613281,2.788358e+04,0.382344,2.524884e+08,0.746142
50%,32.500000,-0.565674,5.829275e+05,1.179335,1.047425e+11,0.779133
75%,48.250000,-0.495972,3.259545e+06,4.648863,3.277289e+12,0.819717
max,64.000000,-0.274170,4.992018e+08,10600.587891,2.998649e+17,0.896226


In [ ]:
# ============================================================
# Step 9-17. 최소 PPO Actor 저장
# ============================================================

PPO_MODEL_DIR = (
    LOCAL_MODELS
    / "m3_ppo_minimal"
)

PPO_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


actor_critic.actor.save_pretrained(
    str(PPO_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(PPO_MODEL_DIR)
)


torch.save(
    {
        "value_head":
            actor_critic
            .value_head
            .state_dict(),

        "ppo_config": {
            "batch_size":
                PPO_BATCH_SIZE,

            "max_new_tokens":
                PPO_MAX_NEW_TOKENS,

            "ppo_epochs":
                PPO_EPOCHS,

            "learning_rate":
                PPO_LR,

            "clip":
                PPO_CLIP,

            "kl_coef":
                PPO_KL_COEF,

            "value_coef":
                PPO_VALUE_COEF,

            "gamma":
                PPO_GAMMA,

            "lambda":
                PPO_LAMBDA,
        },

        "before":
            {
                key: value
                for key, value
                in ppo_before.items()
                if key != "responses"
            },

        "after":
            {
                key: value
                for key, value
                in ppo_after.items()
                if key != "responses"
            },
    },

    PPO_MODEL_DIR
    / "ppo_metadata.pt",
)


print(
    "PPO Actor 저장:",
    PPO_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

PPO Actor 저장: /content/gd08/models/m3_ppo_minimal


## 꼭 알아야할 PPO 용어!



```
Policy / Actor
= 답을 생성하는 현재 모델


Reference Model
= PPO 이전 SFT 모델
  Actor가 너무 멀리 가지 않도록 기준점 제공


Reward
= RM이 생성 답변에 준 점수


Value / Critic
= "이 상태에서 앞으로 reward가 얼마나 나올까?" 예측


Advantage
= 실제 결과가 Critic 예상보다 얼마나 좋거나 나빴나


KL
= 현재 Actor가 원래 SFT 모델에서 얼마나 멀어졌나


PPO Clip
= 한 번의 update가 너무 크지 못하도록 제한
```





```
                         Reward Model
                             │
                             ▼
Prompt → Actor → Response → Reward
          │
          │
          ├──── Reference
          │        │
          │        ▼
          │       KL
          │
          └──── Value Head
                   │
                   ▼
                Advantage
                   │
                   ▼
               PPO Clip
                   │
                   ▼
               Actor update
```



| 진단 지표          |                   결과 | 해석                          |
| -------------- | -------------------: | --------------------------- |
| RM reward      |  -0.362 → **-0.397** | 목표와 반대로 악화                  |
| 한글 비율          |  83.35% → **71.50%** | 생성 품질 저하                    |
| 평균 길이          |        40.58 → 36.13 | 짧아짐                         |
| Mean KL        |       **14,699,340** | 사실상 폭주                      |
| Policy loss 평균 |                207.8 | 비정상                         |
| Value loss 평균  |        **4.94×10¹⁵** | Critic 발산                   |
| Clip fraction  |            **0.778** | 대부분 token update가 clip 범위 밖 |
| 실제 문장          | 따옴표 반복, 사실 오류, 문장 붕괴 | policy collapse 시작          |


# 폭주


```
처음
Actor ≈ Reference

        ↓ PPO

기대:
Actor ── 조금 이동 ──> 더 높은 Reward

실제:
Actor ───────────────────────────────> 폭주
                 KL ≫≫≫≫
```



## 구조 수정


```
                     Reward Model
                         │
                         ▼
Prompt → Actor → Response → reward
          │
          ├──── Reference Model
          │        │
          │       KL(k1)
          │
          │
          └──── hidden state
                   │
                 detach()
                   │
                   ▼
               Value Head
                   │
                 value

Policy Loss
→ Actor만 update

Value Loss
→ Value Head만 update

Critic gradient가
Actor backbone으로 역전파되지 않음
```



In [ ]:
# ============================================================
# Step 9-18. 실패한 PPO 객체 정리
# ============================================================

import gc

for name in [
    "actor_critic",
    "reference_model",
    "ppo_reward_model",
    "ppo_optimizer",
]:

    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

GPU allocated: 3.54 GB
GPU reserved: 4.05 GB


In [ ]:
# ============================================================
# Step 9-19. 안정화된 Actor + Value 구조 정의
# ============================================================

class StableActorCritic(nn.Module):

    def __init__(
        self,
        actor_model,
    ):
        super().__init__()

        self.actor = actor_model

        hidden_size = (
            actor_model.config.hidden_size
        )

        self.value_head = nn.Linear(
            hidden_size,
            1,
        )

        nn.init.zeros_(
            self.value_head.weight
        )

        nn.init.zeros_(
            self.value_head.bias
        )


    def policy_logits(
        self,
        input_ids,
        attention_mask,
    ):

        outputs = (
            self.actor.transformer(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            )
        )

        hidden = outputs.last_hidden_state

        logits = (
            self.actor.lm_head(
                hidden
            )
        )

        return logits


    def values(
        self,
        input_ids,
        attention_mask,
    ):
        # Critic 학습이 Actor backbone을 수정하지 않도록
        # backbone representation에는 gradient를 흘리지 않는다.
        with torch.no_grad():

            outputs = (
                self.actor.transformer(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                    return_dict=True,
                )
            )

            hidden = (
                outputs.last_hidden_state
            )

        values = (
            self.value_head(
                hidden.detach()
            )
            .squeeze(-1)
        )

        return values

In [ ]:
# ============================================================
# Step 9-20. PPO v2 모델을 M2c에서 새로 초기화
# ============================================================

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)

RM_MODEL_PATH = (
    LOCAL_MODELS
    / "reward_model_final"
    / "reward_model.pt"
)


# Actor: 다시 깨끗한 M2c에서 시작
ppo_v2_actor_base = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR)
    )
)

ppo_v2_actor_base.config.use_cache = False

ppo_v2_model = (
    StableActorCritic(
        ppo_v2_actor_base
    )
    .to("cuda")
)


# Reference: 동일한 M2c, frozen
ppo_v2_reference = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR),
        dtype=torch.bfloat16,
    )
    .to("cuda")
)

ppo_v2_reference.eval()

for parameter in (
    ppo_v2_reference.parameters()
):
    parameter.requires_grad = False


# Reward Model: 이미 학습한 최종 RM
ppo_v2_reward_model = (
    KoGPT2RewardModel(
        MODEL_ID,
        cache_dir=str(HF_CACHE),
    )
)

rm_checkpoint = torch.load(
    RM_MODEL_PATH,
    map_location="cpu",
)

ppo_v2_reward_model.load_state_dict(
    rm_checkpoint[
        "model_state_dict"
    ]
)

ppo_v2_reward_model = (
    ppo_v2_reward_model
    .to(
        device="cuda",
        dtype=torch.bfloat16,
    )
)

ppo_v2_reward_model.eval()

for parameter in (
    ppo_v2_reward_model.parameters()
):
    parameter.requires_grad = False


print("Actor       : M2c fresh")
print("Reference   : frozen")
print("Reward Model: frozen")
print("Value Head  : trainable")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Actor       : M2c fresh
Reference   : frozen
Reward Model: frozen
Value Head  : trainable


In [ ]:
# ============================================================
# Step 9-21. PPO v2 안정화 설정
# ============================================================

PPO_V2_BATCH_SIZE = 4

# 이전 256개보다 줄여서 최소 실험
PPO_V2_TRAIN_PROMPTS = (
    ppo_train_prompts[:128]
)

PPO_V2_MAX_NEW_TOKENS = 48

PPO_V2_EPOCHS = 2


# Actor는 매우 작게 업데이트
PPO_V2_ACTOR_LR = 1e-6

# Value Head는 새로 초기화되었으므로 더 빠르게 학습
PPO_V2_VALUE_LR = 5e-5


PPO_V2_CLIP = 0.2
PPO_V2_VALUE_CLIP = 0.2

# 기존보다 강하게 Reference에 묶는다.
PPO_V2_KL_COEF = 0.10

PPO_V2_VALUE_COEF = 0.1

PPO_V2_GAMMA = 1.0
PPO_V2_LAMBDA = 0.95

PPO_V2_MAX_GRAD_NORM = 1.0


actor_optimizer = torch.optim.AdamW(
    ppo_v2_model.actor.parameters(),
    lr=PPO_V2_ACTOR_LR,
)

value_optimizer = torch.optim.AdamW(
    ppo_v2_model.value_head.parameters(),
    lr=PPO_V2_VALUE_LR,
)


print(
    "PPO v2 Train Prompts:",
    len(PPO_V2_TRAIN_PROMPTS)
)

PPO v2 Train Prompts: 128


In [ ]:
# ============================================================
# Step 9-22. Policy log probability 계산
# ============================================================

def stable_policy_action_logprobs(
    model,
    sequences,
    attention_mask,
    prompt_width,
):

    logits = model.policy_logits(
        sequences,
        attention_mask,
    )

    next_logits = (
        logits[:, :-1]
    )

    targets = (
        sequences[:, 1:]
    )

    token_log_probs = (
        torch.log_softmax(
            next_logits,
            dim=-1,
        )
        .gather(
            -1,
            targets.unsqueeze(-1),
        )
        .squeeze(-1)
    )


    start = prompt_width - 1

    action_log_probs = (
        token_log_probs[:, start:]
    )


    action_mask = (
        attention_mask[:, 1:][:, start:]
        .bool()
    )


    return (
        action_log_probs,
        action_mask,
    )

In [ ]:
def stable_action_values(
    model,
    sequences,
    attention_mask,
    prompt_width,
):

    values = model.values(
        sequences,
        attention_mask,
    )

    start = (
        prompt_width - 1
    )

    return (
        values[:, :-1][:, start:]
    )

In [ ]:
# ============================================================
# Step 9-23. Reference Model log probability
# ============================================================

@torch.no_grad()
def stable_reference_logprobs(
    model,
    sequences,
    attention_mask,
    prompt_width,
):

    outputs = model(
        input_ids=sequences,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = (
        outputs.logits[:, :-1]
    )

    targets = (
        sequences[:, 1:]
    )

    token_log_probs = (
        torch.log_softmax(
            logits,
            dim=-1,
        )
        .gather(
            -1,
            targets.unsqueeze(-1),
        )
        .squeeze(-1)
    )


    return token_log_probs[
        :,
        prompt_width - 1:
    ]

In [ ]:
# ============================================================
# Step 9-24. 안정적인 k1 KL 계산
# ============================================================

def k1_kl(
    policy_log_probs,
    reference_log_probs,
):

    # sampling distribution은 policy이므로
    # E_policy[log policy - log reference]
    return (
        policy_log_probs
        -
        reference_log_probs
    )

In [ ]:
# ============================================================
# Step 9-25. PPO v2 Response Reward 계산
# ============================================================

@torch.no_grad()
def score_v2_responses(
    prompts,
    responses,
):

    texts = [
        (
            format_instruction(prompt)
            + response
            + tokenizer.eos_token
        )

        for prompt, response
        in zip(
            prompts,
            responses,
        )
    ]


    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=RM_MAX_LENGTH,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        rewards = (
            ppo_v2_reward_model(
                tokens[
                    "input_ids"
                ],
                tokens[
                    "attention_mask"
                ],
            )
        )


    return rewards.float()

In [ ]:
# ============================================================
# Step 9-26. PPO v2 Advantage 계산
# ============================================================

def compute_gae_v2(
    rewards,
    values,
    action_mask,
):

    advantages = torch.zeros_like(
        rewards
    )

    returns = torch.zeros_like(
        rewards
    )


    for i in range(
        rewards.shape[0]
    ):

        length = int(
            action_mask[i]
            .sum()
            .item()
        )

        if length == 0:
            continue


        gae = torch.tensor(
            0.0,
            device=rewards.device,
        )


        for t in reversed(
            range(length)
        ):

            if t == length - 1:

                next_value = 0.0

            else:

                next_value = (
                    values[i, t + 1]
                )


            delta = (
                rewards[i, t]
                +
                PPO_V2_GAMMA
                * next_value
                -
                values[i, t]
            )


            gae = (
                delta
                +
                PPO_V2_GAMMA
                * PPO_V2_LAMBDA
                * gae
            )


            advantages[i, t] = gae


        returns[i, :length] = (
            advantages[i, :length]
            +
            values[i, :length]
        )


    valid = (
        advantages[action_mask]
    )


    # Advantage whitening
    advantages = (
        advantages
        -
        valid.mean()
    ) / (
        valid.std()
        + 1e-8
    )


    advantages = (
        advantages
        * action_mask
    )


    return (
        advantages.detach(),
        returns.detach(),
    )

In [ ]:
# ============================================================
# Step 9-27. 안정화된 PPO Rollout
# ============================================================

@torch.no_grad()
def make_ppo_v2_rollout(
    prompts,
):

    old_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    texts = [
        format_instruction(prompt)
        for prompt in prompts
    ]


    prompt_tokens = tokenizer(
        texts,
        padding=True,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    prompt_width = (
        prompt_tokens[
            "input_ids"
        ].shape[1]
    )


    ppo_v2_model.actor.eval()


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        sequences = (
            ppo_v2_model.actor.generate(
                **prompt_tokens,

                do_sample=True,

                temperature=1.0,
                top_k=0,
                top_p=1.0,

                repetition_penalty=1.0,

                max_new_tokens=
                    PPO_V2_MAX_NEW_TOKENS,

                eos_token_id=
                    tokenizer.eos_token_id,

                pad_token_id=
                    tokenizer.pad_token_id,

                use_cache=True,
            )
        )


    generated_ids = (
        sequences[:, prompt_width:]
    )


    response_mask = (
        generated_ids
        != tokenizer.pad_token_id
    )


    full_attention_mask = torch.cat(
        [
            prompt_tokens[
                "attention_mask"
            ],

            response_mask.long(),
        ],
        dim=1,
    )


    responses = [
        text.strip()

        for text in (
            tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )
        )
    ]


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        old_log_probs, action_mask = (
            stable_policy_action_logprobs(
                ppo_v2_model,
                sequences,
                full_attention_mask,
                prompt_width,
            )
        )

        old_values = (
            stable_action_values(
                ppo_v2_model,
                sequences,
                full_attention_mask,
                prompt_width,
            )
        )


    ref_log_probs = (
        stable_reference_logprobs(
            ppo_v2_reference,
            sequences,
            full_attention_mask,
            prompt_width,
        )
    )


    rm_rewards = score_v2_responses(
        prompts,
        responses,
    )


    # k1 KL
    kl_tokens = k1_kl(
        old_log_probs,
        ref_log_probs,
    )

    kl_tokens = (
        kl_tokens
        * action_mask
    )


    token_rewards = (
        -PPO_V2_KL_COEF
        * kl_tokens
    )


    # 실제 RM reward는 마지막 response token에 부여
    for i in range(
        len(prompts)
    ):

        length = int(
            action_mask[i]
            .sum()
            .item()
        )

        if length > 0:

            token_rewards[
                i,
                length - 1
            ] += rm_rewards[i]


    advantages, returns = (
        compute_gae_v2(
            token_rewards,
            old_values,
            action_mask,
        )
    )


    tokenizer.padding_side = (
        old_padding_side
    )


    return {
        "prompts":
            prompts,

        "responses":
            responses,

        "sequences":
            sequences,

        "attention_mask":
            full_attention_mask,

        "prompt_width":
            prompt_width,

        "action_mask":
            action_mask,

        "old_log_probs":
            old_log_probs.detach(),

        "old_values":
            old_values.detach(),

        "advantages":
            advantages,

        "returns":
            returns,

        "rm_rewards":
            rm_rewards.detach(),

        "kl_tokens":
            kl_tokens.detach(),
    }

In [ ]:
# ============================================================
# Step 9-28. PPO v2 Clipped Update
# ============================================================

def ppo_v2_update(
    rollout,
):

    final_stats = None


    for epoch in range(
        PPO_V2_EPOCHS
    ):

        # --------------------------------
        # 1. Actor update
        # --------------------------------

        ppo_v2_model.actor.train()


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            new_log_probs, action_mask = (
                stable_policy_action_logprobs(
                    ppo_v2_model,

                    rollout[
                        "sequences"
                    ],

                    rollout[
                        "attention_mask"
                    ],

                    rollout[
                        "prompt_width"
                    ],
                )
            )


            log_ratio = (
                new_log_probs
                -
                rollout[
                    "old_log_probs"
                ]
            )


            # numerical safety
            ratio = torch.exp(
                torch.clamp(
                    log_ratio,
                    min=-20,
                    max=20,
                )
            )


            surrogate_1 = (
                ratio
                * rollout[
                    "advantages"
                ]
            )

            surrogate_2 = (
                torch.clamp(
                    ratio,
                    1 - PPO_V2_CLIP,
                    1 + PPO_V2_CLIP,
                )
                *
                rollout[
                    "advantages"
                ]
            )


            policy_loss = (
                -masked_mean(
                    torch.minimum(
                        surrogate_1,
                        surrogate_2,
                    ),
                    action_mask,
                )
            )


        actor_optimizer.zero_grad(
            set_to_none=True
        )

        policy_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            ppo_v2_model.actor.parameters(),
            PPO_V2_MAX_GRAD_NORM,
        )

        actor_optimizer.step()


        # --------------------------------
        # 2. Value Head update
        # --------------------------------

        new_values = (
            stable_action_values(
                ppo_v2_model,
                rollout[
                    "sequences"
                ],
                rollout[
                    "attention_mask"
                ],
                rollout[
                    "prompt_width"
                ],
            )
        )


        old_values = (
            rollout[
                "old_values"
            ]
        )


        value_clipped = (
            old_values
            +
            torch.clamp(
                new_values
                - old_values,

                -PPO_V2_VALUE_CLIP,
                PPO_V2_VALUE_CLIP,
            )
        )


        value_loss_1 = (
            new_values
            -
            rollout[
                "returns"
            ]
        ) ** 2


        value_loss_2 = (
            value_clipped
            -
            rollout[
                "returns"
            ]
        ) ** 2


        value_loss = (
            0.5
            *
            masked_mean(
                torch.maximum(
                    value_loss_1,
                    value_loss_2,
                ),
                action_mask,
            )
        )


        value_optimizer.zero_grad(
            set_to_none=True
        )

        (
            PPO_V2_VALUE_COEF
            * value_loss
        ).backward()

        torch.nn.utils.clip_grad_norm_(
            ppo_v2_model.value_head.parameters(),
            PPO_V2_MAX_GRAD_NORM,
        )

        value_optimizer.step()


        # --------------------------------
        # 3. PPO update 안정성 진단
        # --------------------------------

        with torch.no_grad():

            approx_kl_old = (
                0.5
                *
                masked_mean(
                    log_ratio ** 2,
                    action_mask,
                )
            )

            clip_fraction = (
                masked_mean(
                    (
                        torch.abs(
                            ratio - 1
                        )
                        > PPO_V2_CLIP
                    ).float(),
                    action_mask,
                )
            )


        final_stats = {
            "policy_loss":
                float(
                    policy_loss.detach()
                ),

            "value_loss":
                float(
                    value_loss.detach()
                ),

            "approx_kl_old":
                float(
                    approx_kl_old
                ),

            "clip_fraction":
                float(
                    clip_fraction
                ),
        }


        # 같은 rollout에 대해 정책이 너무 많이 바뀌면
        # 두 번째 epoch를 즉시 중단한다.
        if (
            approx_kl_old.item()
            > 0.05
        ):
            break


    return final_stats

In [ ]:
# ============================================================
# Step 9-29. PPO v2 평가 함수
# ============================================================

@torch.no_grad()
def evaluate_ppo_v2(
    actor_model,
    prompts,
):

    old_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    responses = []


    for start in range(
        0,
        len(prompts),
        8,
    ):

        batch_prompts = (
            prompts[
                start:start + 8
            ]
        )


        texts = [
            format_instruction(p)
            for p in batch_prompts
        ]


        tokens = tokenizer(
            texts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        ).to("cuda")


        prompt_width = (
            tokens[
                "input_ids"
            ].shape[1]
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            outputs = actor_model.generate(
                **tokens,

                do_sample=False,
                num_beams=4,
                early_stopping=True,

                max_new_tokens=80,

                eos_token_id=
                    tokenizer.eos_token_id,

                pad_token_id=
                    tokenizer.pad_token_id,
            )


        generated = (
            outputs[:, prompt_width:]
        )


        responses.extend(
            [
                text.strip()

                for text in (
                    tokenizer.batch_decode(
                        generated,
                        skip_special_tokens=True,
                    )
                )
            ]
        )


    rewards = score_v2_responses(
        prompts,
        responses,
    )


    tokenizer.padding_side = (
        old_padding_side
    )


    return {
        "avg_reward":
            float(
                rewards.mean().cpu()
            ),

        "ai_boilerplate_count":
            int(
                sum(
                    has_ai_boilerplate(r)
                    for r in responses
                )
            ),

        "avg_korean_ratio":
            float(
                sum(
                    korean_ratio(r)
                    for r in responses
                )
                / len(responses)
            ),

        "avg_response_tokens":
            float(
                sum(
                    count_tokens(r)
                    for r in responses
                )
                / len(responses)
            ),

        "responses":
            responses,
    }

In [ ]:
ppo_v2_before = evaluate_ppo_v2(
    ppo_v2_model.actor,
    ppo_eval_prompts,
)

print("[PPO v2 학습 전]")

for key, value in (
    ppo_v2_before.items()
):

    if key != "responses":
        print(
            f"{key:<24}",
            value
        )

[PPO v2 학습 전]
avg_reward               -0.3616943359375
ai_boilerplate_count     5
avg_korean_ratio         0.8334955591161104
avg_response_tokens      40.578125


In [ ]:
# ============================================================
# Step 9-30A. 중단된 PPO v2 객체 정리
# ============================================================

import gc
import math

for name in [
    "ppo_v2_model",
    "ppo_v2_reference",
    "ppo_v2_reward_model",
    "actor_optimizer",
    "value_optimizer",
]:

    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

GPU allocated: 4.49 GB
GPU reserved: 4.95 GB


In [ ]:
# ============================================================
# Step 9-30B/C. PPO v2 모델·Optimizer 완전 재초기화
# ============================================================

import gc
import math
import random
import torch
import torch.nn as nn

from transformers import AutoModelForCausalLM


# ------------------------------------------------------------
# 1. 이전 PPO 객체 정리
# ------------------------------------------------------------

for name in [
    "ppo_v2_model",
    "ppo_v2_reference",
    "ppo_v2_reward_model",
    "actor_optimizer",
    "value_optimizer",
]:

    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 2. 저장 경로 확인
# ------------------------------------------------------------

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)

RM_MODEL_PATH = (
    LOCAL_MODELS
    / "reward_model_final"
    / "reward_model.pt"
)

print("Actor checkpoint :", M2C_MODEL_DIR)
print("Reward checkpoint:", RM_MODEL_PATH)


# ------------------------------------------------------------
# 3. Actor + Value 구조
# ------------------------------------------------------------

class StableActorCritic(nn.Module):

    def __init__(self, actor_model):
        super().__init__()

        self.actor = actor_model

        hidden_size = (
            actor_model.config.hidden_size
        )

        self.value_head = nn.Linear(
            hidden_size,
            1,
        )

        nn.init.zeros_(
            self.value_head.weight
        )

        nn.init.zeros_(
            self.value_head.bias
        )


    def policy_logits(
        self,
        input_ids,
        attention_mask,
    ):

        outputs = (
            self.actor.transformer(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            )
        )

        hidden = outputs.last_hidden_state

        return self.actor.lm_head(
            hidden
        )


    def values(
        self,
        input_ids,
        attention_mask,
    ):

        # Value loss가 Actor backbone까지
        # 역전파되지 않도록 차단
        with torch.no_grad():

            outputs = (
                self.actor.transformer(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                    return_dict=True,
                )
            )

            hidden = (
                outputs.last_hidden_state
            )

        return (
            self.value_head(
                hidden.detach()
            )
            .squeeze(-1)
        )


# ------------------------------------------------------------
# 4. Actor: 깨끗한 M2c checkpoint에서 다시 시작
# ------------------------------------------------------------

ppo_v2_actor_base = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR)
    )
)

ppo_v2_actor_base.config.use_cache = False

ppo_v2_model = (
    StableActorCritic(
        ppo_v2_actor_base
    )
    .to("cuda")
)


# ------------------------------------------------------------
# 5. Reference Model: 동일한 M2c, frozen
# ------------------------------------------------------------

ppo_v2_reference = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR),
        dtype=torch.bfloat16,
    )
    .to("cuda")
)

ppo_v2_reference.eval()

for parameter in (
    ppo_v2_reference.parameters()
):
    parameter.requires_grad = False


# ------------------------------------------------------------
# 6. Reward Model: 학습 완료된 RM, frozen
# ------------------------------------------------------------

ppo_v2_reward_model = (
    KoGPT2RewardModel(
        MODEL_ID,
        cache_dir=str(HF_CACHE),
    )
)

rm_checkpoint = torch.load(
    RM_MODEL_PATH,
    map_location="cpu",
)

ppo_v2_reward_model.load_state_dict(
    rm_checkpoint[
        "model_state_dict"
    ]
)

ppo_v2_reward_model = (
    ppo_v2_reward_model
    .to(
        device="cuda",
        dtype=torch.bfloat16,
    )
)

ppo_v2_reward_model.eval()

for parameter in (
    ppo_v2_reward_model.parameters()
):
    parameter.requires_grad = False


# ------------------------------------------------------------
# 7. PPO v2 안정화 설정
# ------------------------------------------------------------

PPO_V2_BATCH_SIZE = 4

PPO_V2_TRAIN_PROMPTS = (
    ppo_train_prompts[:128]
)

PPO_V2_MAX_NEW_TOKENS = 48

PPO_V2_EPOCHS = 2

PPO_V2_ACTOR_LR = 2e-7
PPO_V2_VALUE_LR = 5e-5

PPO_V2_CLIP = 0.2
PPO_V2_VALUE_CLIP = 0.2

PPO_V2_KL_COEF = 0.20
PPO_V2_VALUE_COEF = 0.1

PPO_V2_GAMMA = 1.0
PPO_V2_LAMBDA = 0.95

PPO_V2_MAX_GRAD_NORM = 1.0

PPO_V2_TARGET_KL = 0.02


# ------------------------------------------------------------
# 8. Actor / Value optimizer를 분리
# ------------------------------------------------------------

actor_optimizer = torch.optim.AdamW(
    ppo_v2_model.actor.parameters(),
    lr=PPO_V2_ACTOR_LR,
)

value_optimizer = torch.optim.AdamW(
    ppo_v2_model.value_head.parameters(),
    lr=PPO_V2_VALUE_LR,
)


# ------------------------------------------------------------
# 9. 상태 확인
# ------------------------------------------------------------

print("\n[PPO v2 초기화 완료]")

print(
    "Actor:",
    ppo_v2_model.actor.__class__.__name__
)

print(
    "Reference frozen:",
    not any(
        p.requires_grad
        for p in ppo_v2_reference.parameters()
    )
)

print(
    "Reward Model frozen:",
    not any(
        p.requires_grad
        for p in ppo_v2_reward_model.parameters()
    )
)

print(
    "Value Head trainable:",
    any(
        p.requires_grad
        for p in ppo_v2_model.value_head.parameters()
    )
)

print(
    "Train prompts:",
    len(PPO_V2_TRAIN_PROMPTS)
)

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

Actor checkpoint : /content/gd08/models/m2c_clean_v2_stepmatched
Reward checkpoint: /content/gd08/models/reward_model_final/reward_model.pt


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[PPO v2 초기화 완료]
Actor: GPT2LMHeadModel
Reference frozen: True
Reward Model frozen: True
Value Head trainable: True
Train prompts: 128
GPU allocated: 4.49 GB


In [ ]:
# ============================================================
# Step 9-30D. PPO update 후 KL을 즉시 확인하는 안정화 버전
# ============================================================

def ppo_v2_update(
    rollout,
):

    final_stats = None


    for epoch in range(PPO_V2_EPOCHS):

        # --------------------------------
        # 1. Actor update 전 policy
        # --------------------------------

        ppo_v2_model.actor.train()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            new_log_probs, action_mask = (
                stable_policy_action_logprobs(
                    ppo_v2_model,
                    rollout["sequences"],
                    rollout["attention_mask"],
                    rollout["prompt_width"],
                )
            )

            log_ratio = (
                new_log_probs
                - rollout["old_log_probs"]
            )

            ratio = torch.exp(
                torch.clamp(
                    log_ratio,
                    min=-20,
                    max=20,
                )
            )

            surrogate_1 = (
                ratio
                * rollout["advantages"]
            )

            surrogate_2 = (
                torch.clamp(
                    ratio,
                    1 - PPO_V2_CLIP,
                    1 + PPO_V2_CLIP,
                )
                * rollout["advantages"]
            )

            policy_loss = -masked_mean(
                torch.minimum(
                    surrogate_1,
                    surrogate_2,
                ),
                action_mask,
            )


        actor_optimizer.zero_grad(
            set_to_none=True
        )

        policy_loss.backward()

        actor_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                ppo_v2_model.actor.parameters(),
                PPO_V2_MAX_GRAD_NORM,
            )
        )

        actor_optimizer.step()


        # --------------------------------
        # 2. Actor update 직후 실제 policy 이동 측정
        # --------------------------------

        with torch.no_grad():

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):

                post_log_probs, _ = (
                    stable_policy_action_logprobs(
                        ppo_v2_model,
                        rollout["sequences"],
                        rollout["attention_mask"],
                        rollout["prompt_width"],
                    )
                )

            post_log_ratio = (
                post_log_probs
                - rollout["old_log_probs"]
            )

            approx_kl_post = (
                0.5
                * masked_mean(
                    post_log_ratio ** 2,
                    action_mask,
                )
            )

            post_ratio = torch.exp(
                torch.clamp(
                    post_log_ratio,
                    min=-20,
                    max=20,
                )
            )

            clip_fraction = masked_mean(
                (
                    torch.abs(
                        post_ratio - 1
                    )
                    > PPO_V2_CLIP
                ).float(),
                action_mask,
            )


        # --------------------------------
        # 3. Value Head만 update
        # --------------------------------

        new_values = stable_action_values(
            ppo_v2_model,
            rollout["sequences"],
            rollout["attention_mask"],
            rollout["prompt_width"],
        )

        old_values = (
            rollout["old_values"]
        )

        value_clipped = (
            old_values
            +
            torch.clamp(
                new_values - old_values,
                -PPO_V2_VALUE_CLIP,
                PPO_V2_VALUE_CLIP,
            )
        )

        value_loss_1 = (
            new_values
            - rollout["returns"]
        ) ** 2

        value_loss_2 = (
            value_clipped
            - rollout["returns"]
        ) ** 2

        value_loss = (
            0.5
            * masked_mean(
                torch.maximum(
                    value_loss_1,
                    value_loss_2,
                ),
                action_mask,
            )
        )


        value_optimizer.zero_grad(
            set_to_none=True
        )

        (
            PPO_V2_VALUE_COEF
            * value_loss
        ).backward()

        value_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                ppo_v2_model.value_head.parameters(),
                PPO_V2_MAX_GRAD_NORM,
            )
        )

        value_optimizer.step()


        final_stats = {
            "policy_loss":
                float(policy_loss.detach()),

            "value_loss":
                float(value_loss.detach()),

            "approx_kl_post":
                float(approx_kl_post),

            "clip_fraction":
                float(clip_fraction),

            "actor_grad_norm":
                float(actor_grad_norm),

            "value_grad_norm":
                float(value_grad_norm),
        }


        # 첫 update만으로 너무 멀리 갔으면
        # 같은 rollout의 추가 PPO epoch는 수행하지 않음
        if (
            approx_kl_post.item()
            > PPO_V2_TARGET_KL
        ):
            break


    return final_stats

In [ ]:
# ============================================================
# Step 9-30E. 안정화된 최소 PPO 재실행
# ============================================================

import math
import random

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
random.seed(42)


train_prompts_v2 = (
    PPO_V2_TRAIN_PROMPTS.copy()
)

random.shuffle(
    train_prompts_v2
)


ppo_v2_history = []


progress = tqdm(
    range(
        0,
        len(train_prompts_v2),
        PPO_V2_BATCH_SIZE,
    ),
    desc="PPO v2 Stable",
)


for batch_index, start in enumerate(
    progress,
    start=1,
):

    prompts = (
        train_prompts_v2[
            start:
            start + PPO_V2_BATCH_SIZE
        ]
    )

    rollout = make_ppo_v2_rollout(
        prompts
    )

    stats = ppo_v2_update(
        rollout
    )


    valid_ref_kl = (
        rollout["kl_tokens"][
            rollout["action_mask"]
        ]
    )


    row = {
        "batch":
            batch_index,

        "mean_rm_reward":
            float(
                rollout["rm_rewards"]
                .mean()
                .cpu()
            ),

        # k1 sample estimator는 작은 batch에서
        # 소폭 음수가 나올 수 있음
        "mean_ref_kl":
            float(
                valid_ref_kl
                .mean()
                .cpu()
            ),

        **stats,
    }


    ppo_v2_history.append(
        row
    )


    progress.set_postfix({
        "reward":
            f"{row['mean_rm_reward']:.3f}",

        "ref_kl":
            f"{row['mean_ref_kl']:.4f}",

        "post_kl":
            f"{row['approx_kl_post']:.4f}",

        "clip":
            f"{row['clip_fraction']:.2f}",
    })


    values_to_check = [
        row["policy_loss"],
        row["value_loss"],
        row["approx_kl_post"],
        row["mean_ref_kl"],
        row["actor_grad_norm"],
    ]


    if not all(
        math.isfinite(x)
        for x in values_to_check
    ):

        print(
            "\n비정상 수치 발생 → PPO 즉시 중단"
        )

        break


    # 전체 정책이 다시 너무 크게 움직이기 시작하면
    # 실험 자체도 조기 종료
    if row["approx_kl_post"] > 0.10:

        print(
            "\nPolicy 이동량이 너무 큼"
            " → PPO 실험 조기 종료"
        )

        break

PPO v2 Stable:   0%|          | 0/32 [00:00<?, ?it/s]


Policy 이동량이 너무 큼 → PPO 실험 조기 종료


In [ ]:
# ============================================================
# Step 9-34. 중단된 PPO Actor 폐기
# ============================================================

import gc
import math
import random

for name in [
    "ppo_v2_model",
    "ppo_v2_reference",
    "ppo_v2_reward_model",
    "actor_optimizer",
    "value_optimizer",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

GPU allocated: 4.48 GB


In [ ]:
# ============================================================
# Step 9-35. PPO용 모델의 Dropout 완전 비활성화 함수
# ============================================================

def disable_dropout(model):

    dropout_count = 0

    for module in model.modules():

        if isinstance(
            module,
            torch.nn.Dropout,
        ):
            module.p = 0.0
            dropout_count += 1

    model.eval()

    return dropout_count

In [ ]:
# ============================================================
# Step 9-36. PPO v3 모델 완전 재초기화
# ============================================================

M2C_MODEL_DIR = (
    LOCAL_MODELS
    / "m2c_clean_v2_stepmatched"
)

RM_MODEL_PATH = (
    LOCAL_MODELS
    / "reward_model_final"
    / "reward_model.pt"
)


# ---------------------------------------
# Actor
# ---------------------------------------

actor_base = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR)
    )
)

actor_base.config.use_cache = False

ppo_model = StableActorCritic(
    actor_base
).to("cuda")

actor_dropout_count = disable_dropout(
    ppo_model.actor
)


# ---------------------------------------
# Reference
# ---------------------------------------

ppo_reference = (
    AutoModelForCausalLM
    .from_pretrained(
        str(M2C_MODEL_DIR),
        dtype=torch.bfloat16,
    )
    .to("cuda")
)

reference_dropout_count = disable_dropout(
    ppo_reference
)

for parameter in (
    ppo_reference.parameters()
):
    parameter.requires_grad = False


# ---------------------------------------
# Reward Model
# ---------------------------------------

ppo_reward_model = (
    KoGPT2RewardModel(
        MODEL_ID,
        cache_dir=str(HF_CACHE),
    )
)

rm_checkpoint = torch.load(
    RM_MODEL_PATH,
    map_location="cpu",
)

ppo_reward_model.load_state_dict(
    rm_checkpoint[
        "model_state_dict"
    ]
)

ppo_reward_model = (
    ppo_reward_model
    .to(
        device="cuda",
        dtype=torch.bfloat16,
    )
)

disable_dropout(
    ppo_reward_model
)

for parameter in (
    ppo_reward_model.parameters()
):
    parameter.requires_grad = False


print(
    "Actor dropout layers disabled:",
    actor_dropout_count
)

print(
    "Reference dropout layers disabled:",
    reference_dropout_count
)

print(
    "Actor training flag:",
    ppo_model.actor.training
)

print(
    "Reference training flag:",
    ppo_reference.training
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Actor dropout layers disabled: 37
Reference dropout layers disabled: 37
Actor training flag: False
Reference training flag: False


In [ ]:
# ============================================================
# Step 9-37. 최종 최소 PPO 설정
# ============================================================

PPO_BATCH_SIZE = 4

PPO_TRAIN_PROMPTS = (
    ppo_train_prompts[:128]
)

PPO_MAX_NEW_TOKENS = 48

# 하나의 rollout에 update 1회만
PPO_EPOCHS = 1

PPO_ACTOR_LR = 2e-7
PPO_VALUE_LR = 5e-5

PPO_CLIP = 0.2
PPO_VALUE_CLIP = 0.2

PPO_KL_COEF = 0.20

PPO_GAMMA = 1.0
PPO_LAMBDA = 0.95

PPO_MAX_GRAD_NORM = 1.0

PPO_TARGET_KL = 0.02


actor_optimizer = torch.optim.AdamW(
    ppo_model.actor.parameters(),
    lr=PPO_ACTOR_LR,
)

value_optimizer = torch.optim.AdamW(
    ppo_model.value_head.parameters(),
    lr=PPO_VALUE_LR,
)

In [ ]:
# ============================================================
# Step 9-38. PPO update 전 log probability 일치 검증
# ============================================================

@torch.no_grad()
def ppo_preupdate_consistency_check(
    prompts,
):

    old_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    texts = [
        format_instruction(p)
        for p in prompts
    ]

    tokens = tokenizer(
        texts,
        padding=True,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    prompt_width = (
        tokens["input_ids"].shape[1]
    )


    sequences = ppo_model.actor.generate(
        **tokens,

        do_sample=True,
        temperature=1.0,
        top_k=0,
        top_p=1.0,

        max_new_tokens=32,

        eos_token_id=
            tokenizer.eos_token_id,

        pad_token_id=
            tokenizer.pad_token_id,

        use_cache=True,
    )


    generated_ids = (
        sequences[:, prompt_width:]
    )

    response_mask = (
        generated_ids
        != tokenizer.pad_token_id
    )

    full_mask = torch.cat(
        [
            tokens["attention_mask"],
            response_mask.long(),
        ],
        dim=1,
    )


    # 동일 Actor에서 두 번 연속 계산
    log_probs_1, action_mask = (
        stable_policy_action_logprobs(
            ppo_model,
            sequences,
            full_mask,
            prompt_width,
        )
    )

    log_probs_2, _ = (
        stable_policy_action_logprobs(
            ppo_model,
            sequences,
            full_mask,
            prompt_width,
        )
    )


    diff = (
        log_probs_2
        - log_probs_1
    )

    valid_diff = (
        diff[action_mask]
    )


    max_abs_diff = (
        valid_diff
        .abs()
        .max()
        .item()
    )

    mean_abs_diff = (
        valid_diff
        .abs()
        .mean()
        .item()
    )

    approx_kl = (
        0.5
        * (
            valid_diff ** 2
        ).mean()
        .item()
    )


    tokenizer.padding_side = (
        old_padding_side
    )


    return {
        "max_abs_logprob_diff":
            max_abs_diff,

        "mean_abs_logprob_diff":
            mean_abs_diff,

        "approx_kl_same_policy":
            approx_kl,
    }

In [ ]:
consistency_result = (
    ppo_preupdate_consistency_check(
        PPO_TRAIN_PROMPTS[:4]
    )
)

print(consistency_result)

{'max_abs_logprob_diff': 0.0, 'mean_abs_logprob_diff': 0.0, 'approx_kl_same_policy': 0.0}


In [ ]:
# ============================================================
# Step 9-39. 최종 PPO Rollout
# ============================================================

@torch.no_grad()
def make_final_ppo_rollout(
    prompts,
):

    old_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    texts = [
        format_instruction(p)
        for p in prompts
    ]


    prompt_tokens = tokenizer(
        texts,
        padding=True,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    prompt_width = (
        prompt_tokens[
            "input_ids"
        ].shape[1]
    )


    # dropout은 계속 OFF
    ppo_model.actor.eval()


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        sequences = (
            ppo_model.actor.generate(
                **prompt_tokens,

                do_sample=True,
                temperature=1.0,
                top_k=0,
                top_p=1.0,

                repetition_penalty=1.0,

                max_new_tokens=
                    PPO_MAX_NEW_TOKENS,

                eos_token_id=
                    tokenizer.eos_token_id,

                pad_token_id=
                    tokenizer.pad_token_id,

                use_cache=True,
            )
        )


    generated_ids = (
        sequences[:, prompt_width:]
    )


    response_mask = (
        generated_ids
        != tokenizer.pad_token_id
    )


    full_attention_mask = torch.cat(
        [
            prompt_tokens[
                "attention_mask"
            ],

            response_mask.long(),
        ],
        dim=1,
    )


    responses = [
        text.strip()

        for text in (
            tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )
        )
    ]


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        old_log_probs, action_mask = (
            stable_policy_action_logprobs(
                ppo_model,

                sequences,
                full_attention_mask,
                prompt_width,
            )
        )

        old_values = (
            stable_action_values(
                ppo_model,

                sequences,
                full_attention_mask,
                prompt_width,
            )
        )


    ref_log_probs = (
        stable_reference_logprobs(
            ppo_reference,
            sequences,
            full_attention_mask,
            prompt_width,
        )
    )


    # Reward Model 채점
    reward_texts = [
        (
            format_instruction(prompt)
            + response
            + tokenizer.eos_token
        )

        for prompt, response
        in zip(
            prompts,
            responses,
        )
    ]


    reward_tokens = tokenizer(
        reward_texts,
        padding=True,
        truncation=True,
        max_length=RM_MAX_LENGTH,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        rm_rewards = (
            ppo_reward_model(
                reward_tokens[
                    "input_ids"
                ],

                reward_tokens[
                    "attention_mask"
                ],
            )
            .float()
        )


    # k1 KL
    kl_tokens = (
        old_log_probs
        - ref_log_probs
    )

    kl_tokens = (
        kl_tokens
        * action_mask
    )


    token_rewards = (
        -PPO_KL_COEF
        * kl_tokens
    )


    for i in range(
        len(prompts)
    ):

        length = int(
            action_mask[i]
            .sum()
            .item()
        )

        if length > 0:

            token_rewards[
                i,
                length - 1
            ] += rm_rewards[i]


    advantages, returns = (
        compute_gae_v2(
            token_rewards,
            old_values,
            action_mask,
        )
    )


    tokenizer.padding_side = (
        old_padding_side
    )


    return {
        "prompts":
            prompts,

        "responses":
            responses,

        "sequences":
            sequences,

        "attention_mask":
            full_attention_mask,

        "prompt_width":
            prompt_width,

        "action_mask":
            action_mask,

        "old_log_probs":
            old_log_probs.detach(),

        "old_values":
            old_values.detach(),

        "advantages":
            advantages.detach(),

        "returns":
            returns.detach(),

        "rm_rewards":
            rm_rewards.detach(),

        "kl_tokens":
            kl_tokens.detach(),
    }

In [ ]:
# ============================================================
# Step 9-40. Dropout을 끈 최종 PPO update
# ============================================================

def final_ppo_update(
    rollout,
):

    # 중요:
    # eval mode여도 gradient는 정상 계산된다.
    ppo_model.actor.eval()


    # ---------------------------------------
    # Actor
    # ---------------------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):

        new_log_probs, action_mask = (
            stable_policy_action_logprobs(
                ppo_model,

                rollout["sequences"],
                rollout["attention_mask"],
                rollout["prompt_width"],
            )
        )


        log_ratio = (
            new_log_probs
            -
            rollout["old_log_probs"]
        )


        ratio = torch.exp(
            torch.clamp(
                log_ratio,
                -20,
                20,
            )
        )


        surrogate_1 = (
            ratio
            * rollout["advantages"]
        )

        surrogate_2 = (
            torch.clamp(
                ratio,
                1 - PPO_CLIP,
                1 + PPO_CLIP,
            )
            * rollout["advantages"]
        )


        policy_loss = (
            -masked_mean(
                torch.minimum(
                    surrogate_1,
                    surrogate_2,
                ),
                action_mask,
            )
        )


    actor_optimizer.zero_grad(
        set_to_none=True
    )

    policy_loss.backward()


    actor_grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            ppo_model.actor.parameters(),
            PPO_MAX_GRAD_NORM,
        )
    )


    actor_optimizer.step()


    # ---------------------------------------
    # Update 직후 실제 정책 이동
    # ---------------------------------------

    with torch.no_grad():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            post_log_probs, _ = (
                stable_policy_action_logprobs(
                    ppo_model,

                    rollout["sequences"],
                    rollout[
                        "attention_mask"
                    ],

                    rollout[
                        "prompt_width"
                    ],
                )
            )


        post_log_ratio = (
            post_log_probs
            -
            rollout["old_log_probs"]
        )


        approx_kl_post = (
            0.5
            * masked_mean(
                post_log_ratio ** 2,
                action_mask,
            )
        )


        post_ratio = torch.exp(
            torch.clamp(
                post_log_ratio,
                -20,
                20,
            )
        )


        clip_fraction = (
            masked_mean(
                (
                    torch.abs(
                        post_ratio - 1
                    )
                    > PPO_CLIP
                )
                .float(),

                action_mask,
            )
        )


    # ---------------------------------------
    # Value Head
    # ---------------------------------------

    new_values = stable_action_values(
        ppo_model,

        rollout["sequences"],
        rollout["attention_mask"],
        rollout["prompt_width"],
    )


    value_loss = (
        0.5
        * masked_mean(
            (
                new_values
                - rollout["returns"]
            ) ** 2,

            action_mask,
        )
    )


    value_optimizer.zero_grad(
        set_to_none=True
    )

    value_loss.backward()


    value_grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            ppo_model.value_head.parameters(),
            PPO_MAX_GRAD_NORM,
        )
    )


    value_optimizer.step()


    return {
        "policy_loss":
            float(policy_loss.detach()),

        "value_loss":
            float(value_loss.detach()),

        "approx_kl_post":
            float(approx_kl_post),

        "clip_fraction":
            float(clip_fraction),

        "actor_grad_norm":
            float(actor_grad_norm),

        "value_grad_norm":
            float(value_grad_norm),
    }

In [ ]:
# ============================================================
# Step 9-41. PPO 1-batch smoke test
# ============================================================

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


smoke_prompts = (
    PPO_TRAIN_PROMPTS[:4]
)


smoke_rollout = (
    make_final_ppo_rollout(
        smoke_prompts
    )
)


smoke_stats = (
    final_ppo_update(
        smoke_rollout
    )
)


print("[PPO 1-batch smoke test]")

print(
    "RM reward:",
    float(
        smoke_rollout[
            "rm_rewards"
        ].mean()
    )
)

print(
    "Reference KL:",
    float(
        smoke_rollout[
            "kl_tokens"
        ][
            smoke_rollout[
                "action_mask"
            ]
        ].mean()
    )
)

print(smoke_stats)

[PPO 1-batch smoke test]
RM reward: -0.608642578125
Reference KL: 0.006023364141583443
{'policy_loss': 4.0531158873591266e-08, 'value_loss': 0.07355108112096786, 'approx_kl_post': 0.04391741752624512, 'clip_fraction': 0.09000000357627869, 'actor_grad_norm': 46.340274810791016, 'value_grad_norm': 11.7505521774292}


### PPO가 불안정했던 원인 찾기

- PPO는 Reward Model의 점수를 높이도록 SFT 모델을 조금씩 수정하는 학습이다.
- 하지만 모델이 한 번에 너무 많이 바뀌면 기존에 잘하던 한국어 생성 능력까지 무너질 수 있기 때문에 원래 SFT 모델을 Reference Model로 보관하고 두 모델의 차이를 KL 값으로 확인한다.
- 첫 PPO 실험에서는 KL과 Value Loss가 매우 크게 증가했고 실제 생성 문장도 반복되거나 깨지면서 학습이 발산했다.
- 원인을 확인해보니 답변을 생성할 때는 Actor가 `eval` 상태여서 Dropout이 꺼져 있었지만, PPO update에서는 `train` 상태로 바뀌면서 Dropout이 다시 켜지고 있었다.
- 그 결과 실제 parameter가 크게 변하지 않았는데도 같은 문장의 token 확률이 달라져 PPO가 모델이 크게 변했다고 잘못 판단할 수 있었다.
- Dropout을 완전히 끈 뒤 같은 Actor에 같은 문장을 두 번 입력해 확인한 결과 log probability 차이와 KL이 모두 정확히 0으로 나타났다.
- 따라서 Dropout으로 인해 가짜 policy 차이가 생기던 문제는 해결되었다.
- 수정 후 1개의 batch만 PPO update한 결과 post-update KL은 이전 약 1.02에서 약 0.044로 크게 감소했고 clip fraction도 약 78%에서 9%로 감소했다.
- Value Loss도 비정상적으로 큰 값에서 약 0.074로 정상화되어 PPO가 이전처럼 발산하지 않는 것을 확인했다.

# 10. 프로젝트 최종 결과

## 10-1. 프로젝트 목표

본 프로젝트에서는 `skt/kogpt2-base-v2`를 기반으로 다음 질문을 실험적으로 확인했다.

> **동일한 KoGPT-2 기반 모델에서 SFT 데이터 품질과 decoding 전략을 변경하면 한국어 instruction-following 생성 행동이 어떻게 달라지는가? 또한 별도의 Reward Model이 응답 선호 순위를 학습할 수 있는가?**

실험은 `Pretrained KoGPT-2 → Raw SFT → 데이터 정제 SFT → Decoding 비교 → Reward Model → PPO 안정화 시도` 순서로 진행했다.

```text
M0  Pretrained KoGPT-2
 │
 ├─ M1  Raw SFT
 │
 ├─ M2  Clean-v1 SFT
 │      └─ 영어 중심 장문 / 영어 AI 상투 응답 제거
 │
 ├─ M2b Clean-v2 SFT
 │      └─ 한국어 AI 자기소개형 상투 응답 추가 제거
 │
 └─ M2c Clean-v2 Step-matched
        └─ M2와 optimizer update 수를 동일하게 맞춘 ablation

별도 축
KoGPT-2 backbone
 └─ Pairwise Reward Model

확장 실험
M2c Actor + Reference + Reward Model + Value Head
 └─ PPO 안정화 실험

# 실험의 한계

## 한계
- RM ranking의 synthetic label 한계
  - KoChatGPT RM 데이터의 ranking은 실제 사람이 모든 응답을 직접 비교하여 만든 선호도 데이터가 아니라 생성 모델의 우선순위를 활용한 synthetic ranking이다.

  - 실제 샘플 중에는 사람이 보기에 부적절하거나 깨진 응답이 높은 rank를 가진 사례도 있었다.

  - 따라서 Test Pairwise Accuracy 72.22%는 인간 선호 정확도가 아니라 KoChatGPT synthetic preference label 재현 정확도로 해석해야 한다.

- 자동 생성 metric의 한계
  - BLEU, ROUGE-L, chrF는 reference와의 표면적 유사도를 측정하며 사실성이나 사용자 관점의 유용성을 직접 측정하지 않는다.
  - 실제로 Beam Search는 reference metric이 가장 높았지만 AI 상투문구도 많이 재현했다.
  - 반대로 Clean-v2/M2c는 목표 행동은 개선했지만 BLEU와 chrF가 감소했다.

- 데이터 정제의 trade-off
  - Clean-v2는 Raw Train의 약 78%만 유지할 정도로 공격적인 정제를 수행했다.
  - AI 상투 응답은 크게 줄었지만 language-modeling loss와 일부 reference metric은 악화되었다.
  - 따라서 데이터 정제는 단순히 많이 제거할수록 좋은 것이 아니라 목표 행동과 정보 손실 사이의 균형이 필요하다.

- PPO 미완료😢
  - PPO는 최초 구현에서 KL과 Value Loss가 폭발하며 발산했다.
  - Dropout, KL 계산, Actor/Value gradient 경로 등을 수정하며 원인을 좁혔고 1-batch smoke test 수준까지 안정화했지만 전체 PPO 학습은 완료하지 못했다.
  - 따라서 PPO 결과를 최종 모델 성능으로 주장하지 않고 RLHF 안정성 문제를 관찰한 확장 실험으로 기록한다.